## 📑 HAPS FLDT All Applications
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/BADFHQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

In [0]:
# Define the schema name
schema_name = f"`{env_prefix}-cdh-delivery`.repoint"

# Create the schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

table_name = 'HAPS_FLDTAllApplications'

schema = (
"BusinessDate date," 
"AccountImplementationStatus string,"
"app_Id string,"
"app_ApplicationId int,"
"app_ApplicationKey string,"
"app_UserCreated string,"
"app_DateCreated timestamp,"
"app_DateUpdated timestamp,"
"app_ApplicationStatus smallint,"
"app_ApplicationStatusDescription string,"
"app_PortfolioCode string,"
"app_PortfolioName string,"
"app_PortfolioDescription string,"
"app_InvestorCode string,"
"app_InvestorName string,"
"app_InvestorDescription string,"
"app_ProcessingUserId varchar(36),"
"app_ProcessingUserFullName string,"
"app_AssigneeUserId varchar(36),"
"app_AssigneeUserFullName string,"
"app_AssigneePortfolioGroupCode string,"
"app_AssigneePortfolioGroupName string,"
"app_CurrentStepId varchar(36),"
"app_CurrentStepName string,"
"app_UserUnitCode string,"
"app_UserUnitDescription string,"
"app_IntegrationCode string,"
"app_CommunicationChannelCode string,"
"app_CommunicationChannelDescription string,"
"app_CurrentStepIntegrationCode string,"
"app_CustomerCode string,"
"app_FlowSegment string,"
"app_FlowSegmentDescription string,"
"app_UserCreatedQualcoGroupCode string,"
"app_UserCreatedQualcoChannelCode string,"
"app_UserCreatedGroupCategory string,"
"app_UserCreatedUserCreatedGroupCode string,"
"app_UserCreatedGroupName string,"
"app_UserCreatedFullName string,"
"app_AssigneeUserName string,"
"app_ContractSigningMethodCode string,"
"app_ContractSigningMethodDescription string,"
"galaxyApplicationMetadata_SourceApplicationId int,"
"galaxyApplicationMetadata_SourceApplicationKey string,"
"galaxyApplicationMetadata_ApplicationStatus int,"
"galaxyApplicationMetadata_ApplicationStatusDescription string,"
"galaxyApplicationMetadata_CreditInvolvementStatus int,"
"galaxyApplicationMetadata_CreditInvolvementStatusDescription string,"
"galaxyApplicationMetadata_ParentStepGroupingId int,"
"galaxyApplicationMetadata_ParentStepGroupingDescription string,"
"galaxyApplicationMetadata_ParentStepGroupingOwnerId int,"
"galaxyApplicationMetadata_ParentStepGroupingOwnerName string,"
"galaxyApplicationMetadata_StepGroupingId int,"
"galaxyApplicationMetadata_StepGroupingDescription string,"
"galaxyApplicationMetadata_StepGroupingOwnerId int,"
"galaxyApplicationMetadata_StepGroupingOwnerName string,"
"galaxyApplicationMetadata_ClientResponseId int,"
"galaxyApplicationMetadata_ClientResponseDescription string,"
"galaxyApplicationMetadata_OcwSimulationVoteId int,"
"galaxyApplicationMetadata_OcwSimulationVoteDescription string,"
"galaxyApplicationMetadata_QualcoAppStartedDate timestamp,"
"galaxyApplicationMetadata_QualcoAppSubmittedDate timestamp,"
"galaxyApplicationMetadata_QualcoAppPreApprovedDate timestamp,"
"galaxyApplicationMetadata_QualcoAppApprovedDate timestamp,"
"galaxyApplicationMetadata_QualcoAppDisbursedDate timestamp,"
"galaxyApplicationMetadata_QualcoAppRejectedDate timestamp,"
"galaxyApplicationMetadata_QualcoAppCancelledDate timestamp,"
"galaxyApplicationMetadata_QualcoAppFinalStepDate timestamp,"
"galaxyApplicationMetadata_DateCreated timestamp,"
"galaxyApplicationMetadata_DateUpdated timestamp,"
"galaxyApplicationMetadata_IsActive boolean,"
"app_CurrentStepAssignmentDate timestamp,"
"app_InterbankFlag boolean,"
"app_OcwFlag boolean,"
"app_ManagedByPortfolioGroupName string,"
"app_ManagedByPortfolioGroupCode string,"
"app_ContactSigningPlaceId int,"
"app_ContactSigningPlaceName string,"
"app_ContactSigningPlaceTypeId int,"
"app_ContactSigningPlaceTypeName string,"
"app_galaxyApplicationMetadata_StepReportingName string,"
"app_galaxyApplicationMetadata_StepActiveDuration bigint,"
"app_galaxyApplicationMetadata_stepReportingNameId int,"
"ApplicationImplementationStatus string,"
"ApplicationActivationDate date,"
"modification_Id int,"
"modification_CustomerId int,"
"modification_ModificationId int,"
"modification_ModificationRequestKey string,"
"modification_UserName string,"
"modification_Fullname string,"
"modification_Channel smallint,"
"modification_ChannelDescription string,"
"modification_DateOfCalculation timestamp,"
"modification_IsSuccess boolean,"
"modification_ErrorCode string,"
"modification_MinDisposableIncomeForViableSolutions decimal(19,2),"
"modification_MinDisposableIncomeForNonViableSolutions decimal(19,2),"
"modification_IsViableSolution boolean,"
"modification_CalculationMode smallint,"
"modification_CalculationModeDescription string,"
"modification_UnitCode string,"
"modification_UnitName string,"
"modification_UnitDescription string,"
"modification_AutoCreditReasonsVisible boolean,"
"modification_ExtendedLimits smallint,"
"modification_ExtendedLimitsDescription string,"
"modification_CalculatorServiceId int,"
"modification_ModificationAmendmentId int,"
"modification_CreditPolicyId int,"
"modification_CreditPolicyCode string,"
"modification_CreditPolicyName string,"
"modification_CreditPolicyDescription string,"
"customer_Id int,"
"customer_AssetClass string,"
"customer_BalanceCategory string,"
"customer_BeginMonth string,"
"customer_BorrowerIndustry string,"
"customer_BPSegment string,"
"customer_BusinessPlanStrategies string,"
"customer_CustomerFatherName string,"
"customer_CustomerFirstName string,"
"customer_DebtorBankGroupID string,"
"customer_DT_DebtLpsStatus string,"
"customer_DT_RealEstateSearchStatus string,"
"customer_DT_Segment string,"
"customer_EAPSubStatus string,"
"customer_GroupCode string,"
"customer_InvestorSegment string,"
"customer_NegativeEAP string,"
"customer_Occupation string,"
"customer_PositiveEAP string,"
"customer_StrategyID string,"
"customer_UC_ID string,"
"customer_BankruptcyFlag int,"
"customer_BorrowerIndustryCode int,"
"customer_ConfidenceID int,"
"customer_ConnectedBorrowers int,"
"customer_ContactActivities int,"
"customer_ContactActivitiesLast3Months int,"
"customer_ContactActivitiesLastMonth int,"
"customer_CurrentQualcoSegmentRFIs int,"
"customer_CurrentQualcoSegmentRFIsDescription string,"
"customer_CustomerSubSegmentIndication int,"
"customer_CustomerSubSegmentIndicationDescription string,"
"customer_DebtorLPS int,"
"customer_DebtorLPSDescription string,"
"customer_DraftedByID int,"
"customer_DT_BorrowerType int,"
"customer_DT_DenouncementFlag int,"
"customer_DT_ExtrajudicialLetterFlag int,"
"customer_DT_PaymentOrderFlag int,"
"customer_FinalMaxBucketPerPortfpolio int,"
"customer_Gender int,"
"customer_LeverageSegment int,"
"customer_MaritalStatus int,"
"customer_NegativeEAPIndicator int,"
"customer_NegativeRightPersonContactsLast3Months int,"
"customer_NegativeRightPersonContactsLastMonth int,"
"customer_NegativeRightPersonContactsSinceLTSA int,"
"customer_PendingEAPNb int,"
"customer_PositiveEAPIndicator int,"
"customer_PositiveRightPersonContactsLast3Months int,"
"customer_PositiveRightPersonContactsLastMonth int,"
"customer_PositiveRightPersonContactsSinceLTSA int,"
"customer_PostCode int,"
"customer_PromisesLast3Months int,"
"customer_PromisesLastMonth int,"
"customer_PromisesSinceLTSA int,"
"customer_ProposedByID int,"
"customer_RightPersonContactsLast3Months int,"
"customer_RightPersonContactsLastMonth int,"
"customer_RightPersonContactsSinceLTSA int,"
"customer_SolutionTypeID int,"
"customer_StatusID int,"
"customer_TotalAssets int,"
"customer_WatchListType int,"
"customer_CREV decimal(19,2),"
"customer_DT_BorrowerLast12MonthsPayments decimal(19,2),"
"customer_DT_BorrowerOnBalance decimal(19,2),"
"customer_DT_BorrowerTotalBalance decimal(19,2),"
"customer_OffBalanceDebtor decimal(19,2),"
"customer_OnBalanceDebtor decimal(19,2),"
"customer_TotalBalanceDebtor decimal(19,2),"
"customer_TotalPaidAmountLast12Months decimal(19,2),"
"customer_TotalPaidAmountLast3Months decimal(19,2),"
"customer_TotalPaidAmountLastMonth decimal(19,2),"
"customer_TotalPaidAmountSinceLTSA decimal(19,2),"
"customer_ApplicationDate timestamp,"
"customer_CompletionTargetDate timestamp,"
"customer_ContactActivitiesLastDate timestamp,"
"customer_DT_DenouncementDate timestamp,"
"customer_IntentionDate timestamp,"
"customer_LastPaymentDate timestamp,"
"customer_NegativeRightPersonContactLastDate timestamp,"
"customer_PositiveRightPersonContactLastDate timestamp,"
"customer_PromiseLastDate timestamp,"
"customer_RPCLastDate timestamp,"
"customer_PortfolioCode string,"
"customer_PortfolioName string,"
"customer_PortfolioDescription string,"
"customer_InvestorCode string,"
"customer_InvestorName string,"
"customer_InvestorDescription string,"
"customer_CustomerCode string,"
"customer_DateOfBirth timestamp,"
"customer_VATNumber string,"
"customer_CustomerName string,"
"customer_CustomerType string,"
"customer_CustomerTypeDescription string,"
"customer_MonthlyInstallmentsOtherBanks decimal(19,2),"
"customer_MonthlyInstallmentsOtherBanksCurrency string,"
"customer_MonthlyInstallmentsOtherBanksInBaseCurrency decimal(19,2),"
"customer_DepositsAmountTotal decimal(19,2),"
"customer_DepositsAmountTotalCurrency string,"
"customer_DepositsAmountTotalInBaseCurrency decimal(19,2),"
"customer_TimeDepositsAmountTotal decimal(19,2),"
"customer_TimeDepositsAmountTotalCurrency string,"
"customer_TimeDepositsAmountTotalInBaseCurrency decimal(19,2),"
"customer_InvestmentsAmountTotal decimal(19,2),"
"customer_InvestmentsAmountTotalCurrency string,"
"customer_InvestmentsAmountTotalInBaseCurrency decimal(19,2),"
"customer_IsDeceased int,"
"customer_IsDeceasedDesctiption string,"
"customer_WholesaleIndicator int,"
"customer_InsolvencyIndicator int,"
"customer_PrivateBankingIndicator int,"
"customer_StaffIndicator int,"
"customer_GroupID int,"
"customer_BankruptcyFlagDescription string,"
"customer_BorrowerIndustryCodeDescription string,"
"customer_ConnectedBorrowersDescription string,"
"customer_DtBorrowerTypeDescription string,"
"customer_DtDenouncementFlagDescription string,"
"customer_DtExtrajudicialLetterFlagDescription string,"
"customer_DtPaymentOrderFlagDescription string,"
"customer_FinalMaxBucketPerPortfpolioDescription string,"
"customer_GenderDescription string,"
"customer_MaritalStatusDescription string,"
"customer_NegativeEapindicatorDescription string,"
"customer_PositiveEapindicatorDescription string,"
"customer_PostCodeDescription string,"
"customer_SolutionTypeIdDescription string,"
"customer_StatusIdDescription string,"
"customer_WatchListTypeDescription string,"
"customer_WholesaleIndicatorDescription string,"
"customer_InsolvencyIndicatorDescription string,"
"customer_PrivateBankingIndicatorDescription string,"
"customer_StaffIndicatorDescription string,"
"customer_PoliticallyExposedPersons int,"
"customer_PoliticallyExposedPersonsDescription string,"
"customer_CustomerLastName string,"
"customer_CompanyName string,"
"customer_Payer int,"
"customer_PayerDescription string,"
"customer_InterbankIndicator int,"
"customer_InterbankIndicatorDescription string,"
"customer_InterbankPaymentAmount decimal(19,2),"
"customer_PostalAddress string,"
"customer_MobilePhone string,"
"customer_IdNumber string,"
"customer_Email string,"
"customer_HomePhoneNumber string,"
"customer_WorkPhoneNumber string,"
"customer_IsCovid19AffectedIndicator boolean,"
"customer_IsDEKIndicator boolean,"
"customer_IsLaw3869Indicator boolean,"
"customer_IsPEPIndicator boolean,"
"customer_DeclaredAffordability decimal(19,2),"
"customer_StatedAffordability decimal(19,2),"
"customer_IncomeUsedForCalculations decimal(19,2),"
"customer_LegalRepresentativeExpirationDate timestamp,"
"loan_Id int,"
"loan_CustomerId int,"
"loan_AssignmentChannel string,"
"loan_AssignmentOfficerName string,"
"loan_AssignmentType string,"
"loan_CapitalizationCV19 string,"
"loan_CurrentOccupation string,"
"loan_DebtLegalProcessStatus int,"
"loan_EBAStatusBeforCV19 string,"
"loan_EBAStatusSOM string,"
"loan_ProductName string,"
"loan_Queue string,"
"loan_Strategy string,"
"loan_StrategyStepDescription string,"
"loan_ActiveApplication int,"
"loan_AmortizationType int,"
"loan_AssignmentTeam int,"
"loan_CaseWatchlList int,"
"loan_CollateralExistenceFlag int,"
"loan_ContactActivities int,"
"loan_ContactActivitiesLast3Months int,"
"loan_ContactActivitiesLastMonth int,"
"loan_GracePeriodIndicator int,"
"loan_LastApplicationType int,"
"loan_Law3869Type int,"
"loan_LegalProcessStatusDate int,"
"loan_LoanIDSource int,"
"loan_LT12bIndicator int,"
"loan_MinBucket int,"
"loan_NegativeRightPersonContactsLast3Months int,"
"loan_NegativeRightPersonContactsLastMonth int,"
"loan_NegativeRightPersonContactsSinceLTSA int,"
"loan_ObjectiveStatus int,"
"loan_OperationalBucketBeforCV19 int,"
"loan_PerformingStatus int,"
"loan_PositiveRightPersonContactsLast3Months int,"
"loan_PositiveRightPersonContactsLastMonth int,"
"loan_PositiveRightPersonContactsSinceLTSA int,"
"loan_PreviousApplications int,"
"loan_PromisesLast3Months int,"
"loan_PromisesLastMonth int,"
"loan_PromisesSinceLTSA int,"
"loan_QueID int,"
"loan_RemainingProbationMonths int,"
"loan_RightPersonContactsLast3Months int,"
"loan_RightPersonContactsLastMonth int,"
"loan_RightPersonContactsSinceLTSA int,"
"loan_RiskBucketBeforeCV19 int,"
"loan_RiskBucketSOM int,"
"loan_SettlementBucket int,"
"loan_SPV int,"
"loan_AssignmentBalance decimal(19,2),"
"loan_CreditLine decimal(19,2),"
"loan_DT_AccountOnBalance decimal(19,2),"
"loan_DT_AccountTotalBalance decimal(19,2),"
"loan_DT_Last12MonthsPayments decimal(19,2),"
"loan_OffBalance decimal(19,2),"
"loan_OnBalance decimal(19,2),"
"loan_TotalPaidAmountLast12Months decimal(19,2),"
"loan_TotalPaidAmountLast3Months decimal(19,2),"
"loan_TotalPaidAmountLastMonth decimal(19,2),"
"loan_TotalPaidAmountSinceLTSA decimal(19,2),"
"loan_AccountStatusChageDate timestamp,"
"loan_AssignmentDate timestamp,"
"loan_ContactActivitiesLastDate timestamp,"
"loan_FixedInterestEndDate timestamp,"
"loan_L3869CourtDate timestamp,"
"loan_LastApplicationDate timestamp,"
"loan_LastPaymentDate timestamp,"
"loan_NegativeRightPersonContactLastDate timestamp,"
"loan_ObjectiveStatusDate timestamp,"
"loan_PositiveRightPersonContactLastDate timestamp,"
"loan_ProbationStartDate timestamp,"
"loan_PromiseLastDate timestamp,"
"loan_RightPersonContactLastDate timestamp,"
"loan_OpenFlag boolean,"
"loan_AccountNumber string,"
"loan_AssetClass int,"
"loan_AssetClassDesciption string,"
"loan_AssociationType int,"
"loan_AssociationTypeDescription string,"
"loan_ContractNumber string,"
"loan_IsAutomobile int,"
"loan_HasGovernmentGuarantee int,"
"loan_OutstandingBalance decimal(19,2),"
"loan_OutstandingBalanceInBaseCurrency decimal(19,2),"
"loan_RemainingTenorInMonths int,"
"loan_Bucket int,"
"loan_CurrentInterestRate decimal(19,2),"
"loan_EBAStatus int,"
"loan_EBAStatusDescription string,"
"loan_Currency string,"
"loan_CurrentInstallment decimal(19,2),"
"loan_CurrentInstallmentInBaseCurrency decimal(19,2),"
"loan_DaysPastDue int,"
"loan_DebtAmount decimal(19,2),"
"loan_DebtAmountInBaseCurrency decimal(19,2),"
"loan_FrozenInterest decimal(19,2),"
"loan_FrozenInterestInBaseCurrency decimal(19,2),"
"loan_NotAccrudInterest decimal(10,8),"
"loan_NotAccrudInterestInBaseCurrency decimal(19,2),"
"loan_DepreciatedAmount decimal(19,2),"
"loan_DepreciatedAmountInBaseCurrency decimal(19,2),"
"loan_Expenses decimal(19,2),"
"loan_ExpensesInBaseCurrency decimal(19,2),"
"loan_HasLegalActions int,"
"loan_BorrowerAccountSegment int,"
"loan_NumberOfPreviousModifications int,"
"loan_MaxDaysPastDueEver int,"
"loan_LegalProcessStatus int,"
"loan_DenouncedCaseEver int,"
"loan_IsRevolvingLoan int,"
"loan_LoanOriginationDate timestamp,"
"loan_ProductCode string,"
"loan_MonthsSinceMostRecentModification int,"
"loan_AccountStatus string,"
"loan_AccountStatusChangeDate timestamp,"
"loan_PermanentDelayStatusTurnDate timestamp,"
"loan_CardNumber string,"
"loan_IsDenounced int,"
"loan_BaseInterest decimal(19,2),"
"loan_Law128Rate decimal(19,2),"
"loan_Spread decimal(19,2),"
"loan_IsFixedInterest int,"
"loan_ContractualInterestRate decimal(19,2),"
"loan_CurrentInterestAmount decimal(19,2),"
"loan_CurrentInterestAmountInBaseCurrency decimal(19,2),"
"loan_AgencyType int,"
"loan_BaseInterestType int,"
"loan_Delivered int,"
"loan_AccountOwnerId string,"
"loan_LoanPurpose int,"
"loan_ParticipationPercentage decimal(19,2),"
"loan_ExchangeRateAtOrigination decimal(19,2),"
"loan_IsOverdraft int,"
"loan_CreditLimit decimal(19,2),"
"loan_CreditLimitInBaseCurrency decimal(19,2),"
"loan_ActiveApplicationDescription string,"
"loan_AmortizationTypeDescription string,"
"loan_AssignmentTeamDescription string,"
"loan_CaseWatchlListDescription string,"
"loan_GracePeriodIndicatorDescription string,"
"loan_LastApplicationTypeDescription string,"
"loan_Law3869TypeDescription string,"
"loan_LegalProcessStatusDateDescription string,"
"loan_Lt12bIndicatorDescription string,"
"loan_MinBucketDescription string,"
"loan_ObjectiveStatusDescription string,"
"loan_PerformingStatusDescription string,"
"loan_QueIdDescription string,"
"loan_SettlementBucketDescription string,"
"loan_SpvDescription string,"
"loan_IsAutomobileDescription string,"
"loan_HasGovernmentGuaranteeDescription string,"
"loan_HasLegalActionsDescription string,"
"loan_BorrowerAccountSegmentDescription string,"
"loan_LegalProcessStatusDescription string,"
"loan_DenounceCaseEverDescription string,"
"loan_IsRevolvingLoanDescription string,"
"loan_IsDenouncedDescription string,"
"loan_IsFixedInterestDescription string,"
"loan_AgencyTypeDescription string,"
"loan_BaseInterestTypeDescription string,"
"loan_DeliveredDescription string,"
"loan_LoanPurposeDescription string,"
"loan_IsOverdraftDescription string,"
"loan_PositiveLoanEAPIndicator string,"
"loan_PositiveLoanEAPIndicatorDescription string,"
"loan_ContractNumberSContractNumberSystemicNrystemicNr string,"
"loan_EligiblePrenotedAsset int,"
"loan_EligiblePrenotedAssetDescription string,"
"loan_AccountPayer int,"
"loan_AccountPayerDescription string,"
"loan_AccountNumberWithPrefix string,"
"loan_ContractNumberSystemicNr string,"
"loan_UnifiedLoanId int,"
"loan_IsUnifiedLoan boolean,"
"loan_DecisionTreeCode string,"
"loan_DecisionTreeDescription string,"
"loan_DecisionTreeValidPath string,"
"loan_IsForModification boolean,"
"unifiedLoan_Id int,"
"unifiedLoan_AssignmentChannel string,"
"unifiedLoan_AssignmentOfficerName string,"
"unifiedLoan_AssignmentType string,"
"unifiedLoan_CapitalizationCV19 string,"
"unifiedLoan_CurrentOccupation string,"
"unifiedLoan_DebtLegalProcessStatus int,"
"unifiedLoan_EBAStatusBeforCV19 string,"
"unifiedLoan_EBAStatusSOM string,"
"unifiedLoan_ProductName string,"
"unifiedLoan_Queue string,"
"unifiedLoan_Strategy string,"
"unifiedLoan_StrategyStepDescription string,"
"unifiedLoan_ActiveApplication int,"
"unifiedLoan_AmortizationType int,"
"unifiedLoan_AssignmentTeam int,"
"unifiedLoan_CaseWatchlList int,"
"unifiedLoan_CollateralExistenceFlag int,"
"unifiedLoan_ContactActivities int,"
"unifiedLoan_ContactActivitiesLast3Months int,"
"unifiedLoan_ContactActivitiesLastMonth int,"
"unifiedLoan_GracePeriodIndicator int,"
"unifiedLoan_LastApplicationType int,"
"unifiedLoan_Law3869Type int,"
"unifiedLoan_LegalProcessStatusDate int,"
"unifiedLoan_LoanIDSource int,"
"unifiedLoan_LT12bIndicator int,"
"unifiedLoan_MinBucket int,"
"unifiedLoan_NegativeRightPersonContactsLast3Months int,"
"unifiedLoan_NegativeRightPersonContactsLastMonth int,"
"unifiedLoan_NegativeRightPersonContactsSinceLTSA int,"
"unifiedLoan_ObjectiveStatus int,"
"unifiedLoan_OperationalBucketBeforCV19 int,"
"unifiedLoan_PerformingStatus int,"
"unifiedLoan_PositiveRightPersonContactsLast3Months int,"
"unifiedLoan_PositiveRightPersonContactsLastMonth int,"
"unifiedLoan_PositiveRightPersonContactsSinceLTSA int,"
"unifiedLoan_PreviousApplications int,"
"unifiedLoan_PromisesLast3Months int,"
"unifiedLoan_PromisesLastMonth int,"
"unifiedLoan_PromisesSinceLTSA int,"
"unifiedLoan_QueID int,"
"unifiedLoan_RemainingProbationMonths int,"
"unifiedLoan_RightPersonContactsLast3Months int,"
"unifiedLoan_RightPersonContactsLastMonth int,"
"unifiedLoan_RightPersonContactsSinceLTSA int,"
"unifiedLoan_RiskBucketBeforeCV19 int,"
"unifiedLoan_RiskBucketSOM int,"
"unifiedLoan_SettlementBucket int,"
"unifiedLoan_SPV int,"
"unifiedLoan_AssignmentBalance decimal(19,2),"
"unifiedLoan_CreditLine decimal(19,2),"
"unifiedLoan_DT_AccountOnBalance decimal(19,2),"
"unifiedLoan_DT_AccountTotalBalance decimal(19,2),"
"unifiedLoan_DT_Last12MonthsPayments decimal(19,2),"
"unifiedLoan_OffBalance decimal(19,2),"
"unifiedLoan_OnBalance decimal(19,2),"
"unifiedLoan_TotalPaidAmountLast12Months decimal(19,2),"
"unifiedLoan_TotalPaidAmountLast3Months decimal(19,2),"
"unifiedLoan_TotalPaidAmountLastMonth decimal(19,2),"
"unifiedLoan_TotalPaidAmountSinceLTSA decimal(19,2),"
"unifiedLoan_AccountStatusChageDate timestamp,"
"unifiedLoan_AssignmentDate timestamp,"
"unifiedLoan_ContactActivitiesLastDate timestamp,"
"unifiedLoan_FixedInterestEndDate timestamp,"
"unifiedLoan_L3869CourtDate timestamp,"
"unifiedLoan_LastApplicationDate timestamp,"
"unifiedLoan_LastPaymentDate timestamp,"
"unifiedLoan_NegativeRightPersonContactLastDate timestamp,"
"unifiedLoan_ObjectiveStatusDate timestamp,"
"unifiedLoan_PositiveRightPersonContactLastDate timestamp,"
"unifiedLoan_ProbationStartDate timestamp,"
"unifiedLoan_PromiseLastDate timestamp,"
"unifiedLoan_RightPersonContactLastDate timestamp,"
"unifiedLoan_OpenFlag boolean,"
"unifiedLoan_AccountNumber string,"
"unifiedLoan_AssetClass int,"
"unifiedLoan_AssetClassDesciption string,"
"unifiedLoan_AssociationType int,"
"unifiedLoan_AssociationTypeDescription string,"
"unifiedLoan_ContractNumber string,"
"unifiedLoan_IsAutomobile int,"
"unifiedLoan_HasGovernmentGuarantee int,"
"unifiedLoan_OutstandingBalance decimal(19,2),"
"unifiedLoan_OutstandingBalanceInBaseCurrency decimal(19,2),"
"unifiedLoan_RemainingTenorInMonths int,"
"unifiedLoan_Bucket int,"
"unifiedLoan_CurrentInterestRate decimal(19,2),"
"unifiedLoan_EBAStatus int,"
"unifiedLoan_EBAStatusDescription string,"
"unifiedLoan_Currency string,"
"unifiedLoan_CurrentInstallment decimal(19,2),"
"unifiedLoan_CurrentInstallmentInBaseCurrency decimal(19,2),"
"unifiedLoan_DaysPastDue int,"
"unifiedLoan_DebtAmount decimal(19,2),"
"unifiedLoan_DebtAmountInBaseCurrency decimal(19,2),"
"unifiedLoan_FrozenInterest decimal(19,2),"
"unifiedLoan_FrozenInterestInBaseCurrency decimal(19,2),"
"unifiedLoan_NotAccrudInterest decimal(19,2),"
"unifiedLoan_NotAccrudInterestInBaseCurrency decimal(19,2),"
"unifiedLoan_DepreciatedAmount decimal(19,2),"
"unifiedLoan_DepreciatedAmountInBaseCurrency decimal(19,2),"
"unifiedLoan_Expenses decimal(19,2),"
"unifiedLoan_ExpensesInBaseCurrency decimal(19,2),"
"unifiedLoan_HasLegalActions int,"
"unifiedLoan_BorrowerAccountSegment int,"
"unifiedLoan_NumberOfPreviousModifications int,"
"unifiedLoan_MaxDaysPastDueEver int,"
"unifiedLoan_LegalProcessStatus int,"
"unifiedLoan_DenouncedCaseEver int,"
"unifiedLoan_IsRevolvingLoan int,"
"unifiedLoan_LoanOriginationDate timestamp,"
"unifiedLoan_ProductCode string,"
"unifiedLoan_MonthsSinceMostRecentModification int,"
"unifiedLoan_AccountStatus string,"
"unifiedLoan_AccountStatusChangeDate timestamp,"
"unifiedLoan_PermanentDelayStatusTurnDate timestamp,"
"unifiedLoan_CardNumber string,"
"unifiedLoan_IsDenounced int,"
"unifiedLoan_BaseInterest decimal(19,2),"
"unifiedLoan_Law128Rate decimal(19,2),"
"unifiedLoan_Spread decimal(19,2),"
"unifiedLoan_IsFixedInterest int,"
"unifiedLoan_ContractualInterestRate decimal(19,2),"
"unifiedLoan_CurrentInterestAmount decimal(19,2),"
"unifiedLoan_CurrentInterestAmountInBaseCurrency decimal(19,2),"
"unifiedLoan_AgencyType int,"
"unifiedLoan_BaseInterestType int,"
"unifiedLoan_Delivered int,"
"unifiedLoan_AccountOwnerId string,"
"unifiedLoan_LoanPurpose int,"
"unifiedLoan_ParticipationPercentage decimal(19,2),"
"unifiedLoan_ExchangeRateAtOrigination decimal(19,2),"
"unifiedLoan_IsOverdraft int,"
"unifiedLoan_CreditLimit decimal(19,2),"
"unifiedLoan_CreditLimitInBaseCurrency decimal(19,2),"
"unifiedLoan_ActiveApplicationDescription string,"
"unifiedLoan_AmortizationTypeDescription string,"
"unifiedLoan_AssignmentTeamDescription string,"
"unifiedLoan_CaseWatchlListDescription string,"
"unifiedLoan_GracePeriodIndicatorDescription string,"
"unifiedLoan_LastApplicationTypeDescription string,"
"unifiedLoan_Law3869TypeDescription string,"
"unifiedLoan_LegalProcessStatusDateDescription string,"
"unifiedLoan_Lt12bIndicatorDescription string,"
"unifiedLoan_MinBucketDescription string,"
"unifiedLoan_ObjectiveStatusDescription string,"
"unifiedLoan_PerformingStatusDescription string,"
"unifiedLoan_QueIdDescription string,"
"unifiedLoan_SettlementBucketDescription string,"
"unifiedLoan_SpvDescription string,"
"unifiedLoan_IsAutomobileDescription string,"
"unifiedLoan_HasGovernmentGuaranteeDescription string,"
"unifiedLoan_HasLegalActionsDescription string,"
"unifiedLoan_BorrowerAccountSegmentDescription string,"
"unifiedLoan_LegalProcessStatusDescription string,"
"unifiedLoan_DenounceCaseEverDescription string,"
"unifiedLoan_IsRevolvingLoanDescription string,"
"unifiedLoan_IsDenouncedDescription string,"
"unifiedLoan_IsFixedInterestDescription string,"
"unifiedLoan_AgencyTypeDescription string,"
"unifiedLoan_BaseInterestTypeDescription string,"
"unifiedLoan_DeliveredDescription string,"
"unifiedLoan_LoanPurposeDescription string,"
"unifiedLoan_IsOverdraftDescription string,"
"unifiedLoan_PositiveLoanEAPIndicator string,"
"unifiedLoan_PositiveLoanEAPIndicatorDescription string,"
"unifiedLoan_ContractNumberSContractNumberSystemicNrystemicNr string,"
"unifiedLoan_EligiblePrenotedAsset int,"
"unifiedLoan_EligiblePrenotedAssetDescription string,"
"unifiedLoan_AccountPayer int,"
"unifiedLoan_AccountPayerDescription string,"
"unifiedLoan_AccountNumberWithPrefix string,"
"unifiedLoan_ContractNumberSystemicNr string,"
"unifiedLoan_DecisionTreeCode string,"
"unifiedLoan_DecisionTreeDescription string,"
"unifiedLoan_DecisionTreeValidPath string,"
"loanId int,"
"product_Id varchar(36),"
"product_LoanId int,"
"product_ProductKey varchar(36),"
"product_Priority int,"
"product_ProductTypePriority int,"
"product_NPVInBaseCurrency decimal(19,2),"
"product_NPV decimal(19,2),"
"product_Code string,"
"product_Name string,"
"product_ProductCategory int,"
"product_ProductCategoryType string,"
"product_BalanceUsedForNPVCalculationInBaseCurrency decimal(19,2),"
"product_BalanceUsedForNPVCalculation decimal(19,2),"
"product_DownPaymentAmountInBaseCurrency decimal(19,2),"
"product_DownPaymentAmount decimal(19,2),"
"product_BalanceForModificationInBaseCurrency decimal(19,2),"
"product_BalanceForModification decimal(19,2),"
"product_ExpensesOfModificationAmountInBaseCurrency decimal(19,2),"
"product_ExpensesOfModificationAmount decimal(19,2),"
"product_HasStepUp boolean,"
"product_HasHairCut boolean,"
"product_HairCutPercentage decimal(19,2),"
"product_HairCutAmount decimal(19,2),"
"product_HairCutPercentageOnBalance decimal(19,2),"
"product_HairCutAmountOnBalanceInBaseCurrency decimal(19,2),"
"product_HairCutAmountOnBalance decimal(19,2),"
"product_HaircutPercentageWithOffBalance decimal(19,2),"
"product_HaircutAmountWithOffBalanceInBaseCurrency decimal(19,2),"
"product_HaircutAmountWithOffBalance decimal(19,2),"
"product_HasHairCutWithClawBack int,"
"product_HasSplitBalance boolean,"
"product_SplitBalancePercentage decimal(19,2),"
"product_ModificationCurrency string,"
"product_AutoCreditStatus int,"
"product_AutoCreditStatusDescription string,"
"product_HasNonAutoCreditReasons boolean,"
"product_NonAutoCreditReasonsVisible boolean,"
"product_IsViable boolean,"
"product_PenaltyInterestRate decimal(19,2),"
"product_RequiresNewPrenotation boolean,"
"product_TotalTenureInMonths int,"
"product_SmallTicketAmountInBaseCurrency decimal(19,2),"
"product_SmallTicketAmount decimal(19,2),"
"product_IsVisible boolean,"
"product_IsLiquidation boolean,"
"product_InstallmentAmountCalculatedInBaseCurrency decimal(19,2),"
"product_InstallmentAmountCalculated decimal(19,2),"
"product_TotalInterestRate decimal(19,2),"
"product_TenureInMonthsMin int,"
"product_TenureInMonthsMax int,"
"product_HairCutPercentageMin decimal(19,2),"
"product_HairCutPercentageMax decimal(19,2),"
"product_IsEditable boolean,"
"product_IsCustom boolean,"
"product_OriginProductKey varchar(36),"
"product_ModificationAmountAfterHaircut decimal(19,2),"
"product_StepUpPeriodTenureInMonths int,"
"product_StepUpStartMonth int,"
"product_StepUpReadjustmentDuration int,"
"product_StepUpPeriodNumber int,"
"product_StepUpInitialInstallmentAmount decimal(19,2),"
"product_StepUpAmount decimal(19,2),"
"product_DurationAfterStepUpExpirationInMonths int,"
"product_SplitBalanceAmountA decimal(19,2),"
"product_SplitBalanceAmountB decimal(19,2),"
"product_SplitBalanceTenureInMonthsA int,"
"product_SplitBalanceStartMonthB int,"
"product_SplitBalanceTenureInMonthsB int,"
"product_DifferentInterestsDuringPeriods int,"
"product_InterestDurationOfFirstPeriodInMonths int,"
"product_FixedInterestOfFirstPeriod decimal(19,2),"
"product_BaseInterestTypeOfFirstPeriod string,"
"product_SpreadOfFirstPeriod decimal(19,2),"
"product_Law128RateOfFirstPeriodValue decimal(19,2),"
"product_BaseInterest string,"
"product_Spread decimal(19,2),"
"product_Law128RateValue decimal(19,2),"
"product_ReducedInstalmentPeriod int,"
"product_ReducedInstalmentPeriodInMonths int,"
"product_ReducedInstalmentPeriodAmount decimal(19,2),"
"finalprod_Id int,"
"profitsInfo_Id int,"
"profitsInfo_AccountImplementationInfoId int,"
"profitsInfo_FinalModificationProductId int,"
"profitsInfo_ExternalSystemGroupApplicationID string,"
"profitsInfo_ExternalSystemApplicationID string,"
"profitsInfo_NewAccountNumber string,"
"profitsInfo_RestructuringProgram string,"
"profitsInfo_DateOfFirstInstallment date,"
"profitsInfo_InstallmentAmount decimal(19,2),"
"profitsInfo_ProcessStatus smallint,"
"profitsInfo_ErrorDescription string,"
"profitsInfo_SystemCode int,"
"profitsInfo_ProfitsApplicationID string,"
"profitsInfo_CustomerCode string,"
"profitsInfo_InternalSN int,"
"profitsInfo_TransactionDate date,"
"profitsInfo_AccountSystem string,"
"profitsInfo_ProductCode int,"
"profitsInfo_ProcessTimestamp timestamp,"
"profitsInfo_SentStatus smallint,"
"profitsInfo_InterestAmount decimal(19,2),"
"profitsInfo_InterestFirstDate date,"
"profitsInfo_ImplementationStatusId smallint,"
"profitsInfo_ImplementationStatusDescription string,"
"profitsInfo_IsActive boolean,"
"profitsInfo_AccountNumber string,"
"profitsInfo_ProductKey varchar(36),"
"profitsInfo_DateCreated timestamp,"
"profitsInfo_SourceFinalSolutionId int,"
"QualcoAppSubmittedDate timestamp,"
"QualcoAppPreApprovedDate timestamp,"
"QualcoAppApprovedDate timestamp,"
"QualcoAppDisbursedDate timestamp,"
"QualcoAppRejectedDate timestamp,"
"QualcoAppCancelledDate timestamp,"
"DateUpdated timestamp,"
"QualcoAppFinalStepDate timestamp,"
"SnapshotDate timestamp,"
"ActivatedApplicationFlag int,"
"StepSortOrder float,"
"loan_GBV decimal(19,2),"
"app_PolicyOverride boolean,"
"app_ContactSigningPlaceCode string,"
"product_FirstPeriodTotalInterestRate decimal(19,2),"
"product_DownPaymentAmountRate decimal(19,2),"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""

# ================== Create Delta Table ==================
#delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"  

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date  # type: ignore
)


# Create the neccesary sub-tables

##### LatestApplication

In [0]:
LatestApplication = spark.sql(
f'''
SELECT app.*, 
    galaxyApplicationMetadata.SourceApplicationId, 
    galaxyApplicationMetadata.SourceApplicationKey, 
    galaxyApplicationMetadata.ApplicationStatus AS `galaxyApplicationMetadata_ApplicationStatus`, 
    galaxyApplicationMetadata.ApplicationStatusDescription AS `galaxyApplicationMetadata_ApplicationStatusDescription`, 
    galaxyApplicationMetadata.CreditInvolvementStatus, 
    galaxyApplicationMetadata.CreditInvolvementStatusDescription, 
    galaxyApplicationMetadata.ParentStepGroupingId, 
    galaxyApplicationMetadata.ParentStepGroupingDescription, 
    galaxyApplicationMetadata.ParentStepGroupingOwnerId,
    galaxyApplicationMetadata.ParentStepGroupingOwnerName,
    galaxyApplicationMetadata.StepGroupingId, 
    galaxyApplicationMetadata.StepGroupingDescription, 
    galaxyApplicationMetadata.StepGroupingOwnerId, 
    galaxyApplicationMetadata.StepGroupingOwnerName,
    galaxyApplicationMetadata.ClientResponseId, 
    galaxyApplicationMetadata.ClientResponseDescription, 
    galaxyApplicationMetadata.OcwSimulationVoteId, 
    galaxyApplicationMetadata.OcwSimulationVoteDescription, 
    galaxyApplicationMetadata.QualcoAppStartedDate, 
    galaxyApplicationMetadata.QualcoAppSubmittedDate, 
    galaxyApplicationMetadata.QualcoAppPreApprovedDate, 
    galaxyApplicationMetadata.QualcoAppApprovedDate, 
    galaxyApplicationMetadata.QualcoAppDisbursedDate, 
    galaxyApplicationMetadata.QualcoAppRejectedDate, 
    galaxyApplicationMetadata.QualcoAppCancelledDate, 
    galaxyApplicationMetadata.QualcoAppFinalStepDate, 
    galaxyApplicationMetadata.DateCreated AS `galaxyApplicationMetadata_DateCreated`, 
    galaxyApplicationMetadata.DateUpdated AS `galaxyApplicationMetadata_DateUpdated`, 
    galaxyApplicationMetadata.IsActive,
    galaxyApplicationMetadata.StepReportingNameId AS `galaxyApplicationMetadata_stepReportingNameId`,
    galaxyApplicationMetadata.StepReportingName AS `galaxyApplicationMetadata_StepReportingName`,
    galaxyApplicationMetadata.StepActiveDuration AS `galaxyApplicationMetadata_StepActiveDuration`,
    galaxyApplicationMetadata.ApplicationImplementationStatus AS `galaxyApplicationMetadata_ApplicationImplementationStatus`,
    galaxyApplicationMetadata.ApplicationActivationDate AS `galaxyApplicationMetadata_ApplicationActivationDate`   
FROM `p-cdh`.landed.fldt_application AS app   
INNER JOIN `p-cdh`.landed.fldt_applicationmetadata AS galaxyApplicationMetadata 
ON galaxyApplicationMetadata.ApplicationId = app.Id
AND galaxyApplicationMetadata.BusinessDate = app.BusinessDate
INNER JOIN (
            SELECT f.CustomerCode, 
                    MAX(f.DateCreated) AS MaxDateCreated
            FROM `p-cdh`.landed.fldt_application AS f 
            WHERE f.ApplicationStatus = '2'
            AND f.BusinessDate = {run_date!r}
            GROUP BY f.CustomerCode
            ) a 
    ON a.CustomerCode = app.CustomerCode
    AND a.MaxDateCreated = app.DateCreated


-- Apply Filters
AND app.BusinessDate = {run_date!r}
AND galaxyApplicationMetadata.IsActive = TRUE


UNION



SELECT app.*,
    galaxyApplicationMetadata.SourceApplicationId, 
    galaxyApplicationMetadata.SourceApplicationKey, 
    galaxyApplicationMetadata.ApplicationStatus AS `galaxyApplicationMetadata_ApplicationStatus`, 
    galaxyApplicationMetadata.ApplicationStatusDescription AS `galaxyApplicationMetadata_ApplicationStatusDescription`, 
    galaxyApplicationMetadata.CreditInvolvementStatus, 
    galaxyApplicationMetadata.CreditInvolvementStatusDescription, 
    galaxyApplicationMetadata.ParentStepGroupingId, 
    galaxyApplicationMetadata.ParentStepGroupingDescription, 
    galaxyApplicationMetadata.ParentStepGroupingOwnerId,
    galaxyApplicationMetadata.ParentStepGroupingOwnerName,
    galaxyApplicationMetadata.StepGroupingId, 
    galaxyApplicationMetadata.StepGroupingDescription, 
    galaxyApplicationMetadata.StepGroupingOwnerId, 
    galaxyApplicationMetadata.StepGroupingOwnerName,
    galaxyApplicationMetadata.ClientResponseId, 
    galaxyApplicationMetadata.ClientResponseDescription, 
    galaxyApplicationMetadata.OcwSimulationVoteId, 
    galaxyApplicationMetadata.OcwSimulationVoteDescription, 
    galaxyApplicationMetadata.QualcoAppStartedDate, 
    galaxyApplicationMetadata.QualcoAppSubmittedDate, 
    galaxyApplicationMetadata.QualcoAppPreApprovedDate, 
    galaxyApplicationMetadata.QualcoAppApprovedDate, 
    galaxyApplicationMetadata.QualcoAppDisbursedDate, 
    galaxyApplicationMetadata.QualcoAppRejectedDate, 
    galaxyApplicationMetadata.QualcoAppCancelledDate, 
    galaxyApplicationMetadata.QualcoAppFinalStepDate, 
    galaxyApplicationMetadata.DateCreated AS `galaxyApplicationMetadata_DateCreated`, 
    galaxyApplicationMetadata.DateUpdated AS `galaxyApplicationMetadata_DateUpdated`, 
    galaxyApplicationMetadata.IsActive,
    galaxyApplicationMetadata.StepReportingNameId AS `galaxyApplicationMetadata_stepReportingNameId`,
    galaxyApplicationMetadata.StepReportingName AS `galaxyApplicationMetadata_StepReportingName`,
    galaxyApplicationMetadata.StepActiveDuration AS `galaxyApplicationMetadata_StepActiveDuration`,
    galaxyApplicationMetadata.ApplicationImplementationStatus AS `galaxyApplicationMetadata_ApplicationImplementationStatus`,
    galaxyApplicationMetadata.ApplicationActivationDate AS `galaxyApplicationMetadata_ApplicationActivationDate`
FROM `p-cdh`.landed.fldt_application AS app   
INNER JOIN `p-cdh`.landed.fldt_applicationmetadata AS galaxyApplicationMetadata 
ON galaxyApplicationMetadata.ApplicationId = app.Id
AND galaxyApplicationMetadata.BusinessDate = app.BusinessDate
WHERE app.ApplicationStatus <> '2'
AND app.BusinessDate = {run_date!r}
AND galaxyApplicationMetadata.IsActive  = TRUE




'''
)
LatestApplication.createOrReplaceTempView("LatestApplication")

##### LatestCustomer

In [0]:

LatestCustomer = spark.sql(
f'''
SELECT * 
FROM ( 
    SELECT *, ROW_NUMBER() OVER(PARTITION BY ApplicationId ORDER BY Id DESC) as rn 
    FROM (
        SELECT f.*
        FROM (
            SELECT f.ApplicationId, MAX(f.PositionDate) AS MaxDate
            FROM `p-cdh`.landed.fldt_customer AS f
            LEFT JOIN `p-cdh`.landed.fldt_modification AS mod 
                ON mod.CustomerId = f.Id
                AND mod.BusinessDate = f.BusinessDate
            WHERE mod.Id IS NULL OR mod.IsSuccess = true
            AND mod.BusinessDate = {run_date!r}
            GROUP BY f.ApplicationId


            UNION


            SELECT a.ApplicationId, MAX(customer.PositionDate) AS MaxDate
            FROM (

                SELECT COUNT(1) AS cnt, customer.ApplicationId
                FROM `p-cdh`.landed.fldt_customer customer
                INNER JOIN `p-cdh`.landed.fldt_modification modification 
                    ON Customer.Id = modification.CustomerId
                    AND Customer.BusinessDate = modification.BusinessDate
                WHERE Customer.BusinessDate = {run_date!r}
                GROUP BY customer.ApplicationId
                HAVING SUM(CAST(modification.IsSuccess AS INT)) = 0

                ) a

            INNER JOIN `p-cdh`.landed.fldt_customer customer  
                ON customer.ApplicationId = a.ApplicationId
            WHERE Customer.BusinessDate = {run_date!r}
            GROUP BY a.ApplicationId
            ) a

        INNER JOIN `p-cdh`.landed.fldt_customer f  
            ON f.ApplicationId = a.ApplicationId
            AND f.PositionDate = a.MaxDate
        WHERE f.BusinessDate = {run_date!r}
        ) a

        -- Apply Filters
        WHERE a.BusinessDate = {run_date!r}

    ) b 
    WHERE rn = 1
'''
)
LatestCustomer.createOrReplaceTempView("LatestCustomer")

##### LatestModification

In [0]:

LatestModification = spark.sql(
f'''
SELECT f.*
FROM (
    SELECT f.CustomerId, 
            MAX(f.DateOfCalculation) AS MaxDate
    FROM `p-cdh`.landed.fldt_modification f
    WHERE f.IsSuccess = true
    AND f.BusinessDate = {run_date!r}
    GROUP BY f.CustomerId, f.BusinessDate
    ) a
INNER JOIN `p-cdh`.landed.fldt_modification f 
  ON f.CustomerId = a.CustomerId
  AND f.DateOfCalculation = a.MaxDate


-- Apply Filters
WHERE f.BusinessDate = {run_date!r}
'''
)
LatestModification.createOrReplaceTempView("LatestModification")

##### Loan

In [0]:

Loan = spark.sql(
f'''
SELECT *
FROM `p-cdh`.landed.fldt_loan 
-- Apply Filters
WHERE BusinessDate = {run_date!r}
'''
)
Loan.createOrReplaceTempView("Loan")

##### LoansForModification

In [0]:
LoansForModification = spark.sql(
f'''
SELECT *
FROM Loan
WHERE IsForModification = TRUE
'''
)
LoansForModification.createOrReplaceTempView("LoansForModification")

##### UnifiedLoan

In [0]:

UnifiedLoan = spark.sql(
f'''
SELECT *
FROM `p-cdh`.landed.fldt_unifiedloan 
-- Apply Filters
WHERE BusinessDate = {run_date!r}
'''
)
UnifiedLoan.createOrReplaceTempView("UnifiedLoan")

##### SuggestedModificationProduct

In [0]:

SuggestedModificationProduct = spark.sql(
f'''
SELECT *
FROM `p-cdh`.landed.fldt_suggestedmodificationproduct 
-- Apply Filters
WHERE BusinessDate = {run_date!r}
'''
)
SuggestedModificationProduct.createOrReplaceTempView("SuggestedModificationProduct")

##### FinalModificationProduct

In [0]:

FinalModificationProduct = spark.sql(
f'''
SELECT *
FROM `p-cdh`.landed.fldt_finalmodificationproduct 
-- Apply Filters
WHERE BusinessDate = {run_date!r}
'''
)
FinalModificationProduct.createOrReplaceTempView("FinalModificationProduct")

##### LatestFinalSolution

In [0]:

LatestFinalSolution = spark.sql(
f'''
SELECT * 
FROM ( 
    SELECT *, ROW_NUMBER() OVER(PARTITION BY ModificationId ORDER BY Id DESC) as rn 
    FROM (
        SELECT f.*
        FROM (
            SELECT f.ModificationId, MAX(f.DateInserted) AS MaxDateInserted
            FROM `p-cdh`.landed.fldt_finalsolution f 
            WHERE f.BusinessDate = {run_date!r}
            GROUP BY f.ModificationId
            ) a
        INNER JOIN `p-cdh`.landed.fldt_finalsolution f  
		   ON f.ModificationId = a.ModificationId
           AND f.DateInserted = a.MaxDateInserted
        ) a


    -- Apply Filters
    WHERE a.BusinessDate = {run_date!r}
    )
'''
)
LatestFinalSolution.createOrReplaceTempView("LatestFinalSolution")

##### LatestAccountImplementationInfo

In [0]:

LatestAccountImplementationInfo = spark.sql(
f'''
SELECT f.*
FROM (
    SELECT f.FinalModificationProductId, MAX(f.DateCreated) AS MaxDateInserted
    FROM `p-cdh`.landed.fldt_accountimplementationinfo   f 
    GROUP BY f.FinalModificationProductId
    ) a
INNER JOIN `p-cdh`.landed.fldt_accountimplementationinfo   f 
    ON f.FinalModificationProductId = a.FinalModificationProductId
    AND f.DateCreated = a.MaxDateInserted
-- WHERE f.ProcessStatus = 1

-- Apply Filters
AND f.BusinessDate = {run_date!r}
'''
)
LatestAccountImplementationInfo.createOrReplaceTempView("LatestAccountImplementationInfo")

##### FLDT_AllApplications

In [0]:
FLDT_AllApplications = spark.sql(
f'''

SELECT 
	app.BusinessDate
	,CASE
					WHEN profitsInfo.NewAccountNumber IS NOT NULL AND profitsInfo.NewAccountNumber <> ''
     --WHEN profitsInfo.FinalModificationProductId IS NOT NULL 
					THEN 'Implemented'
					ELSE 'Not Implemented'
			END AS `AccountImplementationStatus`
		,app.ApplicationKey as `app_Id`			
		,app.ApplicationId as `app_ApplicationId`			
		,app.ApplicationKey as `app_ApplicationKey`			
		,app.UserCreated as `app_UserCreated`			
		,app.DateCreated as `app_DateCreated`			
		,app.DateUpdated as `app_DateUpdated`			
		,app.ApplicationStatus	 as 	`app_ApplicationStatus`			
		,app.ApplicationStatusDescription	 as 	`app_ApplicationStatusDescription`			
		,app.PortfolioCode	 as 	`app_PortfolioCode`			
		,app.PortfolioName	 as 	`app_PortfolioName`			
		,app.PortfolioName	 as 	`app_PortfolioDescription`			
		,app.InvestorCode	 as 	`app_InvestorCode`			
		,app.InvestorName	 as 	`app_InvestorName`		
		,app.InvestorDescription	 as 	`app_InvestorDescription`			
		,app.ProcessingUserId	 as 	`app_ProcessingUserId`			
		,app.ProcessingUserFullName	 as 	`app_ProcessingUserFullName`			
		,app.AssigneeUserId	 as 	`app_AssigneeUserId`			
		,app.AssigneeUserFullName	 as 	`app_AssigneeUserFullName`			
		,app.AssigneePortfolioGroupCode	 as 	`app_AssigneePortfolioGroupCode`			
		,app.AssigneePortfolioGroupName	 as 	`app_AssigneePortfolioGroupName`			
		,app.CurrentStepId	 as 	`app_CurrentStepId`			
		,app.CurrentStepName	 as 	`app_CurrentStepName`			
		,app.UserUnitCode	 as 	`app_UserUnitCode`			
		,app.UserUnitDescription	 as 	`app_UserUnitDescription`			
		,app.IntegrationCode	 as 	`app_IntegrationCode`			
		,app.CommunicationChannelCode	 as 	`app_CommunicationChannelCode`			
		,app.CommunicationChannelDescription	 as 	`app_CommunicationChannelDescription`			
		,app.CurrentStepIntegrationCode	 as 	`app_CurrentStepIntegrationCode`			
		,app.CustomerCode	 as 	`app_CustomerCode`			
		,app.FlowSegment	 as 	`app_FlowSegment`			
	,case when app.FlowSegmentDescription	= 'Less than 50k' then 'Less than 100k' 
		when app.FlowSegmentDescription = 'More than 50k' then 'More than 100k'
		else app.FlowSegmentDescription 
		end as 	`app_FlowSegmentDescription`      
		,app.UserCreatedQualcoGroupCode	 as 	`app_UserCreatedQualcoGroupCode`			
		,app.UserCreatedQualcoChannelCode	 as 	`app_UserCreatedQualcoChannelCode`		
		,app.UserCreatedGroupCategory	 as 	`app_UserCreatedGroupCategory`		
		,app.UserCreatedGroupCode	 as 	`app_UserCreatedUserCreatedGroupCode`		
		,app.UserCreatedGroupName	 as 	`app_UserCreatedGroupName`		
		,app.UserCreatedFullName	 as 	`app_UserCreatedFullName`
		,app.AssigneeUserName	 as 	`app_AssigneeUserName`	 
		,app.ContractSigningMethodCode	 as 	`app_ContractSigningMethodCode`
		,app.ContractSigningMethodDescription	 as 	`app_ContractSigningMethodDescription`		
		,app.SourceApplicationId `galaxyApplicationMetadata_SourceApplicationId`
		,app.SourceApplicationKey `galaxyApplicationMetadata_SourceApplicationKey`
		,app.galaxyApplicationMetadata_ApplicationStatus AS `galaxyApplicationMetadata_ApplicationStatus`
		,app.galaxyApplicationMetadata_ApplicationStatusDescription AS `galaxyApplicationMetadata_ApplicationStatusDescription`
		,app.CreditInvolvementStatus `galaxyApplicationMetadata_CreditInvolvementStatus`
		,app.CreditInvolvementStatusDescription `galaxyApplicationMetadata_CreditInvolvementStatusDescription`
		,app.ParentStepGroupingId `galaxyApplicationMetadata_ParentStepGroupingId`
		,app.ParentStepGroupingDescription `galaxyApplicationMetadata_ParentStepGroupingDescription`
		,app.ParentStepGroupingOwnerId `galaxyApplicationMetadata_ParentStepGroupingOwnerId`
		,app.ParentStepGroupingOwnerName `galaxyApplicationMetadata_ParentStepGroupingOwnerName`
		,app.StepGroupingId `galaxyApplicationMetadata_StepGroupingId`
		,app.StepGroupingDescription `galaxyApplicationMetadata_StepGroupingDescription`
		,app.StepGroupingOwnerId `galaxyApplicationMetadata_StepGroupingOwnerId`
		,app.StepGroupingOwnerName `galaxyApplicationMetadata_StepGroupingOwnerName`
		,app.ClientResponseId `galaxyApplicationMetadata_ClientResponseId`
		,app.ClientResponseDescription `galaxyApplicationMetadata_ClientResponseDescription`
		,app.OcwSimulationVoteId `galaxyApplicationMetadata_OcwSimulationVoteId`
		,app.OcwSimulationVoteDescription `galaxyApplicationMetadata_OcwSimulationVoteDescription`
		,app.QualcoAppStartedDate `galaxyApplicationMetadata_QualcoAppStartedDate`
		,app.QualcoAppSubmittedDate `galaxyApplicationMetadata_QualcoAppSubmittedDate`
		,app.QualcoAppPreApprovedDate `galaxyApplicationMetadata_QualcoAppPreApprovedDate`
		,app.QualcoAppApprovedDate `galaxyApplicationMetadata_QualcoAppApprovedDate`
		,app.QualcoAppDisbursedDate `galaxyApplicationMetadata_QualcoAppDisbursedDate`
		,app.QualcoAppRejectedDate `galaxyApplicationMetadata_QualcoAppRejectedDate`
		,app.QualcoAppCancelledDate `galaxyApplicationMetadata_QualcoAppCancelledDate`
		,app.QualcoAppFinalStepDate  `galaxyApplicationMetadata_QualcoAppFinalStepDate`
		,app.DateCreated AS `galaxyApplicationMetadata_DateCreated`
		,app.DateUpdated AS `galaxyApplicationMetadata_DateUpdated`
		,app.IsActive `galaxyApplicationMetadata_IsActive`
		,app.CurrentStepAssignmentDate as `app_CurrentStepAssignmentDate`
		,app.InterbankFlag as `app_InterbankFlag`
		,app.OcwFlag as `app_OcwFlag`
		,app.ManagedByPortfolioGroupName as `app_ManagedByPortfolioGroupName`
		,app.ManagedByPortfolioGroupCode as `app_ManagedByPortfolioGroupCode`
		,app.ContactSigningPlaceId as `app_ContactSigningPlaceId`
		,app.ContactSigningPlaceName as `app_ContactSigningPlaceName`
		,app.ContactSigningPlaceTypeId as `app_ContactSigningPlaceTypeId`
		,app.ContactSigningPlaceTypeName as `app_ContactSigningPlaceTypeName`
		,app.galaxyApplicationMetadata_StepReportingName as `app_galaxyApplicationMetadata_StepReportingName`
		,app.galaxyApplicationMetadata_StepActiveDuration as `app_galaxyApplicationMetadata_StepActiveDuration`
		,app.galaxyApplicationMetadata_stepReportingNameId as `app_galaxyApplicationMetadata_stepReportingNameId`
		,app.galaxyApplicationMetadata_ApplicationImplementationStatus as `ApplicationImplementationStatus`
		,app.galaxyApplicationMetadata_ApplicationActivationDate as `ApplicationActivationDate`

	,modification.Id	 as 	`modification_Id`
		,modification.CustomerId	 as 	`modification_CustomerId`
		,modification.ModificationId	 as 	`modification_ModificationId`
		,modification.ModificationRequestKey	 as 	`modification_ModificationRequestKey`
		,modification.UserName	 as 	`modification_UserName`
		,modification.Fullname	 as 	`modification_Fullname`
		,modification.Channel	 as 	`modification_Channel`
		,modification.ChannelDescription	 as 	`modification_ChannelDescription`
		,modification.DateOfCalculation	 as 	`modification_DateOfCalculation`
		,modification.IsSuccess	 as 	`modification_IsSuccess`
		,modification.ErrorCode	 as 	`modification_ErrorCode`
		,modification.MinDisposableIncomeForViableSolutions	 as 	`modification_MinDisposableIncomeForViableSolutions`
		,modification.MinDisposableIncomeForNonViableSolutions	 as 	`modification_MinDisposableIncomeForNonViableSolutions`
		,modification.IsViableSolution	 as 	`modification_IsViableSolution`
		,modification.CalculationMode	 as 	`modification_CalculationMode`
		,modification.CalculationModeDescription	 as 	`modification_CalculationModeDescription`
		,modification.UnitCode	 as 	`modification_UnitCode`
		,modification.UnitName	 as 	`modification_UnitName`
		,modification.UnitDescription	 as 	`modification_UnitDescription`
		,modification.AutoCreditReasonsVisible	 as 	`modification_AutoCreditReasonsVisible`
		,modification.ExtendedLimits	 as 	`modification_ExtendedLimits`
		,modification.ExtendedLimitsDescription	 as 	`modification_ExtendedLimitsDescription`
		,modification.CalculatorServiceId	 as 	`modification_CalculatorServiceId`
		,modification.ModificationAmendmentId	 as 	`modification_ModificationAmendmentId`
		,modification.CreditPolicyId	 as 	`modification_CreditPolicyId`
		,modification.CreditPolicyCode	 as 	`modification_CreditPolicyCode`
		,modification.CreditPolicyName	 as 	`modification_CreditPolicyName`
		,modification.CreditPolicyDescription	 as 	`modification_CreditPolicyDescription`

	,customer.Id	 as 	`customer_Id`
		,customer.AssetClass	 as 	`customer_AssetClass`
		,customer.BalanceCategory	 as 	`customer_BalanceCategory`
		,customer.BeginMonth	 as 	`customer_BeginMonth`
		,customer.BorrowerIndustry	 as 	`customer_BorrowerIndustry`
		,customer.BPSegment	 as 	`customer_BPSegment`
		,customer.BusinessPlanStrategies	 as 	`customer_BusinessPlanStrategies`
		,customer.CustomerFatherName	 as 	`customer_CustomerFatherName`
		,customer.CustomerFirstName	 as 	`customer_CustomerFirstName`
		,customer.DebtorBankGroupID	 as 	`customer_DebtorBankGroupID`
		,customer.DT_DebtLpsStatus	 as 	`customer_DT_DebtLpsStatus`
		,customer.DT_RealEstateSearchStatus	 as 	`customer_DT_RealEstateSearchStatus`
		,customer.DT_Segment	 as 	`customer_DT_Segment`
		,customer.EAPSubStatus	 as 	`customer_EAPSubStatus`
		,customer.GroupCode	 as 	`customer_GroupCode`
		,customer.InvestorSegment	 as 	`customer_InvestorSegment`
		,customer.NegativeEAP	 as 	`customer_NegativeEAP`
		,customer.Occupation	 as 	`customer_Occupation`
		,customer.PositiveEAP	 as 	`customer_PositiveEAP`
		,customer.StrategyID	 as 	`customer_StrategyID`
		,customer.UC_ID	 as 	`customer_UC_ID`
		,customer.BankruptcyFlag	 as 	`customer_BankruptcyFlag`
		,customer.BorrowerIndustryCode	 as 	`customer_BorrowerIndustryCode`
		,customer.ConfidenceID	 as 	`customer_ConfidenceID`
		,customer.ConnectedBorrowers	 as 	`customer_ConnectedBorrowers`
		,customer.ContactActivities	 as 	`customer_ContactActivities`
		,customer.ContactActivitiesLast3Months	 as 	`customer_ContactActivitiesLast3Months`
		,customer.ContactActivitiesLastMonth	 as 	`customer_ContactActivitiesLastMonth`
		,customer.CurrentQualcoSegmentRFIs	 as 	`customer_CurrentQualcoSegmentRFIs`
		,customer.CurrentQualcoSegmentRFIsDescription	 as 	`customer_CurrentQualcoSegmentRFIsDescription`
		,customer.CustomerSubSegmentIndication	 as 	`customer_CustomerSubSegmentIndication`
		,customer.CustomerSubSegmentIndicationDescription	 as 	`customer_CustomerSubSegmentIndicationDescription`
		,customer.DebtorLPS	 as 	`customer_DebtorLPS`
		,customer.DebtorLPSDescription	 as 	`customer_DebtorLPSDescription`
		,customer.DraftedByID	 as 	`customer_DraftedByID`
		,customer.DT_BorrowerType	 as 	`customer_DT_BorrowerType`
		,customer.DT_DenouncementFlag	 as 	`customer_DT_DenouncementFlag`
		,customer.DT_ExtrajudicialLetterFlag	 as 	`customer_DT_ExtrajudicialLetterFlag`
		,customer.DT_PaymentOrderFlag	 as 	`customer_DT_PaymentOrderFlag`
		,customer.FinalMaxBucketPerPortfpolio	 as 	`customer_FinalMaxBucketPerPortfpolio`
		,customer.Gender	 as 	`customer_Gender`
		,customer.LeverageSegment	 as 	`customer_LeverageSegment`
		,customer.MaritalStatus	 as 	`customer_MaritalStatus`
		,customer.NegativeEAPIndicator	 as 	`customer_NegativeEAPIndicator`
		,customer.NegativeRightPersonContactsLast3Months	 as 	`customer_NegativeRightPersonContactsLast3Months`
		,customer.NegativeRightPersonContactsLastMonth	 as 	`customer_NegativeRightPersonContactsLastMonth`
		,customer.NegativeRightPersonContactsSinceLTSA	 as 	`customer_NegativeRightPersonContactsSinceLTSA`
		,customer.PendingEAPNb	 as 	`customer_PendingEAPNb`
		,customer.PositiveEAPIndicator	 as 	`customer_PositiveEAPIndicator`
		,customer.PositiveRightPersonContactsLast3Months	 as 	`customer_PositiveRightPersonContactsLast3Months`
		,customer.PositiveRightPersonContactsLastMonth	 as 	`customer_PositiveRightPersonContactsLastMonth`
		,customer.PositiveRightPersonContactsSinceLTSA	 as 	`customer_PositiveRightPersonContactsSinceLTSA`
		,customer.PostCode	 as 	`customer_PostCode`
		,customer.PromisesLast3Months	 as 	`customer_PromisesLast3Months`
		,customer.PromisesLastMonth	 as 	`customer_PromisesLastMonth`
		,customer.PromisesSinceLTSA	 as 	`customer_PromisesSinceLTSA`
		,customer.ProposedByID	 as 	`customer_ProposedByID`
		,customer.RightPersonContactsLast3Months	 as 	`customer_RightPersonContactsLast3Months`
		,customer.RightPersonContactsLastMonth	 as 	`customer_RightPersonContactsLastMonth`
		,customer.RightPersonContactsSinceLTSA	 as 	`customer_RightPersonContactsSinceLTSA`
		,customer.SolutionTypeID	 as 	`customer_SolutionTypeID`
		,customer.StatusID	 as 	`customer_StatusID`
		,customer.TotalAssets	 as 	`customer_TotalAssets`
		,customer.WatchListType	 as 	`customer_WatchListType`
		,customer.CREV	 as 	`customer_CREV`
		,customer.DT_BorrowerLast12MonthsPayments	 as 	`customer_DT_BorrowerLast12MonthsPayments`
		,customer.DT_BorrowerOnBalance	 as 	`customer_DT_BorrowerOnBalance`
		,customer.DT_BorrowerTotalBalance	 as 	`customer_DT_BorrowerTotalBalance`
		,customer.OffBalanceDebtor	 as 	`customer_OffBalanceDebtor`
		,customer.OnBalanceDebtor	 as 	`customer_OnBalanceDebtor`
		,customer.TotalBalanceDebtor	 as 	`customer_TotalBalanceDebtor`
		,customer.TotalPaidAmountLast12Months	 as 	`customer_TotalPaidAmountLast12Months`
		,customer.TotalPaidAmountLast3Months	 as 	`customer_TotalPaidAmountLast3Months`
		,customer.TotalPaidAmountLastMonth	 as 	`customer_TotalPaidAmountLastMonth`
		,customer.TotalPaidAmountSinceLTSA	 as 	`customer_TotalPaidAmountSinceLTSA`
		,customer.ApplicationDate	 as 	`customer_ApplicationDate`
		,customer.CompletionTargetDate	 as 	`customer_CompletionTargetDate`
		,customer.ContactActivitiesLastDate	 as 	`customer_ContactActivitiesLastDate`
		,customer.DT_DenouncementDate	 as 	`customer_DT_DenouncementDate`
		,customer.IntentionDate	 as 	`customer_IntentionDate`
		,customer.LastPaymentDate	 as 	`customer_LastPaymentDate`
		,customer.NegativeRightPersonContactLastDate	 as 	`customer_NegativeRightPersonContactLastDate`
		,customer.PositiveRightPersonContactLastDate	 as 	`customer_PositiveRightPersonContactLastDate`
		,customer.PromiseLastDate	 as 	`customer_PromiseLastDate`
		,customer.RPCLastDate	 as 	`customer_RPCLastDate`
		,customer.PortfolioCode	 as 	`customer_PortfolioCode`
		,customer.PortfolioName	 as 	`customer_PortfolioName`
		,customer.PortfolioDescription	 as 	`customer_PortfolioDescription`
		,customer.InvestorCode	 as 	`customer_InvestorCode`
		,customer.InvestorDescription	 as 	`customer_InvestorName`
		,customer.InvestorDescription	 as 	`customer_InvestorDescription`
		,customer.CustomerCode	 as 	`customer_CustomerCode`
		,customer.DateOfBirth	 as 	`customer_DateOfBirth`
		,customer.VATNumber	 as 	`customer_VATNumber`
		,customer.CustomerName	 as 	`customer_CustomerName`
		,customer.CustomerType	 as 	`customer_CustomerType`
		,customer.CustomerTypeDescription AS `customer_CustomerTypeDescription`
		,customer.MonthlyInstallmentsOtherBanks	 as 	`customer_MonthlyInstallmentsOtherBanks`
		,customer.MonthlyInstallmentsOtherBanksCurrency	 as 	`customer_MonthlyInstallmentsOtherBanksCurrency`
		,customer.MonthlyInstallmentsOtherBanksInBaseCurrency	 as 	`customer_MonthlyInstallmentsOtherBanksInBaseCurrency`
		,customer.DepositsAmountTotal	 as 	`customer_DepositsAmountTotal`
		,customer.DepositsAmountTotalCurrency	 as 	`customer_DepositsAmountTotalCurrency`
		,customer.DepositsAmountTotalInBaseCurrency	 as 	`customer_DepositsAmountTotalInBaseCurrency`
		,customer.TimeDepositsAmountTotal	 as 	`customer_TimeDepositsAmountTotal`
		,customer.TimeDepositsAmountTotalCurrency	 as 	`customer_TimeDepositsAmountTotalCurrency`
		,customer.TimeDepositsAmountTotalInBaseCurrency	 as 	`customer_TimeDepositsAmountTotalInBaseCurrency`
		,customer.InvestmentsAmountTotal	 as 	`customer_InvestmentsAmountTotal`
		,customer.InvestmentsAmountTotalCurrency	 as 	`customer_InvestmentsAmountTotalCurrency`
		,customer.InvestmentsAmountTotalInBaseCurrency	 as 	`customer_InvestmentsAmountTotalInBaseCurrency`
		,customer.IsDeceased	 as 	`customer_IsDeceased`
		,customer.IsDeceasedDesctiption	 as 	`customer_IsDeceasedDesctiption`
		,customer.WholesaleIndicator	 as 	`customer_WholesaleIndicator`
		,customer.InsolvencyIndicator	 as 	`customer_InsolvencyIndicator`
		,customer.PrivateBankingIndicator	 as 	`customer_PrivateBankingIndicator`
		,customer.StaffIndicator	 as 	`customer_StaffIndicator`
		,customer.GroupID	 as 	`customer_GroupID`
		,customer.BankruptcyFlagDescription	 as 	`customer_BankruptcyFlagDescription`
		,customer.BorrowerIndustryCodeDescription	 as 	`customer_BorrowerIndustryCodeDescription`
		,customer.ConnectedBorrowersDescription	 as 	`customer_ConnectedBorrowersDescription`
		,customer.DtBorrowerTypeDescription	 as 	`customer_DtBorrowerTypeDescription`
		,customer.DtDenouncementFlagDescription	 as 	`customer_DtDenouncementFlagDescription`
		,customer.DtExtrajudicialLetterFlagDescription	 as 	`customer_DtExtrajudicialLetterFlagDescription`
		,customer.DtPaymentOrderFlagDescription	 as 	`customer_DtPaymentOrderFlagDescription`
		,customer.FinalMaxBucketPerPortfpolioDescription	 as 	`customer_FinalMaxBucketPerPortfpolioDescription`
		,customer.GenderDescription	 as 	`customer_GenderDescription`
		,customer.MaritalStatusDescription	 as 	`customer_MaritalStatusDescription`
		,customer.NegativeEapindicatorDescription	 as 	`customer_NegativeEapindicatorDescription`
		,customer.PositiveEapindicatorDescription	 as 	`customer_PositiveEapindicatorDescription`
		,customer.PostCodeDescription	 as 	`customer_PostCodeDescription`
		,customer.SolutionTypeIdDescription	 as 	`customer_SolutionTypeIdDescription`
		,customer.StatusIdDescription	 as 	`customer_StatusIdDescription`
		,customer.WatchListTypeDescription	 as 	`customer_WatchListTypeDescription`
		,customer.WholesaleIndicatorDescription	 as 	`customer_WholesaleIndicatorDescription`
		,customer.InsolvencyIndicatorDescription	 as 	`customer_InsolvencyIndicatorDescription`
		,customer.PrivateBankingIndicatorDescription	 as 	`customer_PrivateBankingIndicatorDescription`
		,customer.StaffIndicatorDescription	 as 	`customer_StaffIndicatorDescription`
		,customer.PoliticallyExposedPersons	 as 	`customer_PoliticallyExposedPersons`
		,customer.PoliticallyExposedPersonsDescription	 as 	`customer_PoliticallyExposedPersonsDescription`
		,customer.CustomerLastName	 as 	`customer_CustomerLastName`
		,customer.CompanyName	 as 	`customer_CompanyName`
		,customer.Payer	 as 	`customer_Payer`
		,customer.PayerDescription	 as 	`customer_PayerDescription`
		,customer.InterbankIndicator	 as 	`customer_InterbankIndicator`
		,customer.InterbankIndicatorDescription	 as 	`customer_InterbankIndicatorDescription`
		,customer.InterbankPaymentAmount	 as `customer_InterbankPaymentAmount`
		,customer.PostalAddress	 as 	`customer_PostalAddress`
		,customer.MobilePhone	 as 	`customer_MobilePhone`
		,customer.IdNumber	 as 	`customer_IdNumber`
		,customer.Email	 as 	`customer_Email`
		,customer.HomePhoneNumber	 as 	`customer_HomePhoneNumber`
		,customer.WorkPhoneNumber	 as 	`customer_WorkPhoneNumber`
		,customer.IsCovid19AffectedIndicator	 as 	`customer_IsCovid19AffectedIndicator`
		,customer.IsDEKIndicator	 as 	`customer_IsDEKIndicator`
		,customer.IsLaw3869Indicator	 as 	`customer_IsLaw3869Indicator`
		,customer.IsPEPIndicator	 as 	`customer_IsPEPIndicator`
		,customer.DeclaredAffordability	 as 	`customer_DeclaredAffordability`
		,customer.StatedAffordability	 as 	`customer_StatedAffordability`
		,customer.IncomeUsedForCalculations	 as 	`customer_IncomeUsedForCalculations`
		,customer.LegalRepresentativeExpirationDate as `customer_LegalRepresentativeExpirationDate`

	,loan.Id	 as 	loan_Id
		,loan.CustomerId	 as 	loan_CustomerId
		,loan.AssignmentChannel	 as 	loan_AssignmentChannel
		,loan.AssignmentOfficerName	 as 	loan_AssignmentOfficerName
		,loan.AssignmentType	 as 	loan_AssignmentType
		,loan.CapitalizationCV19	 as 	loan_CapitalizationCV19
		,loan.CurrentOccupation	 as 	loan_CurrentOccupation
		,loan.DebtLegalProcessStatus	 as 	loan_DebtLegalProcessStatus
		,loan.EBAStatusBeforCV19	 as 	loan_EBAStatusBeforCV19
		,loan.EBAStatusSOM	 as 	loan_EBAStatusSOM
		,loan.ProductName	 as 	loan_ProductName
		,loan.Queue	 as 	loan_Queue
		,loan.Strategy	 as 	loan_Strategy
		,loan.StrategyStepDescription	 as 	loan_StrategyStepDescription
		,loan.ActiveApplication	 as 	loan_ActiveApplication
		,loan.AmortizationType	 as 	loan_AmortizationType
		,loan.AssignmentTeam	 as 	loan_AssignmentTeam
		,loan.CaseWatchlList	 as 	loan_CaseWatchlList
		,loan.CollateralExistenceFlag	 as 	loan_CollateralExistenceFlag
		,loan.ContactActivities	 as 	loan_ContactActivities
		,loan.ContactActivitiesLast3Months	 as 	loan_ContactActivitiesLast3Months
		,loan.ContactActivitiesLastMonth	 as 	loan_ContactActivitiesLastMonth
		,loan.GracePeriodIndicator	 as 	loan_GracePeriodIndicator
		,loan.LastApplicationType	 as 	loan_LastApplicationType
		,loan.Law3869Type	 as 	loan_Law3869Type
		,loan.LegalProcessStatusDate	 as 	loan_LegalProcessStatusDate
		,loan.LoanIDSource	 as 	loan_LoanIDSource
		,loan.LT12bIndicator	 as 	loan_LT12bIndicator
		,loan.MinBucket	 as 	loan_MinBucket
		,loan.NegativeRightPersonContactsLast3Months	 as 	loan_NegativeRightPersonContactsLast3Months
		,loan.NegativeRightPersonContactsLastMonth	 as 	loan_NegativeRightPersonContactsLastMonth
		,loan.NegativeRightPersonContactsSinceLTSA	 as 	loan_NegativeRightPersonContactsSinceLTSA
		,loan.ObjectiveStatus	 as 	loan_ObjectiveStatus
		,loan.OperationalBucketBeforCV19	 as 	loan_OperationalBucketBeforCV19
		,loan.PerformingStatus	 as 	loan_PerformingStatus
		,loan.PositiveRightPersonContactsLast3Months	 as 	loan_PositiveRightPersonContactsLast3Months
		,loan.PositiveRightPersonContactsLastMonth	 as 	loan_PositiveRightPersonContactsLastMonth
		,loan.PositiveRightPersonContactsSinceLTSA	 as 	loan_PositiveRightPersonContactsSinceLTSA
		,loan.PreviousApplications	 as 	loan_PreviousApplications
		,loan.PromisesLast3Months	 as 	loan_PromisesLast3Months
		,loan.PromisesLastMonth	 as 	loan_PromisesLastMonth
		,loan.PromisesSinceLTSA	 as 	loan_PromisesSinceLTSA
		,loan.QueID	 as 	loan_QueID
		,loan.RemainingProbationMonths	 as 	loan_RemainingProbationMonths
		,loan.RightPersonContactsLast3Months	 as 	loan_RightPersonContactsLast3Months
		,loan.RightPersonContactsLastMonth	 as 	loan_RightPersonContactsLastMonth
		,loan.RightPersonContactsSinceLTSA	 as 	loan_RightPersonContactsSinceLTSA
		,loan.RiskBucketBeforeCV19	 as 	loan_RiskBucketBeforeCV19
		,loan.RiskBucketSOM	 as 	loan_RiskBucketSOM
		,loan.SettlementBucket	 as 	loan_SettlementBucket
		,loan.SPV	 as 	loan_SPV
		,loan.AssignmentBalance	 as 	loan_AssignmentBalance
		,loan.CreditLine	 as 	loan_CreditLine
		,loan.DT_AccountOnBalance	 as 	loan_DT_AccountOnBalance
		,loan.DT_AccountTotalBalance	 as 	loan_DT_AccountTotalBalance
		,loan.DT_Last12MonthsPayments	 as 	loan_DT_Last12MonthsPayments
		,loan.OffBalance	 as 	loan_OffBalance
		,loan.OnBalance	 as 	loan_OnBalance
		,loan.TotalPaidAmountLast12Months	 as 	loan_TotalPaidAmountLast12Months
		,loan.TotalPaidAmountLast3Months	 as 	loan_TotalPaidAmountLast3Months
		,loan.TotalPaidAmountLastMonth	 as 	loan_TotalPaidAmountLastMonth
		,loan.TotalPaidAmountSinceLTSA	 as 	loan_TotalPaidAmountSinceLTSA
		,loan.AccountStatusChageDate	 as 	loan_AccountStatusChageDate
		,loan.AssignmentDate	 as 	loan_AssignmentDate
		,loan.ContactActivitiesLastDate	 as 	loan_ContactActivitiesLastDate
		,loan.FixedInterestEndDate	 as 	loan_FixedInterestEndDate
		,loan.L3869CourtDate	 as 	loan_L3869CourtDate
		,loan.LastApplicationDate	 as 	loan_LastApplicationDate
		,loan.LastPaymentDate	 as 	loan_LastPaymentDate
		,loan.NegativeRightPersonContactLastDate	 as 	loan_NegativeRightPersonContactLastDate
		,loan.ObjectiveStatusDate	 as 	loan_ObjectiveStatusDate
		,loan.PositiveRightPersonContactLastDate	 as 	loan_PositiveRightPersonContactLastDate
		,loan.ProbationStartDate	 as 	loan_ProbationStartDate
		,loan.PromiseLastDate	 as 	loan_PromiseLastDate
		,loan.RightPersonContactLastDate	 as 	loan_RightPersonContactLastDate
		,loan.OpenFlag	 as 	loan_OpenFlag
		,loan.AccountNumber	 as 	loan_AccountNumber
		,loan.AssetClass	 as 	loan_AssetClass
		,loan.AssetClassDesciption	 as 	loan_AssetClassDesciption
		,loan.AssociationType	 as 	loan_AssociationType
		,loan.AssociationTypeDescription	 as 	loan_AssociationTypeDescription
		,loan.ContractNumber	 as 	loan_ContractNumber
		,loan.IsAutomobile	 as 	loan_IsAutomobile
		,loan.HasGovernmentGuarantee	 as 	loan_HasGovernmentGuarantee
		,loan.OutstandingBalance	 as 	loan_OutstandingBalance
		,loan.OutstandingBalanceInBaseCurrency	 as 	loan_OutstandingBalanceInBaseCurrency
		,loan.RemainingTenorInMonths	 as 	loan_RemainingTenorInMonths
		,loan.Bucket	 as 	loan_Bucket
		,loan.CurrentInterestRate	 as 	loan_CurrentInterestRate
		,loan.EBAStatus	 as 	loan_EBAStatus
		,loan.EBAStatusDescription	 as 	loan_EBAStatusDescription
		,loan.Currency	 as 	loan_Currency
		,loan.CurrentInstallment	 as 	loan_CurrentInstallment
		,loan.CurrentInstallmentInBaseCurrency	 as 	loan_CurrentInstallmentInBaseCurrency
		,loan.DaysPastDue	 as 	loan_DaysPastDue
		,loan.DebtAmount	 as 	loan_DebtAmount
		,loan.DebtAmountInBaseCurrency	 as 	loan_DebtAmountInBaseCurrency
		,loan.FrozenInterest	 as 	loan_FrozenInterest
		,loan.FrozenInterestInBaseCurrency	 as 	loan_FrozenInterestInBaseCurrency
		,loan.NotAccrudInterest	 as 	loan_NotAccrudInterest
		,loan.NotAccrudInterestInBaseCurrency	 as 	loan_NotAccrudInterestInBaseCurrency
		,loan.DepreciatedAmount	 as 	loan_DepreciatedAmount
		,loan.DepreciatedAmountInBaseCurrency	 as 	loan_DepreciatedAmountInBaseCurrency
		,loan.Expenses	 as 	loan_Expenses
		,loan.ExpensesInBaseCurrency	 as 	loan_ExpensesInBaseCurrency
		,loan.HasLegalActions	 as 	loan_HasLegalActions
		,loan.BorrowerAccountSegment	 as 	loan_BorrowerAccountSegment
		,loan.NumberOfPreviousModifications	 as 	loan_NumberOfPreviousModifications
		,loan.MaxDaysPastDueEver	 as 	loan_MaxDaysPastDueEver
		,loan.LegalProcessStatus	 as 	loan_LegalProcessStatus
		,loan.DenouncedCaseEver	 as 	loan_DenouncedCaseEver
		,loan.IsRevolvingLoan	 as 	loan_IsRevolvingLoan
		,loan.LoanOriginationDate	 as 	loan_LoanOriginationDate
		,loan.ProductCode	 as 	loan_ProductCode
		,loan.MonthsSinceMostRecentModification	 as 	loan_MonthsSinceMostRecentModification
		,loan.AccountStatus	 as 	loan_AccountStatus
		,loan.AccountStatusChangeDate	 as 	loan_AccountStatusChangeDate
		,loan.PermanentDelayStatusTurnDate	 as 	loan_PermanentDelayStatusTurnDate
		,loan.CardNumber	 as 	loan_CardNumber
		,loan.IsDenounced	 as 	loan_IsDenounced
		,loan.BaseInterest	 as 	loan_BaseInterest
		,loan.Law128Rate	 as 	loan_Law128Rate
		,loan.Spread	 as 	loan_Spread
		,loan.IsFixedInterest	 as 	loan_IsFixedInterest
		,loan.ContractualInterestRate	 as 	loan_ContractualInterestRate
		,loan.CurrentInterestAmount	 as 	loan_CurrentInterestAmount
		,loan.CurrentInterestAmountInBaseCurrency	 as 	loan_CurrentInterestAmountInBaseCurrency
		,loan.AgencyType	 as 	loan_AgencyType
		,loan.BaseInterestType	 as 	loan_BaseInterestType
		,loan.Delivered	 as 	loan_Delivered
		,loan.AccountOwnerId	 as 	loan_AccountOwnerId
		,loan.LoanPurpose	 as 	loan_LoanPurpose
		,loan.ParticipationPercentage	 as 	loan_ParticipationPercentage
		,loan.ExchangeRateAtOrigination	 as 	loan_ExchangeRateAtOrigination
		,loan.IsOverdraft	 as 	loan_IsOverdraft
		,loan.CreditLimit	 as 	loan_CreditLimit
		,loan.CreditLimitInBaseCurrency	 as 	loan_CreditLimitInBaseCurrency
		,loan.ActiveApplicationDescription	 as 	loan_ActiveApplicationDescription
		,loan.AmortizationTypeDescription	 as 	loan_AmortizationTypeDescription
		,loan.AssignmentTeamDescription	 as 	loan_AssignmentTeamDescription
		,loan.CaseWatchlListDescription	 as 	loan_CaseWatchlListDescription
		,loan.GracePeriodIndicatorDescription	 as 	loan_GracePeriodIndicatorDescription
		,loan.LastApplicationTypeDescription	 as 	loan_LastApplicationTypeDescription
		,loan.Law3869TypeDescription	 as 	loan_Law3869TypeDescription
		,loan.LegalProcessStatusDateDescription	 as 	loan_LegalProcessStatusDateDescription
		,loan.Lt12bIndicatorDescription	 as 	loan_Lt12bIndicatorDescription
		,loan.MinBucketDescription	 as 	loan_MinBucketDescription
		,loan.ObjectiveStatusDescription	 as 	loan_ObjectiveStatusDescription
		,loan.PerformingStatusDescription	 as 	loan_PerformingStatusDescription
		,loan.QueIdDescription	 as 	loan_QueIdDescription
		,loan.SettlementBucketDescription	 as 	loan_SettlementBucketDescription
		,loan.SpvDescription	 as 	loan_SpvDescription
		,loan.IsAutomobileDescription	 as 	loan_IsAutomobileDescription
		,loan.HasGovernmentGuaranteeDescription	 as 	loan_HasGovernmentGuaranteeDescription
		,loan.HasLegalActionsDescription	 as 	loan_HasLegalActionsDescription
		,loan.BorrowerAccountSegmentDescription	 as 	loan_BorrowerAccountSegmentDescription
		,loan.LegalProcessStatusDescription	 as 	loan_LegalProcessStatusDescription
		,loan.DenounceCaseEverDescription	 as 	loan_DenounceCaseEverDescription
		,loan.IsRevolvingLoanDescription	 as 	loan_IsRevolvingLoanDescription
		,loan.IsDenouncedDescription	 as 	loan_IsDenouncedDescription
		,loan.IsFixedInterestDescription	 as 	loan_IsFixedInterestDescription
		,loan.AgencyTypeDescription	 as 	loan_AgencyTypeDescription
		,loan.BaseInterestTypeDescription	 as 	loan_BaseInterestTypeDescription
		,loan.DeliveredDescription	 as 	loan_DeliveredDescription
		,loan.LoanPurposeDescription	 as 	loan_LoanPurposeDescription
		,loan.IsOverdraftDescription	 as 	loan_IsOverdraftDescription
		,loan.PositiveLoanEAPIndicator	 as 	loan_PositiveLoanEAPIndicator
		,loan.PositiveLoanEAPIndicatorDescription	 as 	loan_PositiveLoanEAPIndicatorDescription
		,loan.ContractNumberSContractNumberSystemicNrystemicNr	 as 	loan_ContractNumberSContractNumberSystemicNrystemicNr
		,loan.EligiblePrenotedAsset	 as 	loan_EligiblePrenotedAsset
		,loan.EligiblePrenotedAssetDescription	 as 	loan_EligiblePrenotedAssetDescription
		,loan.AccountPayer	 as 	loan_AccountPayer
		,loan.AccountPayerDescription	 as 	loan_AccountPayerDescription
		,loan.AccountNumberWithPrefix	 as 	loan_AccountNumberWithPrefix
		,loan.ContractNumberSystemicNr	 as 	loan_ContractNumberSystemicNr
		,loan.UnifiedLoanId	 as 	loan_UnifiedLoanId
		,loan.IsUnifiedLoan	 as 	loan_IsUnifiedLoan
		,loan.DecisionTreeCode	 as 	loan_DecisionTreeCode
		,loan.DecisionTreeDescription	 as 	loan_DecisionTreeDescription
		,loan.DecisionTreeValidPath	 as 	loan_DecisionTreeValidPath
		,loan.IsForModification	 as 	loan_IsForModification

	,case when unifiedLoan.Id is null then loan.Id else unifiedLoan.Id end as 	unifiedLoan_Id
		,case when unifiedLoan.Id is null then loan.AssignmentChannel else unifiedLoan.AssignmentChannel end	 as 	unifiedLoan_AssignmentChannel
		,case when unifiedLoan.Id is null then Loan.AssignmentOfficerName else unifiedLoan.AssignmentOfficerName	 end as 	unifiedLoan_AssignmentOfficerName
		,case when unifiedLoan.Id is null then Loan.AssignmentType else unifiedLoan.AssignmentType	 end as 	unifiedLoan_AssignmentType
		,case when unifiedLoan.Id is null then Loan.CapitalizationCV19 else unifiedLoan.CapitalizationCV19	 end as 	unifiedLoan_CapitalizationCV19
		,case when unifiedLoan.Id is null then Loan.CurrentOccupation else unifiedLoan.CurrentOccupation	 end as 	unifiedLoan_CurrentOccupation
		,case when unifiedLoan.Id is null then Loan.DebtLegalProcessStatus else unifiedLoan.DebtLegalProcessStatus	 end as 	unifiedLoan_DebtLegalProcessStatus
		,case when unifiedLoan.Id is null then Loan.EBAStatusBeforCV19 else unifiedLoan.EBAStatusBeforCV19	 end as 	unifiedLoan_EBAStatusBeforCV19
		,case when unifiedLoan.Id is null then Loan.EBAStatusSOM else unifiedLoan.EBAStatusSOM	 end as 	unifiedLoan_EBAStatusSOM
		,case when unifiedLoan.Id is null then Loan.ProductName else unifiedLoan.ProductName	 end as 	unifiedLoan_ProductName
		,case when unifiedLoan.Id is null then Loan.Queue else unifiedLoan.Queue	 end as 	unifiedLoan_Queue
		,case when unifiedLoan.Id is null then Loan.Strategy else unifiedLoan.Strategy	 end as 	unifiedLoan_Strategy
		,case when unifiedLoan.Id is null then Loan.StrategyStepDescription else unifiedLoan.StrategyStepDescription	 end as 	unifiedLoan_StrategyStepDescription
		,case when unifiedLoan.Id is null then Loan.ActiveApplication else unifiedLoan.ActiveApplication	 end as 	unifiedLoan_ActiveApplication
		,case when unifiedLoan.Id is null then Loan.AmortizationType else unifiedLoan.AmortizationType	 end as 	unifiedLoan_AmortizationType
		,case when unifiedLoan.Id is null then Loan.AssignmentTeam else unifiedLoan.AssignmentTeam	 end as 	unifiedLoan_AssignmentTeam
		,case when unifiedLoan.Id is null then Loan.CaseWatchlList else unifiedLoan.CaseWatchlList	 end as 	unifiedLoan_CaseWatchlList
		,case when unifiedLoan.Id is null then Loan.CollateralExistenceFlag else unifiedLoan.CollateralExistenceFlag	 end as 	unifiedLoan_CollateralExistenceFlag
		,case when unifiedLoan.Id is null then Loan.ContactActivities else unifiedLoan.ContactActivities	 end as 	unifiedLoan_ContactActivities
		,case when unifiedLoan.Id is null then Loan.ContactActivitiesLast3Months else unifiedLoan.ContactActivitiesLast3Months	 end as 	unifiedLoan_ContactActivitiesLast3Months
		,case when unifiedLoan.Id is null then Loan.ContactActivitiesLastMonth else unifiedLoan.ContactActivitiesLastMonth	 end as 	unifiedLoan_ContactActivitiesLastMonth
		,case when unifiedLoan.Id is null then Loan.GracePeriodIndicator else unifiedLoan.GracePeriodIndicator	 end as 	unifiedLoan_GracePeriodIndicator
		,case when unifiedLoan.Id is null then Loan.LastApplicationType else unifiedLoan.LastApplicationType	 end as 	unifiedLoan_LastApplicationType
		,case when unifiedLoan.Id is null then Loan.Law3869Type else unifiedLoan.Law3869Type	 end as 	unifiedLoan_Law3869Type
		,case when unifiedLoan.Id is null then Loan.LegalProcessStatusDate else unifiedLoan.LegalProcessStatusDate	 end as 	unifiedLoan_LegalProcessStatusDate
		,case when unifiedLoan.Id is null then Loan.LoanIDSource else unifiedLoan.LoanIDSource	 end as 	unifiedLoan_LoanIDSource
		,case when unifiedLoan.Id is null then Loan.LT12bIndicator else unifiedLoan.LT12bIndicator	 end as 	unifiedLoan_LT12bIndicator
		,case when unifiedLoan.Id is null then Loan.MinBucket else unifiedLoan.MinBucket	 end as 	unifiedLoan_MinBucket
		,case when unifiedLoan.Id is null then Loan.NegativeRightPersonContactsLast3Months else unifiedLoan.NegativeRightPersonContactsLast3Months	 end as 	unifiedLoan_NegativeRightPersonContactsLast3Months
		,case when unifiedLoan.Id is null then Loan.NegativeRightPersonContactsLastMonth else unifiedLoan.NegativeRightPersonContactsLastMonth	 end as 	unifiedLoan_NegativeRightPersonContactsLastMonth
		,case when unifiedLoan.Id is null then Loan.NegativeRightPersonContactsSinceLTSA else unifiedLoan.NegativeRightPersonContactsSinceLTSA	 end as 	unifiedLoan_NegativeRightPersonContactsSinceLTSA
		,case when unifiedLoan.Id is null then Loan.ObjectiveStatus else unifiedLoan.ObjectiveStatus	 end as 	unifiedLoan_ObjectiveStatus
		,case when unifiedLoan.Id is null then Loan.OperationalBucketBeforCV19 else unifiedLoan.OperationalBucketBeforCV19	 end as 	unifiedLoan_OperationalBucketBeforCV19
		,case when unifiedLoan.Id is null then Loan.PerformingStatus else unifiedLoan.PerformingStatus	 end as 	unifiedLoan_PerformingStatus
		,case when unifiedLoan.Id is null then Loan.PositiveRightPersonContactsLast3Months else unifiedLoan.PositiveRightPersonContactsLast3Months	 end as 	unifiedLoan_PositiveRightPersonContactsLast3Months
		,case when unifiedLoan.Id is null then Loan.PositiveRightPersonContactsLastMonth else unifiedLoan.PositiveRightPersonContactsLastMonth	 end as 	unifiedLoan_PositiveRightPersonContactsLastMonth
		,case when unifiedLoan.Id is null then Loan.PositiveRightPersonContactsSinceLTSA else unifiedLoan.PositiveRightPersonContactsSinceLTSA	 end as 	unifiedLoan_PositiveRightPersonContactsSinceLTSA
		,case when unifiedLoan.Id is null then Loan.PreviousApplications else unifiedLoan.PreviousApplications	 end as 	unifiedLoan_PreviousApplications
		,case when unifiedLoan.Id is null then Loan.PromisesLast3Months else unifiedLoan.PromisesLast3Months	 end as 	unifiedLoan_PromisesLast3Months
		,case when unifiedLoan.Id is null then Loan.PromisesLastMonth else unifiedLoan.PromisesLastMonth	 end as 	unifiedLoan_PromisesLastMonth
		,case when unifiedLoan.Id is null then Loan.PromisesSinceLTSA else unifiedLoan.PromisesSinceLTSA	 end as 	unifiedLoan_PromisesSinceLTSA
		,case when unifiedLoan.Id is null then Loan.QueID else unifiedLoan.QueID	 end as 	unifiedLoan_QueID
		,case when unifiedLoan.Id is null then Loan.RemainingProbationMonths else unifiedLoan.RemainingProbationMonths	 end as 	unifiedLoan_RemainingProbationMonths
		,case when unifiedLoan.Id is null then Loan.RightPersonContactsLast3Months else unifiedLoan.RightPersonContactsLast3Months	 end as 	unifiedLoan_RightPersonContactsLast3Months
		,case when unifiedLoan.Id is null then Loan.RightPersonContactsLastMonth else unifiedLoan.RightPersonContactsLastMonth	 end as 	unifiedLoan_RightPersonContactsLastMonth
		,case when unifiedLoan.Id is null then Loan.RightPersonContactsSinceLTSA else unifiedLoan.RightPersonContactsSinceLTSA	 end as 	unifiedLoan_RightPersonContactsSinceLTSA
		,case when unifiedLoan.Id is null then Loan.RiskBucketBeforeCV19 else unifiedLoan.RiskBucketBeforeCV19	 end as 	unifiedLoan_RiskBucketBeforeCV19
		,case when unifiedLoan.Id is null then Loan.RiskBucketSOM else unifiedLoan.RiskBucketSOM	 end as 	unifiedLoan_RiskBucketSOM
		,case when unifiedLoan.Id is null then Loan.SettlementBucket else unifiedLoan.SettlementBucket	 end as 	unifiedLoan_SettlementBucket
		,case when unifiedLoan.Id is null then Loan.SPV else unifiedLoan.SPV	 end as 	unifiedLoan_SPV
		,case when unifiedLoan.Id is null then Loan.AssignmentBalance else unifiedLoan.AssignmentBalance	 end as 	unifiedLoan_AssignmentBalance
		,case when unifiedLoan.Id is null then Loan.CreditLine else unifiedLoan.CreditLine	 end as 	unifiedLoan_CreditLine
		,case when unifiedLoan.Id is null then Loan.DT_AccountOnBalance else unifiedLoan.DT_AccountOnBalance	 end as 	unifiedLoan_DT_AccountOnBalance
		,case when unifiedLoan.Id is null then Loan.DT_AccountTotalBalance else unifiedLoan.DT_AccountTotalBalance	 end as 	unifiedLoan_DT_AccountTotalBalance
		,case when unifiedLoan.Id is null then Loan.DT_Last12MonthsPayments else unifiedLoan.DT_Last12MonthsPayments	 end as 	unifiedLoan_DT_Last12MonthsPayments
		,case when unifiedLoan.Id is null then Loan.OffBalance else unifiedLoan.OffBalance	 end as 	unifiedLoan_OffBalance
		,case when unifiedLoan.Id is null then Loan.OnBalance else unifiedLoan.OnBalance	 end as 	unifiedLoan_OnBalance
		,case when unifiedLoan.Id is null then Loan.TotalPaidAmountLast12Months else unifiedLoan.TotalPaidAmountLast12Months	 end as 	unifiedLoan_TotalPaidAmountLast12Months
		,case when unifiedLoan.Id is null then Loan.TotalPaidAmountLast3Months else unifiedLoan.TotalPaidAmountLast3Months	 end as 	unifiedLoan_TotalPaidAmountLast3Months
		,case when unifiedLoan.Id is null then Loan.TotalPaidAmountLastMonth else unifiedLoan.TotalPaidAmountLastMonth	 end as 	unifiedLoan_TotalPaidAmountLastMonth
		,case when unifiedLoan.Id is null then Loan.TotalPaidAmountSinceLTSA else unifiedLoan.TotalPaidAmountSinceLTSA	 end as 	unifiedLoan_TotalPaidAmountSinceLTSA
		,case when unifiedLoan.Id is null then Loan.AccountStatusChageDate else unifiedLoan.AccountStatusChageDate	 end as 	unifiedLoan_AccountStatusChageDate
		,case when unifiedLoan.Id is null then Loan.AssignmentDate else unifiedLoan.AssignmentDate	 end as 	unifiedLoan_AssignmentDate
		,case when unifiedLoan.Id is null then Loan.ContactActivitiesLastDate else unifiedLoan.ContactActivitiesLastDate	 end as 	unifiedLoan_ContactActivitiesLastDate
		,case when unifiedLoan.Id is null then Loan.FixedInterestEndDate else unifiedLoan.FixedInterestEndDate	 end as 	unifiedLoan_FixedInterestEndDate
		,case when unifiedLoan.Id is null then Loan.L3869CourtDate else unifiedLoan.L3869CourtDate	 end as 	unifiedLoan_L3869CourtDate
		,case when unifiedLoan.Id is null then Loan.LastApplicationDate else unifiedLoan.LastApplicationDate	 end as 	unifiedLoan_LastApplicationDate
		,case when unifiedLoan.Id is null then Loan.LastPaymentDate else unifiedLoan.LastPaymentDate	 end as 	unifiedLoan_LastPaymentDate
		,case when unifiedLoan.Id is null then Loan.NegativeRightPersonContactLastDate else unifiedLoan.NegativeRightPersonContactLastDate	 end as 	unifiedLoan_NegativeRightPersonContactLastDate
		,case when unifiedLoan.Id is null then Loan.ObjectiveStatusDate else unifiedLoan.ObjectiveStatusDate	 end as 	unifiedLoan_ObjectiveStatusDate
		,case when unifiedLoan.Id is null then Loan.PositiveRightPersonContactLastDate else unifiedLoan.PositiveRightPersonContactLastDate	 end as 	unifiedLoan_PositiveRightPersonContactLastDate
		,case when unifiedLoan.Id is null then Loan.ProbationStartDate else unifiedLoan.ProbationStartDate	 end as 	unifiedLoan_ProbationStartDate
		,case when unifiedLoan.Id is null then Loan.PromiseLastDate else unifiedLoan.PromiseLastDate	 end as 	unifiedLoan_PromiseLastDate
		,case when unifiedLoan.Id is null then Loan.RightPersonContactLastDate else unifiedLoan.RightPersonContactLastDate	 end as 	unifiedLoan_RightPersonContactLastDate
		,case when unifiedLoan.Id is null then Loan.OpenFlag else unifiedLoan.OpenFlag	 end as 	unifiedLoan_OpenFlag
		--,case when unifiedLoan.Id is null then Loan.AccountNumber else unifiedLoan.DWHAccountNumber end as 	unifiedLoan_AccountNumber
  		,case when unifiedLoan.Id is null then Loan.AccountNumber else unifiedLoan.AccountNumber end as 	unifiedLoan_AccountNumber
		,case when unifiedLoan.Id is null then Loan.AssetClass else unifiedLoan.AssetClass	 end as 	unifiedLoan_AssetClass
		,case when unifiedLoan.Id is null then Loan.AssetClassDesciption else unifiedLoan.AssetClassDesciption	 end as 	unifiedLoan_AssetClassDesciption
		,case when unifiedLoan.Id is null then Loan.AssociationType else unifiedLoan.AssociationType	 end as 	unifiedLoan_AssociationType
		,case when unifiedLoan.Id is null then Loan.AssociationTypeDescription else unifiedLoan.AssociationTypeDescription	 end as 	unifiedLoan_AssociationTypeDescription
		,case when unifiedLoan.Id is null then Loan.ContractNumber else unifiedLoan.ContractNumber	 end as 	unifiedLoan_ContractNumber
		,case when unifiedLoan.Id is null then Loan.IsAutomobile else unifiedLoan.IsAutomobile	 end as 	unifiedLoan_IsAutomobile
		,case when unifiedLoan.Id is null then Loan.HasGovernmentGuarantee else unifiedLoan.HasGovernmentGuarantee	 end as 	unifiedLoan_HasGovernmentGuarantee
		,case when unifiedLoan.Id is null then Loan.OutstandingBalance else unifiedLoan.OutstandingBalance	 end as 	unifiedLoan_OutstandingBalance
		,case when unifiedLoan.Id is null then Loan.OutstandingBalanceInBaseCurrency else unifiedLoan.OutstandingBalanceInBaseCurrency	 end as 	unifiedLoan_OutstandingBalanceInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.RemainingTenorInMonths else unifiedLoan.RemainingTenorInMonths	 end as 	unifiedLoan_RemainingTenorInMonths
		,case when unifiedLoan.Id is null then Loan.Bucket else unifiedLoan.Bucket	 end as 	unifiedLoan_Bucket
		,case when unifiedLoan.Id is null then Loan.CurrentInterestRate else unifiedLoan.CurrentInterestRate	 end as 	unifiedLoan_CurrentInterestRate
		,case when unifiedLoan.Id is null then Loan.EBAStatus else unifiedLoan.EBAStatus	 end as 	unifiedLoan_EBAStatus
		,case when unifiedLoan.Id is null then Loan.EBAStatusDescription else unifiedLoan.EBAStatusDescription	 end as 	unifiedLoan_EBAStatusDescription
		,case when unifiedLoan.Id is null then Loan.Currency else unifiedLoan.Currency	 end as 	unifiedLoan_Currency
		,case when unifiedLoan.Id is null then Loan.CurrentInstallment else unifiedLoan.CurrentInstallment	 end as 	unifiedLoan_CurrentInstallment
		,case when unifiedLoan.Id is null then Loan.CurrentInstallmentInBaseCurrency else unifiedLoan.CurrentInstallmentInBaseCurrency	 end as 	unifiedLoan_CurrentInstallmentInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.DaysPastDue else unifiedLoan.DaysPastDue	 end as 	unifiedLoan_DaysPastDue
		,case when unifiedLoan.Id is null then Loan.DebtAmount else unifiedLoan.DebtAmount	 end as 	unifiedLoan_DebtAmount
		,case when unifiedLoan.Id is null then Loan.DebtAmountInBaseCurrency else unifiedLoan.DebtAmountInBaseCurrency	 end as 	unifiedLoan_DebtAmountInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.FrozenInterest else unifiedLoan.FrozenInterest	 end as 	unifiedLoan_FrozenInterest
		,case when unifiedLoan.Id is null then Loan.FrozenInterestInBaseCurrency else unifiedLoan.FrozenInterestInBaseCurrency	 end as 	unifiedLoan_FrozenInterestInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.NotAccrudInterest else unifiedLoan.NotAccrudInterest	 end as 	unifiedLoan_NotAccrudInterest
		,case when unifiedLoan.Id is null then Loan.NotAccrudInterestInBaseCurrency else unifiedLoan.NotAccrudInterestInBaseCurrency	 end as 	unifiedLoan_NotAccrudInterestInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.DepreciatedAmount else unifiedLoan.DepreciatedAmount	 end as 	unifiedLoan_DepreciatedAmount
		,case when unifiedLoan.Id is null then Loan.DepreciatedAmountInBaseCurrency else unifiedLoan.DepreciatedAmountInBaseCurrency	 end as 	unifiedLoan_DepreciatedAmountInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.Expenses else unifiedLoan.Expenses	 end as 	unifiedLoan_Expenses
		,case when unifiedLoan.Id is null then Loan.ExpensesInBaseCurrency else unifiedLoan.ExpensesInBaseCurrency	 end as 	unifiedLoan_ExpensesInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.HasLegalActions else unifiedLoan.HasLegalActions	 end as 	unifiedLoan_HasLegalActions
		,case when unifiedLoan.Id is null then Loan.BorrowerAccountSegment else unifiedLoan.BorrowerAccountSegment	 end as 	unifiedLoan_BorrowerAccountSegment
		,case when unifiedLoan.Id is null then Loan.NumberOfPreviousModifications else unifiedLoan.NumberOfPreviousModifications	 end as 	unifiedLoan_NumberOfPreviousModifications
		,case when unifiedLoan.Id is null then Loan.MaxDaysPastDueEver else unifiedLoan.MaxDaysPastDueEver	 end as 	unifiedLoan_MaxDaysPastDueEver
		,case when unifiedLoan.Id is null then Loan.LegalProcessStatus else unifiedLoan.LegalProcessStatus	 end as 	unifiedLoan_LegalProcessStatus
		,case when unifiedLoan.Id is null then Loan.DenouncedCaseEver else unifiedLoan.DenouncedCaseEver	 end as 	unifiedLoan_DenouncedCaseEver
		,case when unifiedLoan.Id is null then Loan.IsRevolvingLoan else unifiedLoan.IsRevolvingLoan	 end as 	unifiedLoan_IsRevolvingLoan
		,case when unifiedLoan.Id is null then Loan.LoanOriginationDate else unifiedLoan.LoanOriginationDate	 end as 	unifiedLoan_LoanOriginationDate
		,case when unifiedLoan.Id is null then Loan.ProductCode else unifiedLoan.ProductCode	 end as 	unifiedLoan_ProductCode
		,case when unifiedLoan.Id is null then Loan.MonthsSinceMostRecentModification else unifiedLoan.MonthsSinceMostRecentModification	 end as 	unifiedLoan_MonthsSinceMostRecentModification
		,case when unifiedLoan.Id is null then Loan.AccountStatus else unifiedLoan.AccountStatus	 end as 	unifiedLoan_AccountStatus
		,case when unifiedLoan.Id is null then Loan.AccountStatusChangeDate else unifiedLoan.AccountStatusChangeDate	 end as 	unifiedLoan_AccountStatusChangeDate
		,case when unifiedLoan.Id is null then Loan.PermanentDelayStatusTurnDate else unifiedLoan.PermanentDelayStatusTurnDate	 end as 	unifiedLoan_PermanentDelayStatusTurnDate
		,case when unifiedLoan.Id is null then Loan.CardNumber else unifiedLoan.CardNumber	 end as 	unifiedLoan_CardNumber
		,case when unifiedLoan.Id is null then Loan.IsDenounced else unifiedLoan.IsDenounced	 end as 	unifiedLoan_IsDenounced
		,case when unifiedLoan.Id is null then Loan.BaseInterest else unifiedLoan.BaseInterest	 end as 	unifiedLoan_BaseInterest
		,case when unifiedLoan.Id is null then Loan.Law128Rate else unifiedLoan.Law128Rate	 end as 	unifiedLoan_Law128Rate
		,case when unifiedLoan.Id is null then Loan.Spread else unifiedLoan.Spread	 end as 	unifiedLoan_Spread
		,case when unifiedLoan.Id is null then Loan.IsFixedInterest else unifiedLoan.IsFixedInterest	 end as 	unifiedLoan_IsFixedInterest
		,case when unifiedLoan.Id is null then Loan.ContractualInterestRate else unifiedLoan.ContractualInterestRate	 end as 	unifiedLoan_ContractualInterestRate
		,case when unifiedLoan.Id is null then Loan.CurrentInterestAmount else unifiedLoan.CurrentInterestAmount	 end as 	unifiedLoan_CurrentInterestAmount
		,case when unifiedLoan.Id is null then Loan.CurrentInterestAmountInBaseCurrency else unifiedLoan.CurrentInterestAmountInBaseCurrency	 end as 	unifiedLoan_CurrentInterestAmountInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.AgencyType else unifiedLoan.AgencyType	 end as 	unifiedLoan_AgencyType
		,case when unifiedLoan.Id is null then Loan.BaseInterestType else unifiedLoan.BaseInterestType	 end as 	unifiedLoan_BaseInterestType
		,case when unifiedLoan.Id is null then Loan.Delivered else unifiedLoan.Delivered	 end as 	unifiedLoan_Delivered
		,case when unifiedLoan.Id is null then Loan.AccountOwnerId else unifiedLoan.AccountOwnerId	 end as 	unifiedLoan_AccountOwnerId
		,case when unifiedLoan.Id is null then Loan.LoanPurpose else unifiedLoan.LoanPurpose	 end as 	unifiedLoan_LoanPurpose
		,case when unifiedLoan.Id is null then Loan.ParticipationPercentage else unifiedLoan.ParticipationPercentage	 end as 	unifiedLoan_ParticipationPercentage
		,case when unifiedLoan.Id is null then Loan.ExchangeRateAtOrigination else unifiedLoan.ExchangeRateAtOrigination	 end as 	unifiedLoan_ExchangeRateAtOrigination
		,case when unifiedLoan.Id is null then Loan.IsOverdraft else unifiedLoan.IsOverdraft	 end as 	unifiedLoan_IsOverdraft
		,case when unifiedLoan.Id is null then Loan.CreditLimit else unifiedLoan.CreditLimit	 end as 	unifiedLoan_CreditLimit
		,case when unifiedLoan.Id is null then Loan.CreditLimitInBaseCurrency else unifiedLoan.CreditLimitInBaseCurrency	 end as 	unifiedLoan_CreditLimitInBaseCurrency
		,case when unifiedLoan.Id is null then Loan.ActiveApplicationDescription else unifiedLoan.ActiveApplicationDescription	 end as 	unifiedLoan_ActiveApplicationDescription
		,case when unifiedLoan.Id is null then Loan.AmortizationTypeDescription else unifiedLoan.AmortizationTypeDescription	 end as 	unifiedLoan_AmortizationTypeDescription
		,case when unifiedLoan.Id is null then Loan.AssignmentTeamDescription else unifiedLoan.AssignmentTeamDescription	 end as 	unifiedLoan_AssignmentTeamDescription
		,case when unifiedLoan.Id is null then Loan.CaseWatchlListDescription else unifiedLoan.CaseWatchlListDescription	 end as 	unifiedLoan_CaseWatchlListDescription
		,case when unifiedLoan.Id is null then Loan.GracePeriodIndicatorDescription else unifiedLoan.GracePeriodIndicatorDescription	 end as 	unifiedLoan_GracePeriodIndicatorDescription
		,case when unifiedLoan.Id is null then Loan.LastApplicationTypeDescription else unifiedLoan.LastApplicationTypeDescription	 end as 	unifiedLoan_LastApplicationTypeDescription
		,case when unifiedLoan.Id is null then Loan.Law3869TypeDescription else unifiedLoan.Law3869TypeDescription	 end as 	unifiedLoan_Law3869TypeDescription
		,case when unifiedLoan.Id is null then Loan.LegalProcessStatusDateDescription else unifiedLoan.LegalProcessStatusDateDescription	 end as 	unifiedLoan_LegalProcessStatusDateDescription
		,case when unifiedLoan.Id is null then Loan.Lt12bIndicatorDescription else unifiedLoan.Lt12bIndicatorDescription	 end as 	unifiedLoan_Lt12bIndicatorDescription
		,case when unifiedLoan.Id is null then Loan.MinBucketDescription else unifiedLoan.MinBucketDescription	 end as 	unifiedLoan_MinBucketDescription
		,case when unifiedLoan.Id is null then Loan.ObjectiveStatusDescription else unifiedLoan.ObjectiveStatusDescription	 end as 	unifiedLoan_ObjectiveStatusDescription
		,case when unifiedLoan.Id is null then Loan.PerformingStatusDescription else unifiedLoan.PerformingStatusDescription	 end as 	unifiedLoan_PerformingStatusDescription
		,case when unifiedLoan.Id is null then Loan.QueIdDescription else unifiedLoan.QueIdDescription	 end as 	unifiedLoan_QueIdDescription
		,case when unifiedLoan.Id is null then Loan.SettlementBucketDescription else unifiedLoan.SettlementBucketDescription	 end as 	unifiedLoan_SettlementBucketDescription
		,case when unifiedLoan.Id is null then Loan.SpvDescription else unifiedLoan.SpvDescription	 end as 	unifiedLoan_SpvDescription
		,case when unifiedLoan.Id is null then Loan.IsAutomobileDescription else unifiedLoan.IsAutomobileDescription	 end as 	unifiedLoan_IsAutomobileDescription
		,case when unifiedLoan.Id is null then Loan.HasGovernmentGuaranteeDescription else unifiedLoan.HasGovernmentGuaranteeDescription	 end as 	unifiedLoan_HasGovernmentGuaranteeDescription
		,case when unifiedLoan.Id is null then Loan.HasLegalActionsDescription else unifiedLoan.HasLegalActionsDescription	 end as 	unifiedLoan_HasLegalActionsDescription
		,case when unifiedLoan.Id is null then Loan.BorrowerAccountSegmentDescription else unifiedLoan.BorrowerAccountSegmentDescription	 end as 	unifiedLoan_BorrowerAccountSegmentDescription
		,case when unifiedLoan.Id is null then Loan.LegalProcessStatusDescription else unifiedLoan.LegalProcessStatusDescription	 end as 	unifiedLoan_LegalProcessStatusDescription
		,case when unifiedLoan.Id is null then Loan.DenounceCaseEverDescription else unifiedLoan.DenounceCaseEverDescription	 end as 	unifiedLoan_DenounceCaseEverDescription
		,case when unifiedLoan.Id is null then Loan.IsRevolvingLoanDescription else unifiedLoan.IsRevolvingLoanDescription	 end as 	unifiedLoan_IsRevolvingLoanDescription
		,case when unifiedLoan.Id is null then Loan.IsDenouncedDescription else unifiedLoan.IsDenouncedDescription	 end as 	unifiedLoan_IsDenouncedDescription
		,case when unifiedLoan.Id is null then Loan.IsFixedInterestDescription else unifiedLoan.IsFixedInterestDescription	 end as 	unifiedLoan_IsFixedInterestDescription
		,case when unifiedLoan.Id is null then Loan.AgencyTypeDescription else unifiedLoan.AgencyTypeDescription	 end as 	unifiedLoan_AgencyTypeDescription
		,case when unifiedLoan.Id is null then Loan.BaseInterestTypeDescription else unifiedLoan.BaseInterestTypeDescription	 end as 	unifiedLoan_BaseInterestTypeDescription
		,case when unifiedLoan.Id is null then Loan.DeliveredDescription else unifiedLoan.DeliveredDescription	 end as 	unifiedLoan_DeliveredDescription
		,case when unifiedLoan.Id is null then Loan.LoanPurposeDescription else unifiedLoan.LoanPurposeDescription	 end as 	unifiedLoan_LoanPurposeDescription
		,case when unifiedLoan.Id is null then Loan.IsOverdraftDescription else unifiedLoan.IsOverdraftDescription	 end as 	unifiedLoan_IsOverdraftDescription
		,case when unifiedLoan.Id is null then Loan.PositiveLoanEAPIndicator else unifiedLoan.PositiveLoanEAPIndicator	 end as 	unifiedLoan_PositiveLoanEAPIndicator
		,case when unifiedLoan.Id is null then Loan.PositiveLoanEAPIndicatorDescription else unifiedLoan.PositiveLoanEAPIndicatorDescription	 end as 	unifiedLoan_PositiveLoanEAPIndicatorDescription
		,case when unifiedLoan.Id is null then Loan.ContractNumberSContractNumberSystemicNrystemicNr else unifiedLoan.ContractNumberSContractNumberSystemicNrystemicNr	 end as 	unifiedLoan_ContractNumberSContractNumberSystemicNrystemicNr
		,case when unifiedLoan.Id is null then Loan.EligiblePrenotedAsset else unifiedLoan.EligiblePrenotedAsset	 end as 	unifiedLoan_EligiblePrenotedAsset
		,case when unifiedLoan.Id is null then Loan.EligiblePrenotedAssetDescription else unifiedLoan.EligiblePrenotedAssetDescription	 end as 	unifiedLoan_EligiblePrenotedAssetDescription
		,case when unifiedLoan.Id is null then Loan.AccountPayer else unifiedLoan.AccountPayer	 end as 	unifiedLoan_AccountPayer
		,case when unifiedLoan.Id is null then Loan.AccountPayerDescription else unifiedLoan.AccountPayerDescription	 end as 	unifiedLoan_AccountPayerDescription
		,case when unifiedLoan.Id is null then Loan.AccountNumberWithPrefix else unifiedLoan.AccountNumberWithPrefix	 end as 	unifiedLoan_AccountNumberWithPrefix
		,case when unifiedLoan.Id is null then Loan.ContractNumberSystemicNr else unifiedLoan.ContractNumberSystemicNr	 end as 	unifiedLoan_ContractNumberSystemicNr
		,case when unifiedLoan.Id is null then Loan.DecisionTreeCode else unifiedLoan.DecisionTreeCode	 end as 	unifiedLoan_DecisionTreeCode
		,Loan.DecisionTreeDescription	as 	unifiedLoan_DecisionTreeDescription
		,Loan.DecisionTreeValidPath	as 	unifiedLoan_DecisionTreeValidPath

	
	,product.loanId
	,product.product_Id
	,product.product_LoanId
	,product.product_ProductKey
	,product.product_Priority
	,product.product_ProductTypePriority
	,product.product_NPVInBaseCurrency
	,product.product_NPV
	,product.product_Code
	,product.product_Name
	,product.product_ProductCategory
	,product.product_ProductCategoryType
	,product.product_BalanceUsedForNPVCalculationInBaseCurrency
	,product.product_BalanceUsedForNPVCalculation
	,product.product_DownPaymentAmountInBaseCurrency
	,product.product_DownPaymentAmount
	,product.product_BalanceForModificationInBaseCurrency
	,product.product_BalanceForModification
	,product.product_ExpensesOfModificationAmountInBaseCurrency
	,product.product_ExpensesOfModificationAmount
	,product.product_HasStepUp
	,product.product_HasHairCut
	,product.product_HairCutPercentage
	,product.product_HairCutAmount
	,product.product_HairCutPercentageOnBalance
	,product.product_HairCutAmountOnBalanceInBaseCurrency
	,product.product_HairCutAmountOnBalance
	,product.product_HaircutPercentageWithOffBalance
	,product.product_HaircutAmountWithOffBalanceInBaseCurrency
	,product.product_HaircutAmountWithOffBalance
	,product.product_HasHairCutWithClawBack
	,product.product_HasSplitBalance
	,product.product_SplitBalancePercentage
	,product.product_ModificationCurrency
	,product.product_AutoCreditStatus
	,product.product_AutoCreditStatusDescription
	,product.product_HasNonAutoCreditReasons
	,product.product_NonAutoCreditReasonsVisible
	,product.product_IsViable
	,product.product_PenaltyInterestRate
	,product.product_RequiresNewPrenotation
	,product.product_TotalTenureInMonths
	,product.product_SmallTicketAmountInBaseCurrency
	,product.product_SmallTicketAmount
	,product.product_IsVisible
	,product.product_IsLiquidation
	,product.product_InstallmentAmountCalculatedInBaseCurrency
	,product.product_InstallmentAmountCalculated
	,product.product_TotalInterestRate
	,product.product_TenureInMonthsMin
	,product.product_TenureInMonthsMax
	,product.product_HairCutPercentageMin
	,product.product_HairCutPercentageMax
	,product.product_IsEditable
	,product.product_IsCustom
	,product.product_OriginProductKey
	,product.product_ModificationAmountAfterHaircut
	,product.product_StepUpPeriodTenureInMonths
	,product.product_StepUpStartMonth
	,product.product_StepUpReadjustmentDuration
	,product.product_StepUpPeriodNumber
	,product.product_StepUpInitialInstallmentAmount
	,product.product_StepUpAmount
	,product.product_DurationAfterStepUpExpirationInMonths
	,product.product_SplitBalanceAmountA
	,product.product_SplitBalanceAmountB
	,product.product_SplitBalanceTenureInMonthsA
	,product.product_SplitBalanceStartMonthB
	,product.product_SplitBalanceTenureInMonthsB
	,product.product_DifferentInterestsDuringPeriods
	,product.product_InterestDurationOfFirstPeriodInMonths
	,product.product_FixedInterestOfFirstPeriod
	,product.product_BaseInterestTypeOfFirstPeriod
	,product.product_SpreadOfFirstPeriod
	,product.product_Law128RateOfFirstPeriodValue
	,product.product_BaseInterest
	,product.product_Spread
	,product.product_Law128RateValue
	,product.product_ReducedInstalmentPeriod
	,product.product_ReducedInstalmentPeriodInMonths
	,product.product_ReducedInstalmentPeriodAmount
	,product.finalprod_Id

	,profitsInfo.Id profitsInfo_Id
		,profitsInfo.AccountImplementationInfoId profitsInfo_AccountImplementationInfoId
		,profitsInfo.FinalModificationProductId profitsInfo_FinalModificationProductId
		,profitsInfo.ExternalSystemGroupApplicationID profitsInfo_ExternalSystemGroupApplicationID
		,profitsInfo.ExternalSystemApplicationID profitsInfo_ExternalSystemApplicationID
		,profitsInfo.NewAccountNumber profitsInfo_NewAccountNumber
		,profitsInfo.RestructuringProgram profitsInfo_RestructuringProgram
		,profitsInfo.DateOfFirstInstallment profitsInfo_DateOfFirstInstallment
		,profitsInfo.InstallmentAmount profitsInfo_InstallmentAmount
		,profitsInfo.ProcessStatus profitsInfo_ProcessStatus
		,profitsInfo.ErrorDescription profitsInfo_ErrorDescription
		,profitsInfo.SystemCode profitsInfo_SystemCode
		,profitsInfo.ProfitsApplicationID profitsInfo_ProfitsApplicationID
		,profitsInfo.CustomerCode profitsInfo_CustomerCode
		,profitsInfo.InternalSN profitsInfo_InternalSN
		,profitsInfo.TransactionDate profitsInfo_TransactionDate
		,profitsInfo.AccountSystem profitsInfo_AccountSystem
		,profitsInfo.ProductCode profitsInfo_ProductCode
		,profitsInfo.ProcessTimestamp profitsInfo_ProcessTimestamp
		,profitsInfo.SentStatus profitsInfo_SentStatus
		,profitsInfo.InterestAmount profitsInfo_InterestAmount
		,profitsInfo.InterestFirstDate profitsInfo_InterestFirstDate
		,profitsInfo.ImplementationStatusId profitsInfo_ImplementationStatusId
		,profitsInfo.ImplementationStatusDescription profitsInfo_ImplementationStatusDescription
		,profitsInfo.IsActive profitsInfo_IsActive
		,profitsInfo.AccountNumber profitsInfo_AccountNumber
		,profitsInfo.ProductKey profitsInfo_ProductKey
		,profitsInfo.DateCreated profitsInfo_DateCreated
		,profitsInfo.SourceFinalSolutionId profitsInfo_SourceFinalSolutionId

	,app.QualcoAppSubmittedDate
	,app.QualcoAppPreApprovedDate
	,app.QualcoAppApprovedDate
	,app.QualcoAppDisbursedDate
	,app.QualcoAppRejectedDate
	,app.QualcoAppCancelledDate
	,app.DateUpdated
	,app.QualcoAppFinalStepDate


	,GETDATE() as SnapshotDate
	
	,case when modification.CustomerId is not null and loanForModification.CustomerId is not null and product.loanId is not null then 1 else 0 end as ActivatedApplicationFlag


	,case when app.ParentStepGroupingId in ('1') then CAST (CONCAT('1.', app.galaxyApplicationMetadata_stepReportingNameId) as float)
		when app.ParentStepGroupingId in ('2') then CAST (CONCAT('2.', app.galaxyApplicationMetadata_stepReportingNameId) as float)
		when app.ParentStepGroupingId in ('3') then CAST (CONCAT('3.', app.galaxyApplicationMetadata_stepReportingNameId) as float)
		when app.ParentStepGroupingId in ('7') then CAST (CONCAT('4.', app.galaxyApplicationMetadata_stepReportingNameId) as float)
		when app.ParentStepGroupingId in ('4') then CAST (CONCAT('5.', app.galaxyApplicationMetadata_stepReportingNameId) as float)
		when app.ParentStepGroupingId in ('5') then CAST (CONCAT('6.', app.galaxyApplicationMetadata_stepReportingNameId) as float)
		when app.ParentStepGroupingId in ('6') then CAST (CONCAT('7.', app.galaxyApplicationMetadata_stepReportingNameId) as float)
	end as StepSortOrder
		,loan.GBV as loan_GBV
		,app.PolicyOverride as app_PolicyOverride 
		,app.ContactSigningPlaceCode as app_ContactSigningPlaceCode
		,product.product_FirstPeriodTotalInterestRate
		 ,case when  product.product_ModificationAmountAfterHaircut IS NOT NULL OR product.product_ModificationAmountAfterHaircut <> 0 then product.product_DownPaymentAmount/ product.product_ModificationAmountAfterHaircut end product_DownPaymentAmountRate

	FROM LatestApplication app
				INNER JOIN LatestCustomer customer  ON customer.ApplicationId = app.Id
				LEFT OUTER JOIN LatestModification modification ON modification.CustomerId = customer.Id
				LEFT OUTER JOIN Loan loan ON loan.CustomerId = customer.Id
				LEFT OUTER JOIN UnifiedLoan unifiedLoan ON unifiedLoan.LoanId = loan.Id
				LEFT OUTER JOIN LoansForModification loanForModification ON loanForModification.Id = loan.Id
				LEFT JOIN
	(
			SELECT loan.ID AS loanId
		,product.ProductKey	 as 	product_Id
  		,product.LoanId	 as 	product_LoanId
		,product.ProductKey	 as 	product_ProductKey
		,product.Priority	 as 	product_Priority
		,product.ProductTypePriority	 as 	product_ProductTypePriority
		,product.NPVInBaseCurrency	 as 	product_NPVInBaseCurrency
		,product.NPV	 as 	product_NPV
		,product.Code	 as 	product_Code
		,product.Name	 as 	product_Name
		,product.ProductCategory	 as 	product_ProductCategory
		,product.ProductCategoryType	 as 	product_ProductCategoryType
		,product.BalanceUsedForNPVCalculationInBaseCurrency	 as 	product_BalanceUsedForNPVCalculationInBaseCurrency
		,product.BalanceUsedForNPVCalculation	 as 	product_BalanceUsedForNPVCalculation
		,product.DownPaymentAmountInBaseCurrency	 as 	product_DownPaymentAmountInBaseCurrency
		,product.DownPaymentAmount	 as 	product_DownPaymentAmount
		,product.BalanceForModificationInBaseCurrency	 as 	product_BalanceForModificationInBaseCurrency
		,product.BalanceForModification	 as 	product_BalanceForModification
		,product.ExpensesOfModificationAmountInBaseCurrency	 as 	product_ExpensesOfModificationAmountInBaseCurrency
		,product.ExpensesOfModificationAmount	 as 	product_ExpensesOfModificationAmount
		,product.HasStepUp	 as 	product_HasStepUp
		,product.HasHairCut	 as 	product_HasHairCut
		,product.HairCutPercentage	 as 	product_HairCutPercentage
		,product.HairCutAmount	 as 	product_HairCutAmount
		,product.HairCutPercentageOnBalance	 as 	product_HairCutPercentageOnBalance
		,product.HairCutAmountOnBalanceInBaseCurrency	 as 	product_HairCutAmountOnBalanceInBaseCurrency
		,product.HairCutAmountOnBalance	 as 	product_HairCutAmountOnBalance
		,product.HaircutPercentageWithOffBalance	 as 	product_HaircutPercentageWithOffBalance
		,product.HaircutAmountWithOffBalanceInBaseCurrency	 as 	product_HaircutAmountWithOffBalanceInBaseCurrency
		,product.HaircutAmountWithOffBalance	 as 	product_HaircutAmountWithOffBalance
		,product.HasHairCutWithClawBack	 as 	product_HasHairCutWithClawBack
		,product.HasSplitBalance	 as 	product_HasSplitBalance
		,product.SplitBalancePercentage	 as 	product_SplitBalancePercentage
		,product.ModificationCurrency	 as 	product_ModificationCurrency
		,product.AutoCreditStatus	 as 	product_AutoCreditStatus
		,product.AutoCreditStatusDescription	 as 	product_AutoCreditStatusDescription
		,product.HasNonAutoCreditReasons	 as 	product_HasNonAutoCreditReasons
		,product.NonAutoCreditReasonsVisible	 as 	product_NonAutoCreditReasonsVisible
		,product.IsViable	 as 	product_IsViable
		,product.PenaltyInterestRate	 as 	product_PenaltyInterestRate
		,product.RequiresNewPrenotation	 as 	product_RequiresNewPrenotation
		,product.TotalTenureInMonths	 as 	product_TotalTenureInMonths
		,product.SmallTicketAmountInBaseCurrency	 as 	product_SmallTicketAmountInBaseCurrency
		,product.SmallTicketAmount	 as 	product_SmallTicketAmount
		,product.IsVisible	 as 	product_IsVisible
		,product.IsLiquidation	 as 	product_IsLiquidation
		,product.InstallmentAmountCalculatedInBaseCurrency	 as 	product_InstallmentAmountCalculatedInBaseCurrency
		,product.InstallmentAmountCalculated	 as 	product_InstallmentAmountCalculated
		,product.TotalInterestRate	 as 	product_TotalInterestRate
		,product.TenureInMonthsMin	 as 	product_TenureInMonthsMin
		,product.TenureInMonthsMax	 as 	product_TenureInMonthsMax
		,product.HairCutPercentageMin	 as 	product_HairCutPercentageMin
		,product.HairCutPercentageMax	 as 	product_HairCutPercentageMax
		,product.IsEditable	 as 	product_IsEditable
		,product.IsCustom	 as 	product_IsCustom
		,product.OriginProductKey	 as 	product_OriginProductKey
		,product.ModificationAmountAfterHaircut	 as 	product_ModificationAmountAfterHaircut
		,product.StepUpPeriodTenureInMonths	 as 	product_StepUpPeriodTenureInMonths
		,product.StepUpStartMonth	 as 	product_StepUpStartMonth
		,product.StepUpReadjustmentDuration	 as 	product_StepUpReadjustmentDuration
		,product.StepUpPeriodNumber	 as 	product_StepUpPeriodNumber
		,product.StepUpInitialInstallmentAmount	 as 	product_StepUpInitialInstallmentAmount
		,product.StepUpAmount	 as 	product_StepUpAmount
		,product.DurationAfterStepUpExpirationInMonths	 as 	product_DurationAfterStepUpExpirationInMonths
		,product.SplitBalanceAmountA	 as 	product_SplitBalanceAmountA
		,product.SplitBalanceAmountB	 as 	product_SplitBalanceAmountB
		,product.SplitBalanceTenureInMonthsA	 as 	product_SplitBalanceTenureInMonthsA
		,product.SplitBalanceStartMonthB	 as 	product_SplitBalanceStartMonthB
		,product.SplitBalanceTenureInMonthsB	 as 	product_SplitBalanceTenureInMonthsB
		,product.DifferentInterestsDuringPeriods	 as 	product_DifferentInterestsDuringPeriods
		,product.InterestDurationOfFirstPeriodInMonths	 as 	product_InterestDurationOfFirstPeriodInMonths
		,product.FixedInterestOfFirstPeriod	 as 	product_FixedInterestOfFirstPeriod
		,product.BaseInterestTypeOfFirstPeriod	 as 	product_BaseInterestTypeOfFirstPeriod
		,product.SpreadOfFirstPeriod	 as 	product_SpreadOfFirstPeriod
		,product.Law128RateOfFirstPeriodValue	 as 	product_Law128RateOfFirstPeriodValue
		,product.BaseInterest	 as 	product_BaseInterest
		,product.Spread	 as 	product_Spread
		,product.Law128RateValue	 as 	product_Law128RateValue
		,product.ReducedInstalmentPeriod	 as 	product_ReducedInstalmentPeriod
		,product.ReducedInstalmentPeriodInMonths	 as 	product_ReducedInstalmentPeriodInMonths
		,product.ReducedInstalmentPeriodAmount	 as 	product_ReducedInstalmentPeriodAmount
	,product.FirstPeriodTotalInterestRate as product_FirstPeriodTotalInterestRate
	,finalprod.Id as finalprod_Id
			FROM LoansForModification loan
	    				INNER JOIN SuggestedModificationProduct  product ON loan.Id = product.Loanid  
						INNER JOIN FinalModificationProduct finalprod ON finalprod.ProductKey = product.ProductKey  AND product.ProductKey <> '00000000-0000-0000-0000-000000000000'
						INNER JOIN LatestFinalSolution finalSolution on finalSolution.Id = finalprod.FinalSolutionId
	) product ON product.loanId = loan.Id  
	LEFT OUTER JOIN LatestAccountImplementationInfo profitsInfo on profitsInfo.FinalModificationProductId = product.finalprod_Id
'''
)
FLDT_AllApplications.createOrReplaceTempView("FLDT_AllApplications")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(A.BusinessDate AS DATE) AS BusinessDate
, CAST(A.AccountImplementationStatus AS string)  AS  AccountImplementationStatus
, CAST(A.app_Id AS string)  AS  app_Id
, CAST(A.app_ApplicationId AS int)  AS  app_ApplicationId
, CAST(A.app_ApplicationKey AS string)  AS  app_ApplicationKey
, CAST(A.app_UserCreated AS string)  AS  app_UserCreated
, CAST(A.app_DateCreated AS timestamp)  AS  app_DateCreated
, CAST(A.app_DateUpdated AS timestamp)  AS  app_DateUpdated
, CAST(A.app_ApplicationStatus AS smallint)  AS  app_ApplicationStatus
, CAST(A.app_ApplicationStatusDescription AS string)  AS  app_ApplicationStatusDescription
, CAST(A.app_PortfolioCode AS string)  AS  app_PortfolioCode
, CAST(A.app_PortfolioName AS string)  AS  app_PortfolioName
, CAST(A.app_PortfolioDescription AS string)  AS  app_PortfolioDescription
, CAST(A.app_InvestorCode AS string)  AS  app_InvestorCode
, CAST(A.app_InvestorName AS string)  AS  app_InvestorName
, CAST(A.app_InvestorDescription AS string)  AS  app_InvestorDescription
, CAST(A.app_ProcessingUserId AS varchar(36))  AS  app_ProcessingUserId
, CAST(A.app_ProcessingUserFullName AS string)  AS  app_ProcessingUserFullName
, CAST(A.app_AssigneeUserId AS varchar(36))  AS  app_AssigneeUserId
, CAST(A.app_AssigneeUserFullName AS string)  AS  app_AssigneeUserFullName
, CAST(A.app_AssigneePortfolioGroupCode AS string)  AS  app_AssigneePortfolioGroupCode
, CAST(A.app_AssigneePortfolioGroupName AS string)  AS  app_AssigneePortfolioGroupName
, CAST(A.app_CurrentStepId AS varchar(36))  AS  app_CurrentStepId
, CAST(A.app_CurrentStepName AS string)  AS  app_CurrentStepName
, CAST(A.app_UserUnitCode AS string)  AS  app_UserUnitCode
, CAST(A.app_UserUnitDescription AS string)  AS  app_UserUnitDescription
, CAST(A.app_IntegrationCode AS string)  AS  app_IntegrationCode
, CAST(A.app_CommunicationChannelCode AS string)  AS  app_CommunicationChannelCode
, CAST(A.app_CommunicationChannelDescription AS string)  AS  app_CommunicationChannelDescription
, CAST(A.app_CurrentStepIntegrationCode AS string)  AS  app_CurrentStepIntegrationCode
, CAST(A.app_CustomerCode AS string)  AS  app_CustomerCode
, CAST(A.app_FlowSegment AS string)  AS  app_FlowSegment
, CAST(A.app_FlowSegmentDescription AS string)  AS  app_FlowSegmentDescription
, CAST(A.app_UserCreatedQualcoGroupCode AS string)  AS  app_UserCreatedQualcoGroupCode
, CAST(A.app_UserCreatedQualcoChannelCode AS string)  AS  app_UserCreatedQualcoChannelCode
, CAST(A.app_UserCreatedGroupCategory AS string)  AS  app_UserCreatedGroupCategory
, CAST(A.app_UserCreatedUserCreatedGroupCode AS string)  AS  app_UserCreatedUserCreatedGroupCode
, CAST(A.app_UserCreatedGroupName AS string)  AS  app_UserCreatedGroupName
, CAST(A.app_UserCreatedFullName AS string)  AS  app_UserCreatedFullName
, CAST(A.app_AssigneeUserName AS string)  AS  app_AssigneeUserName
, CAST(A.app_ContractSigningMethodCode AS string)  AS  app_ContractSigningMethodCode
, CAST(A.app_ContractSigningMethodDescription AS string)  AS  app_ContractSigningMethodDescription
, CAST(A.galaxyApplicationMetadata_SourceApplicationId AS int)  AS  galaxyApplicationMetadata_SourceApplicationId
, CAST(A.galaxyApplicationMetadata_SourceApplicationKey AS string)  AS  galaxyApplicationMetadata_SourceApplicationKey
, CAST(A.galaxyApplicationMetadata_ApplicationStatus AS int)  AS  galaxyApplicationMetadata_ApplicationStatus
, CAST(A.galaxyApplicationMetadata_ApplicationStatusDescription AS string)  AS  galaxyApplicationMetadata_ApplicationStatusDescription
, CAST(A.galaxyApplicationMetadata_CreditInvolvementStatus AS int)  AS  galaxyApplicationMetadata_CreditInvolvementStatus
, CAST(A.galaxyApplicationMetadata_CreditInvolvementStatusDescription AS string)  AS  galaxyApplicationMetadata_CreditInvolvementStatusDescription
, CAST(A.galaxyApplicationMetadata_ParentStepGroupingId AS int)  AS  galaxyApplicationMetadata_ParentStepGroupingId
, CAST(A.galaxyApplicationMetadata_ParentStepGroupingDescription AS string)  AS  galaxyApplicationMetadata_ParentStepGroupingDescription
, CAST(A.galaxyApplicationMetadata_ParentStepGroupingOwnerId AS int)  AS  galaxyApplicationMetadata_ParentStepGroupingOwnerId
, CAST(A.galaxyApplicationMetadata_ParentStepGroupingOwnerName AS string)  AS  galaxyApplicationMetadata_ParentStepGroupingOwnerName
, CAST(A.galaxyApplicationMetadata_StepGroupingId AS int)  AS  galaxyApplicationMetadata_StepGroupingId
, CAST(A.galaxyApplicationMetadata_StepGroupingDescription AS string)  AS  galaxyApplicationMetadata_StepGroupingDescription
, CAST(A.galaxyApplicationMetadata_StepGroupingOwnerId AS int)  AS  galaxyApplicationMetadata_StepGroupingOwnerId
, CAST(A.galaxyApplicationMetadata_StepGroupingOwnerName AS string)  AS  galaxyApplicationMetadata_StepGroupingOwnerName
, CAST(A.galaxyApplicationMetadata_ClientResponseId AS int)  AS  galaxyApplicationMetadata_ClientResponseId
, CAST(A.galaxyApplicationMetadata_ClientResponseDescription AS string)  AS  galaxyApplicationMetadata_ClientResponseDescription
, CAST(A.galaxyApplicationMetadata_OcwSimulationVoteId AS int)  AS  galaxyApplicationMetadata_OcwSimulationVoteId
, CAST(A.galaxyApplicationMetadata_OcwSimulationVoteDescription AS string)  AS  galaxyApplicationMetadata_OcwSimulationVoteDescription
, CAST(A.galaxyApplicationMetadata_QualcoAppStartedDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppStartedDate
, CAST(A.galaxyApplicationMetadata_QualcoAppSubmittedDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppSubmittedDate
, CAST(A.galaxyApplicationMetadata_QualcoAppPreApprovedDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppPreApprovedDate
, CAST(A.galaxyApplicationMetadata_QualcoAppApprovedDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppApprovedDate
, CAST(A.galaxyApplicationMetadata_QualcoAppDisbursedDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppDisbursedDate
, CAST(A.galaxyApplicationMetadata_QualcoAppRejectedDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppRejectedDate
, CAST(A.galaxyApplicationMetadata_QualcoAppCancelledDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppCancelledDate
, CAST(A.galaxyApplicationMetadata_QualcoAppFinalStepDate AS timestamp)  AS  galaxyApplicationMetadata_QualcoAppFinalStepDate
, CAST(A.galaxyApplicationMetadata_DateCreated AS timestamp)  AS  galaxyApplicationMetadata_DateCreated
, CAST(A.galaxyApplicationMetadata_DateUpdated AS timestamp)  AS  galaxyApplicationMetadata_DateUpdated
, CAST(A.galaxyApplicationMetadata_IsActive AS boolean)  AS  galaxyApplicationMetadata_IsActive
, CAST(A.app_CurrentStepAssignmentDate AS timestamp)  AS  app_CurrentStepAssignmentDate
, CAST(A.app_InterbankFlag AS boolean)  AS  app_InterbankFlag
, CAST(A.app_OcwFlag AS boolean)  AS  app_OcwFlag
, CAST(A.app_ManagedByPortfolioGroupName AS string)  AS  app_ManagedByPortfolioGroupName
, CAST(A.app_ManagedByPortfolioGroupCode AS string)  AS  app_ManagedByPortfolioGroupCode
, CAST(A.app_ContactSigningPlaceId AS int)  AS  app_ContactSigningPlaceId
, CAST(A.app_ContactSigningPlaceName AS string)  AS  app_ContactSigningPlaceName
, CAST(A.app_ContactSigningPlaceTypeId AS int)  AS  app_ContactSigningPlaceTypeId
, CAST(A.app_ContactSigningPlaceTypeName AS string)  AS  app_ContactSigningPlaceTypeName
, CAST(A.app_galaxyApplicationMetadata_StepReportingName AS string)  AS  app_galaxyApplicationMetadata_StepReportingName
, CAST(A.app_galaxyApplicationMetadata_StepActiveDuration AS bigint)  AS  app_galaxyApplicationMetadata_StepActiveDuration
, CAST(A.app_galaxyApplicationMetadata_stepReportingNameId AS int)  AS  app_galaxyApplicationMetadata_stepReportingNameId
, CAST(A.ApplicationImplementationStatus AS string)  AS  ApplicationImplementationStatus
, CAST(A.ApplicationActivationDate AS date)  AS  ApplicationActivationDate
, CAST(A.modification_Id AS int)  AS  modification_Id
, CAST(A.modification_CustomerId AS int)  AS  modification_CustomerId
, CAST(A.modification_ModificationId AS int)  AS  modification_ModificationId
, CAST(A.modification_ModificationRequestKey AS string)  AS  modification_ModificationRequestKey
, CAST(A.modification_UserName AS string)  AS  modification_UserName
, CAST(A.modification_Fullname AS string)  AS  modification_Fullname
, CAST(A.modification_Channel AS smallint)  AS  modification_Channel
, CAST(A.modification_ChannelDescription AS string)  AS  modification_ChannelDescription
, CAST(A.modification_DateOfCalculation AS timestamp)  AS  modification_DateOfCalculation
, CAST(A.modification_IsSuccess AS boolean)  AS  modification_IsSuccess
, CAST(A.modification_ErrorCode AS string)  AS  modification_ErrorCode
, CAST(A.modification_MinDisposableIncomeForViableSolutions AS decimal(19,2))  AS  modification_MinDisposableIncomeForViableSolutions
, CAST(A.modification_MinDisposableIncomeForNonViableSolutions AS decimal(19,2))  AS  modification_MinDisposableIncomeForNonViableSolutions
, CAST(A.modification_IsViableSolution AS boolean)  AS  modification_IsViableSolution
, CAST(A.modification_CalculationMode AS smallint)  AS  modification_CalculationMode
, CAST(A.modification_CalculationModeDescription AS string)  AS  modification_CalculationModeDescription
, CAST(A.modification_UnitCode AS string)  AS  modification_UnitCode
, CAST(A.modification_UnitName AS string)  AS  modification_UnitName
, CAST(A.modification_UnitDescription AS string)  AS  modification_UnitDescription
, CAST(A.modification_AutoCreditReasonsVisible AS boolean)  AS  modification_AutoCreditReasonsVisible
, CAST(A.modification_ExtendedLimits AS smallint)  AS  modification_ExtendedLimits
, CAST(A.modification_ExtendedLimitsDescription AS string)  AS  modification_ExtendedLimitsDescription
, CAST(A.modification_CalculatorServiceId AS int)  AS  modification_CalculatorServiceId
, CAST(A.modification_ModificationAmendmentId AS int)  AS  modification_ModificationAmendmentId
, CAST(A.modification_CreditPolicyId AS int)  AS  modification_CreditPolicyId
, CAST(A.modification_CreditPolicyCode AS string)  AS  modification_CreditPolicyCode
, CAST(A.modification_CreditPolicyName AS string)  AS  modification_CreditPolicyName
, CAST(A.modification_CreditPolicyDescription AS string)  AS  modification_CreditPolicyDescription
, CAST(A.customer_Id AS int)  AS  customer_Id
, CAST(A.customer_AssetClass AS string)  AS  customer_AssetClass
, CAST(A.customer_BalanceCategory AS string)  AS  customer_BalanceCategory
, CAST(A.customer_BeginMonth AS string)  AS  customer_BeginMonth
, CAST(A.customer_BorrowerIndustry AS string)  AS  customer_BorrowerIndustry
, CAST(A.customer_BPSegment AS string)  AS  customer_BPSegment
, CAST(A.customer_BusinessPlanStrategies AS string)  AS  customer_BusinessPlanStrategies
, CAST(A.customer_CustomerFatherName AS string)  AS  customer_CustomerFatherName
, CAST(A.customer_CustomerFirstName AS string)  AS  customer_CustomerFirstName
, CAST(A.customer_DebtorBankGroupID AS string)  AS  customer_DebtorBankGroupID
, CAST(A.customer_DT_DebtLpsStatus AS string)  AS  customer_DT_DebtLpsStatus
, CAST(A.customer_DT_RealEstateSearchStatus AS string)  AS  customer_DT_RealEstateSearchStatus
, CAST(A.customer_DT_Segment AS string)  AS  customer_DT_Segment
, CAST(A.customer_EAPSubStatus AS string)  AS  customer_EAPSubStatus
, CAST(A.customer_GroupCode AS string)  AS  customer_GroupCode
, CAST(A.customer_InvestorSegment AS string)  AS  customer_InvestorSegment
, CAST(A.customer_NegativeEAP AS string)  AS  customer_NegativeEAP
, CAST(A.customer_Occupation AS string)  AS  customer_Occupation
, CAST(A.customer_PositiveEAP AS string)  AS  customer_PositiveEAP
, CAST(A.customer_StrategyID AS string)  AS  customer_StrategyID
, CAST(A.customer_UC_ID AS string)  AS  customer_UC_ID
, CAST(A.customer_BankruptcyFlag AS int)  AS  customer_BankruptcyFlag
, CAST(A.customer_BorrowerIndustryCode AS int)  AS  customer_BorrowerIndustryCode
, CAST(A.customer_ConfidenceID AS int)  AS  customer_ConfidenceID
, CAST(A.customer_ConnectedBorrowers AS int)  AS  customer_ConnectedBorrowers
, CAST(A.customer_ContactActivities AS int)  AS  customer_ContactActivities
, CAST(A.customer_ContactActivitiesLast3Months AS int)  AS  customer_ContactActivitiesLast3Months
, CAST(A.customer_ContactActivitiesLastMonth AS int)  AS  customer_ContactActivitiesLastMonth
, CAST(A.customer_CurrentQualcoSegmentRFIs AS int)  AS  customer_CurrentQualcoSegmentRFIs
, CAST(A.customer_CurrentQualcoSegmentRFIsDescription AS string)  AS  customer_CurrentQualcoSegmentRFIsDescription
, CAST(A.customer_CustomerSubSegmentIndication AS int)  AS  customer_CustomerSubSegmentIndication
, CAST(A.customer_CustomerSubSegmentIndicationDescription AS string)  AS  customer_CustomerSubSegmentIndicationDescription
, CAST(A.customer_DebtorLPS AS int)  AS  customer_DebtorLPS
, CAST(A.customer_DebtorLPSDescription AS string)  AS  customer_DebtorLPSDescription
, CAST(A.customer_DraftedByID AS int)  AS  customer_DraftedByID
, CAST(A.customer_DT_BorrowerType AS int)  AS  customer_DT_BorrowerType
, CAST(A.customer_DT_DenouncementFlag AS int)  AS  customer_DT_DenouncementFlag
, CAST(A.customer_DT_ExtrajudicialLetterFlag AS int)  AS  customer_DT_ExtrajudicialLetterFlag
, CAST(A.customer_DT_PaymentOrderFlag AS int)  AS  customer_DT_PaymentOrderFlag
, CAST(A.customer_FinalMaxBucketPerPortfpolio AS int)  AS  customer_FinalMaxBucketPerPortfpolio
, CAST(A.customer_Gender AS int)  AS  customer_Gender
, CAST(A.customer_LeverageSegment AS int)  AS  customer_LeverageSegment
, CAST(A.customer_MaritalStatus AS int)  AS  customer_MaritalStatus
, CAST(A.customer_NegativeEAPIndicator AS int)  AS  customer_NegativeEAPIndicator
, CAST(A.customer_NegativeRightPersonContactsLast3Months AS int)  AS  customer_NegativeRightPersonContactsLast3Months
, CAST(A.customer_NegativeRightPersonContactsLastMonth AS int)  AS  customer_NegativeRightPersonContactsLastMonth
, CAST(A.customer_NegativeRightPersonContactsSinceLTSA AS int)  AS  customer_NegativeRightPersonContactsSinceLTSA
, CAST(A.customer_PendingEAPNb AS int)  AS  customer_PendingEAPNb
, CAST(A.customer_PositiveEAPIndicator AS int)  AS  customer_PositiveEAPIndicator
, CAST(A.customer_PositiveRightPersonContactsLast3Months AS int)  AS  customer_PositiveRightPersonContactsLast3Months
, CAST(A.customer_PositiveRightPersonContactsLastMonth AS int)  AS  customer_PositiveRightPersonContactsLastMonth
, CAST(A.customer_PositiveRightPersonContactsSinceLTSA AS int)  AS  customer_PositiveRightPersonContactsSinceLTSA
, CAST(A.customer_PostCode AS int)  AS  customer_PostCode
, CAST(A.customer_PromisesLast3Months AS int)  AS  customer_PromisesLast3Months
, CAST(A.customer_PromisesLastMonth AS int)  AS  customer_PromisesLastMonth
, CAST(A.customer_PromisesSinceLTSA AS int)  AS  customer_PromisesSinceLTSA
, CAST(A.customer_ProposedByID AS int)  AS  customer_ProposedByID
, CAST(A.customer_RightPersonContactsLast3Months AS int)  AS  customer_RightPersonContactsLast3Months
, CAST(A.customer_RightPersonContactsLastMonth AS int)  AS  customer_RightPersonContactsLastMonth
, CAST(A.customer_RightPersonContactsSinceLTSA AS int)  AS  customer_RightPersonContactsSinceLTSA
, CAST(A.customer_SolutionTypeID AS int)  AS  customer_SolutionTypeID
, CAST(A.customer_StatusID AS int)  AS  customer_StatusID
, CAST(A.customer_TotalAssets AS int)  AS  customer_TotalAssets
, CAST(A.customer_WatchListType AS int)  AS  customer_WatchListType
, CAST(A.customer_CREV AS decimal(19,2))  AS  customer_CREV
, CAST(A.customer_DT_BorrowerLast12MonthsPayments AS decimal(19,2))  AS  customer_DT_BorrowerLast12MonthsPayments
, CAST(A.customer_DT_BorrowerOnBalance AS decimal(19,2))  AS  customer_DT_BorrowerOnBalance
, CAST(A.customer_DT_BorrowerTotalBalance AS decimal(19,2))  AS  customer_DT_BorrowerTotalBalance
, CAST(A.customer_OffBalanceDebtor AS decimal(19,2))  AS  customer_OffBalanceDebtor
, CAST(A.customer_OnBalanceDebtor AS decimal(19,2))  AS  customer_OnBalanceDebtor
, CAST(A.customer_TotalBalanceDebtor AS decimal(19,2))  AS  customer_TotalBalanceDebtor
, CAST(A.customer_TotalPaidAmountLast12Months AS decimal(19,2))  AS  customer_TotalPaidAmountLast12Months
, CAST(A.customer_TotalPaidAmountLast3Months AS decimal(19,2))  AS  customer_TotalPaidAmountLast3Months
, CAST(A.customer_TotalPaidAmountLastMonth AS decimal(19,2))  AS  customer_TotalPaidAmountLastMonth
, CAST(A.customer_TotalPaidAmountSinceLTSA AS decimal(19,2))  AS  customer_TotalPaidAmountSinceLTSA
, CAST(A.customer_ApplicationDate AS timestamp)  AS  customer_ApplicationDate
, CAST(A.customer_CompletionTargetDate AS timestamp)  AS  customer_CompletionTargetDate
, CAST(A.customer_ContactActivitiesLastDate AS timestamp)  AS  customer_ContactActivitiesLastDate
, CAST(A.customer_DT_DenouncementDate AS timestamp)  AS  customer_DT_DenouncementDate
, CAST(A.customer_IntentionDate AS timestamp)  AS  customer_IntentionDate
, CAST(A.customer_LastPaymentDate AS timestamp)  AS  customer_LastPaymentDate
, CAST(A.customer_NegativeRightPersonContactLastDate AS timestamp)  AS  customer_NegativeRightPersonContactLastDate
, CAST(A.customer_PositiveRightPersonContactLastDate AS timestamp)  AS  customer_PositiveRightPersonContactLastDate
, CAST(A.customer_PromiseLastDate AS timestamp)  AS  customer_PromiseLastDate
, CAST(A.customer_RPCLastDate AS timestamp)  AS  customer_RPCLastDate
, CAST(A.customer_PortfolioCode AS string)  AS  customer_PortfolioCode
, CAST(A.customer_PortfolioName AS string)  AS  customer_PortfolioName
, CAST(A.customer_PortfolioDescription AS string)  AS  customer_PortfolioDescription
, CAST(A.customer_InvestorCode AS string)  AS  customer_InvestorCode
, CAST(A.customer_InvestorName AS string)  AS  customer_InvestorName
, CAST(A.customer_InvestorDescription AS string)  AS  customer_InvestorDescription
, CAST(A.customer_CustomerCode AS string)  AS  customer_CustomerCode
, CAST(A.customer_DateOfBirth AS timestamp)  AS  customer_DateOfBirth
, CAST(A.customer_VATNumber AS string)  AS  customer_VATNumber
, CAST(A.customer_CustomerName AS string)  AS  customer_CustomerName
, CAST(A.customer_CustomerType AS string)  AS  customer_CustomerType
, CAST(A.customer_CustomerTypeDescription AS string)  AS  customer_CustomerTypeDescription
, CAST(A.customer_MonthlyInstallmentsOtherBanks AS decimal(19,2))  AS  customer_MonthlyInstallmentsOtherBanks
, CAST(A.customer_MonthlyInstallmentsOtherBanksCurrency AS string)  AS  customer_MonthlyInstallmentsOtherBanksCurrency
, CAST(A.customer_MonthlyInstallmentsOtherBanksInBaseCurrency AS decimal(19,2))  AS  customer_MonthlyInstallmentsOtherBanksInBaseCurrency
, CAST(A.customer_DepositsAmountTotal AS decimal(19,2))  AS  customer_DepositsAmountTotal
, CAST(A.customer_DepositsAmountTotalCurrency AS string)  AS  customer_DepositsAmountTotalCurrency
, CAST(A.customer_DepositsAmountTotalInBaseCurrency AS decimal(19,2))  AS  customer_DepositsAmountTotalInBaseCurrency
, CAST(A.customer_TimeDepositsAmountTotal AS decimal(19,2))  AS  customer_TimeDepositsAmountTotal
, CAST(A.customer_TimeDepositsAmountTotalCurrency AS string)  AS  customer_TimeDepositsAmountTotalCurrency
, CAST(A.customer_TimeDepositsAmountTotalInBaseCurrency AS decimal(19,2))  AS  customer_TimeDepositsAmountTotalInBaseCurrency
, CAST(A.customer_InvestmentsAmountTotal AS decimal(19,2))  AS  customer_InvestmentsAmountTotal
, CAST(A.customer_InvestmentsAmountTotalCurrency AS string)  AS  customer_InvestmentsAmountTotalCurrency
, CAST(A.customer_InvestmentsAmountTotalInBaseCurrency AS decimal(19,2))  AS  customer_InvestmentsAmountTotalInBaseCurrency
, CAST(A.customer_IsDeceased AS int)  AS  customer_IsDeceased
, CAST(A.customer_IsDeceasedDesctiption AS string)  AS  customer_IsDeceasedDesctiption
, CAST(A.customer_WholesaleIndicator AS int)  AS  customer_WholesaleIndicator
, CAST(A.customer_InsolvencyIndicator AS int)  AS  customer_InsolvencyIndicator
, CAST(A.customer_PrivateBankingIndicator AS int)  AS  customer_PrivateBankingIndicator
, CAST(A.customer_StaffIndicator AS int)  AS  customer_StaffIndicator
, CAST(A.customer_GroupID AS int)  AS  customer_GroupID
, CAST(A.customer_BankruptcyFlagDescription AS string)  AS  customer_BankruptcyFlagDescription
, CAST(A.customer_BorrowerIndustryCodeDescription AS string)  AS  customer_BorrowerIndustryCodeDescription
, CAST(A.customer_ConnectedBorrowersDescription AS string)  AS  customer_ConnectedBorrowersDescription
, CAST(A.customer_DtBorrowerTypeDescription AS string)  AS  customer_DtBorrowerTypeDescription
, CAST(A.customer_DtDenouncementFlagDescription AS string)  AS  customer_DtDenouncementFlagDescription
, CAST(A.customer_DtExtrajudicialLetterFlagDescription AS string)  AS  customer_DtExtrajudicialLetterFlagDescription
, CAST(A.customer_DtPaymentOrderFlagDescription AS string)  AS  customer_DtPaymentOrderFlagDescription
, CAST(A.customer_FinalMaxBucketPerPortfpolioDescription AS string)  AS  customer_FinalMaxBucketPerPortfpolioDescription
, CAST(A.customer_GenderDescription AS string)  AS  customer_GenderDescription
, CAST(A.customer_MaritalStatusDescription AS string)  AS  customer_MaritalStatusDescription
, CAST(A.customer_NegativeEapindicatorDescription AS string)  AS  customer_NegativeEapindicatorDescription
, CAST(A.customer_PositiveEapindicatorDescription AS string)  AS  customer_PositiveEapindicatorDescription
, CAST(A.customer_PostCodeDescription AS string)  AS  customer_PostCodeDescription
, CAST(A.customer_SolutionTypeIdDescription AS string)  AS  customer_SolutionTypeIdDescription
, CAST(A.customer_StatusIdDescription AS string)  AS  customer_StatusIdDescription
, CAST(A.customer_WatchListTypeDescription AS string)  AS  customer_WatchListTypeDescription
, CAST(A.customer_WholesaleIndicatorDescription AS string)  AS  customer_WholesaleIndicatorDescription
, CAST(A.customer_InsolvencyIndicatorDescription AS string)  AS  customer_InsolvencyIndicatorDescription
, CAST(A.customer_PrivateBankingIndicatorDescription AS string)  AS  customer_PrivateBankingIndicatorDescription
, CAST(A.customer_StaffIndicatorDescription AS string)  AS  customer_StaffIndicatorDescription
, CAST(A.customer_PoliticallyExposedPersons AS int)  AS  customer_PoliticallyExposedPersons
, CAST(A.customer_PoliticallyExposedPersonsDescription AS string)  AS  customer_PoliticallyExposedPersonsDescription
, CAST(A.customer_CustomerLastName AS string)  AS  customer_CustomerLastName
, CAST(A.customer_CompanyName AS string)  AS  customer_CompanyName
, CAST(A.customer_Payer AS int)  AS  customer_Payer
, CAST(A.customer_PayerDescription AS string)  AS  customer_PayerDescription
, CAST(A.customer_InterbankIndicator AS int)  AS  customer_InterbankIndicator
, CAST(A.customer_InterbankIndicatorDescription AS string)  AS  customer_InterbankIndicatorDescription
, CAST(A.customer_InterbankPaymentAmount AS decimal(19,2))  AS  customer_InterbankPaymentAmount
, CAST(A.customer_PostalAddress AS string)  AS  customer_PostalAddress
, CAST(A.customer_MobilePhone AS string)  AS  customer_MobilePhone
, CAST(A.customer_IdNumber AS string)  AS  customer_IdNumber
, CAST(A.customer_Email AS string)  AS  customer_Email
, CAST(A.customer_HomePhoneNumber AS string)  AS  customer_HomePhoneNumber
, CAST(A.customer_WorkPhoneNumber AS string)  AS  customer_WorkPhoneNumber
, CAST(A.customer_IsCovid19AffectedIndicator AS boolean)  AS  customer_IsCovid19AffectedIndicator
, CAST(A.customer_IsDEKIndicator AS boolean)  AS  customer_IsDEKIndicator
, CAST(A.customer_IsLaw3869Indicator AS boolean)  AS  customer_IsLaw3869Indicator
, CAST(A.customer_IsPEPIndicator AS boolean)  AS  customer_IsPEPIndicator
, CAST(A.customer_DeclaredAffordability AS decimal(19,2))  AS  customer_DeclaredAffordability
, CAST(A.customer_StatedAffordability AS decimal(19,2))  AS  customer_StatedAffordability
, CAST(A.customer_IncomeUsedForCalculations AS decimal(19,2))  AS  customer_IncomeUsedForCalculations
, CAST(A.customer_LegalRepresentativeExpirationDate AS timestamp)  AS  customer_LegalRepresentativeExpirationDate
, CAST(A.loan_Id AS int)  AS  loan_Id
, CAST(A.loan_CustomerId AS int)  AS  loan_CustomerId
, CAST(A.loan_AssignmentChannel AS string)  AS  loan_AssignmentChannel
, CAST(A.loan_AssignmentOfficerName AS string)  AS  loan_AssignmentOfficerName
, CAST(A.loan_AssignmentType AS string)  AS  loan_AssignmentType
, CAST(A.loan_CapitalizationCV19 AS string)  AS  loan_CapitalizationCV19
, CAST(A.loan_CurrentOccupation AS string)  AS  loan_CurrentOccupation
, CAST(A.loan_DebtLegalProcessStatus AS int)  AS  loan_DebtLegalProcessStatus
, CAST(A.loan_EBAStatusBeforCV19 AS string)  AS  loan_EBAStatusBeforCV19
, CAST(A.loan_EBAStatusSOM AS string)  AS  loan_EBAStatusSOM
, CAST(A.loan_ProductName AS string)  AS  loan_ProductName
, CAST(A.loan_Queue AS string)  AS  loan_Queue
, CAST(A.loan_Strategy AS string)  AS  loan_Strategy
, CAST(A.loan_StrategyStepDescription AS string)  AS  loan_StrategyStepDescription
, CAST(A.loan_ActiveApplication AS int)  AS  loan_ActiveApplication
, CAST(A.loan_AmortizationType AS int)  AS  loan_AmortizationType
, CAST(A.loan_AssignmentTeam AS int)  AS  loan_AssignmentTeam
, CAST(A.loan_CaseWatchlList AS int)  AS  loan_CaseWatchlList
, CAST(A.loan_CollateralExistenceFlag AS int)  AS  loan_CollateralExistenceFlag
, CAST(A.loan_ContactActivities AS int)  AS  loan_ContactActivities
, CAST(A.loan_ContactActivitiesLast3Months AS int)  AS  loan_ContactActivitiesLast3Months
, CAST(A.loan_ContactActivitiesLastMonth AS int)  AS  loan_ContactActivitiesLastMonth
, CAST(A.loan_GracePeriodIndicator AS int)  AS  loan_GracePeriodIndicator
, CAST(A.loan_LastApplicationType AS int)  AS  loan_LastApplicationType
, CAST(A.loan_Law3869Type AS int)  AS  loan_Law3869Type
, CAST(A.loan_LegalProcessStatusDate AS int)  AS  loan_LegalProcessStatusDate
, CAST(A.loan_LoanIDSource AS int)  AS  loan_LoanIDSource
, CAST(A.loan_LT12bIndicator AS int)  AS  loan_LT12bIndicator
, CAST(A.loan_MinBucket AS int)  AS  loan_MinBucket
, CAST(A.loan_NegativeRightPersonContactsLast3Months AS int)  AS  loan_NegativeRightPersonContactsLast3Months
, CAST(A.loan_NegativeRightPersonContactsLastMonth AS int)  AS  loan_NegativeRightPersonContactsLastMonth
, CAST(A.loan_NegativeRightPersonContactsSinceLTSA AS int)  AS  loan_NegativeRightPersonContactsSinceLTSA
, CAST(A.loan_ObjectiveStatus AS int)  AS  loan_ObjectiveStatus
, CAST(A.loan_OperationalBucketBeforCV19 AS int)  AS  loan_OperationalBucketBeforCV19
, CAST(A.loan_PerformingStatus AS int)  AS  loan_PerformingStatus
, CAST(A.loan_PositiveRightPersonContactsLast3Months AS int)  AS  loan_PositiveRightPersonContactsLast3Months
, CAST(A.loan_PositiveRightPersonContactsLastMonth AS int)  AS  loan_PositiveRightPersonContactsLastMonth
, CAST(A.loan_PositiveRightPersonContactsSinceLTSA AS int)  AS  loan_PositiveRightPersonContactsSinceLTSA
, CAST(A.loan_PreviousApplications AS int)  AS  loan_PreviousApplications
, CAST(A.loan_PromisesLast3Months AS int)  AS  loan_PromisesLast3Months
, CAST(A.loan_PromisesLastMonth AS int)  AS  loan_PromisesLastMonth
, CAST(A.loan_PromisesSinceLTSA AS int)  AS  loan_PromisesSinceLTSA
, CAST(A.loan_QueID AS int)  AS  loan_QueID
, CAST(A.loan_RemainingProbationMonths AS int)  AS  loan_RemainingProbationMonths
, CAST(A.loan_RightPersonContactsLast3Months AS int)  AS  loan_RightPersonContactsLast3Months
, CAST(A.loan_RightPersonContactsLastMonth AS int)  AS  loan_RightPersonContactsLastMonth
, CAST(A.loan_RightPersonContactsSinceLTSA AS int)  AS  loan_RightPersonContactsSinceLTSA
, CAST(A.loan_RiskBucketBeforeCV19 AS int)  AS  loan_RiskBucketBeforeCV19
, CAST(A.loan_RiskBucketSOM AS int)  AS  loan_RiskBucketSOM
, CAST(A.loan_SettlementBucket AS int)  AS  loan_SettlementBucket
, CAST(A.loan_SPV AS int)  AS  loan_SPV
, CAST(A.loan_AssignmentBalance AS decimal(19,2))  AS  loan_AssignmentBalance
, CAST(A.loan_CreditLine AS decimal(19,2))  AS  loan_CreditLine
, CAST(A.loan_DT_AccountOnBalance AS decimal(19,2))  AS  loan_DT_AccountOnBalance
, CAST(A.loan_DT_AccountTotalBalance AS decimal(19,2))  AS  loan_DT_AccountTotalBalance
, CAST(A.loan_DT_Last12MonthsPayments AS decimal(19,2))  AS  loan_DT_Last12MonthsPayments
, CAST(A.loan_OffBalance AS decimal(19,2))  AS  loan_OffBalance
, CAST(A.loan_OnBalance AS decimal(19,2))  AS  loan_OnBalance
, CAST(A.loan_TotalPaidAmountLast12Months AS decimal(19,2))  AS  loan_TotalPaidAmountLast12Months
, CAST(A.loan_TotalPaidAmountLast3Months AS decimal(19,2))  AS  loan_TotalPaidAmountLast3Months
, CAST(A.loan_TotalPaidAmountLastMonth AS decimal(19,2))  AS  loan_TotalPaidAmountLastMonth
, CAST(A.loan_TotalPaidAmountSinceLTSA AS decimal(19,2))  AS  loan_TotalPaidAmountSinceLTSA
, CAST(A.loan_AccountStatusChageDate AS timestamp)  AS  loan_AccountStatusChageDate
, CAST(A.loan_AssignmentDate AS timestamp)  AS  loan_AssignmentDate
, CAST(A.loan_ContactActivitiesLastDate AS timestamp)  AS  loan_ContactActivitiesLastDate
, CAST(A.loan_FixedInterestEndDate AS timestamp)  AS  loan_FixedInterestEndDate
, CAST(A.loan_L3869CourtDate AS timestamp)  AS  loan_L3869CourtDate
, CAST(A.loan_LastApplicationDate AS timestamp)  AS  loan_LastApplicationDate
, CAST(A.loan_LastPaymentDate AS timestamp)  AS  loan_LastPaymentDate
, CAST(A.loan_NegativeRightPersonContactLastDate AS timestamp)  AS  loan_NegativeRightPersonContactLastDate
, CAST(A.loan_ObjectiveStatusDate AS timestamp)  AS  loan_ObjectiveStatusDate
, CAST(A.loan_PositiveRightPersonContactLastDate AS timestamp)  AS  loan_PositiveRightPersonContactLastDate
, CAST(A.loan_ProbationStartDate AS timestamp)  AS  loan_ProbationStartDate
, CAST(A.loan_PromiseLastDate AS timestamp)  AS  loan_PromiseLastDate
, CAST(A.loan_RightPersonContactLastDate AS timestamp)  AS  loan_RightPersonContactLastDate
, CAST(A.loan_OpenFlag AS boolean)  AS  loan_OpenFlag
, CAST(A.loan_AccountNumber AS string)  AS  loan_AccountNumber
, CAST(A.loan_AssetClass AS int)  AS  loan_AssetClass
, CAST(A.loan_AssetClassDesciption AS string)  AS  loan_AssetClassDesciption
, CAST(A.loan_AssociationType AS int)  AS  loan_AssociationType
, CAST(A.loan_AssociationTypeDescription AS string)  AS  loan_AssociationTypeDescription
, CAST(A.loan_ContractNumber AS string)  AS  loan_ContractNumber
, CAST(A.loan_IsAutomobile AS int)  AS  loan_IsAutomobile
, CAST(A.loan_HasGovernmentGuarantee AS int)  AS  loan_HasGovernmentGuarantee
, CAST(A.loan_OutstandingBalance AS decimal(19,2))  AS  loan_OutstandingBalance
, CAST(A.loan_OutstandingBalanceInBaseCurrency AS decimal(19,2))  AS  loan_OutstandingBalanceInBaseCurrency
, CAST(A.loan_RemainingTenorInMonths AS int)  AS  loan_RemainingTenorInMonths
, CAST(A.loan_Bucket AS int)  AS  loan_Bucket
, CAST(A.loan_CurrentInterestRate AS decimal(19,2))  AS  loan_CurrentInterestRate
, CAST(A.loan_EBAStatus AS int)  AS  loan_EBAStatus
, CAST(A.loan_EBAStatusDescription AS string)  AS  loan_EBAStatusDescription
, CAST(A.loan_Currency AS string)  AS  loan_Currency
, CAST(A.loan_CurrentInstallment AS decimal(19,2))  AS  loan_CurrentInstallment
, CAST(A.loan_CurrentInstallmentInBaseCurrency AS decimal(19,2))  AS  loan_CurrentInstallmentInBaseCurrency
, CAST(A.loan_DaysPastDue AS int)  AS  loan_DaysPastDue
, CAST(A.loan_DebtAmount AS decimal(19,2))  AS  loan_DebtAmount
, CAST(A.loan_DebtAmountInBaseCurrency AS decimal(19,2))  AS  loan_DebtAmountInBaseCurrency
, CAST(A.loan_FrozenInterest AS decimal(19,2))  AS  loan_FrozenInterest
, CAST(A.loan_FrozenInterestInBaseCurrency AS decimal(19,2))  AS  loan_FrozenInterestInBaseCurrency
, CAST(A.loan_NotAccrudInterest AS decimal(10,8))  AS  loan_NotAccrudInterest
, CAST(A.loan_NotAccrudInterestInBaseCurrency AS decimal(19,2))  AS  loan_NotAccrudInterestInBaseCurrency
, CAST(A.loan_DepreciatedAmount AS decimal(19,2))  AS  loan_DepreciatedAmount
, CAST(A.loan_DepreciatedAmountInBaseCurrency AS decimal(19,2))  AS  loan_DepreciatedAmountInBaseCurrency
, CAST(A.loan_Expenses AS decimal(19,2))  AS  loan_Expenses
, CAST(A.loan_ExpensesInBaseCurrency AS decimal(19,2))  AS  loan_ExpensesInBaseCurrency
, CAST(A.loan_HasLegalActions AS int)  AS  loan_HasLegalActions
, CAST(A.loan_BorrowerAccountSegment AS int)  AS  loan_BorrowerAccountSegment
, CAST(A.loan_NumberOfPreviousModifications AS int)  AS  loan_NumberOfPreviousModifications
, CAST(A.loan_MaxDaysPastDueEver AS int)  AS  loan_MaxDaysPastDueEver
, CAST(A.loan_LegalProcessStatus AS int)  AS  loan_LegalProcessStatus
, CAST(A.loan_DenouncedCaseEver AS int)  AS  loan_DenouncedCaseEver
, CAST(A.loan_IsRevolvingLoan AS int)  AS  loan_IsRevolvingLoan
, CAST(A.loan_LoanOriginationDate AS timestamp)  AS  loan_LoanOriginationDate
, CAST(A.loan_ProductCode AS string)  AS  loan_ProductCode
, CAST(A.loan_MonthsSinceMostRecentModification AS int)  AS  loan_MonthsSinceMostRecentModification
, CAST(A.loan_AccountStatus AS string)  AS  loan_AccountStatus
, CAST(A.loan_AccountStatusChangeDate AS timestamp)  AS  loan_AccountStatusChangeDate
, CAST(A.loan_PermanentDelayStatusTurnDate AS timestamp)  AS  loan_PermanentDelayStatusTurnDate
, CAST(A.loan_CardNumber AS string)  AS  loan_CardNumber
, CAST(A.loan_IsDenounced AS int)  AS  loan_IsDenounced
, CAST(A.loan_BaseInterest AS decimal(19,2))  AS  loan_BaseInterest
, CAST(A.loan_Law128Rate AS decimal(19,2))  AS  loan_Law128Rate
, CAST(A.loan_Spread AS decimal(19,2))  AS  loan_Spread
, CAST(A.loan_IsFixedInterest AS int)  AS  loan_IsFixedInterest
, CAST(A.loan_ContractualInterestRate AS decimal(19,2))  AS  loan_ContractualInterestRate
, CAST(A.loan_CurrentInterestAmount AS decimal(19,2))  AS  loan_CurrentInterestAmount
, CAST(A.loan_CurrentInterestAmountInBaseCurrency AS decimal(19,2))  AS  loan_CurrentInterestAmountInBaseCurrency
, CAST(A.loan_AgencyType AS int)  AS  loan_AgencyType
, CAST(A.loan_BaseInterestType AS int)  AS  loan_BaseInterestType
, CAST(A.loan_Delivered AS int)  AS  loan_Delivered
, CAST(A.loan_AccountOwnerId AS string)  AS  loan_AccountOwnerId
, CAST(A.loan_LoanPurpose AS int)  AS  loan_LoanPurpose
, CAST(A.loan_ParticipationPercentage AS decimal(19,2))  AS  loan_ParticipationPercentage
, CAST(A.loan_ExchangeRateAtOrigination AS decimal(19,2))  AS  loan_ExchangeRateAtOrigination
, CAST(A.loan_IsOverdraft AS int)  AS  loan_IsOverdraft
, CAST(A.loan_CreditLimit AS decimal(19,2))  AS  loan_CreditLimit
, CAST(A.loan_CreditLimitInBaseCurrency AS decimal(19,2))  AS  loan_CreditLimitInBaseCurrency
, CAST(A.loan_ActiveApplicationDescription AS string)  AS  loan_ActiveApplicationDescription
, CAST(A.loan_AmortizationTypeDescription AS string)  AS  loan_AmortizationTypeDescription
, CAST(A.loan_AssignmentTeamDescription AS string)  AS  loan_AssignmentTeamDescription
, CAST(A.loan_CaseWatchlListDescription AS string)  AS  loan_CaseWatchlListDescription
, CAST(A.loan_GracePeriodIndicatorDescription AS string)  AS  loan_GracePeriodIndicatorDescription
, CAST(A.loan_LastApplicationTypeDescription AS string)  AS  loan_LastApplicationTypeDescription
, CAST(A.loan_Law3869TypeDescription AS string)  AS  loan_Law3869TypeDescription
, CAST(A.loan_LegalProcessStatusDateDescription AS string)  AS  loan_LegalProcessStatusDateDescription
, CAST(A.loan_Lt12bIndicatorDescription AS string)  AS  loan_Lt12bIndicatorDescription
, CAST(A.loan_MinBucketDescription AS string)  AS  loan_MinBucketDescription
, CAST(A.loan_ObjectiveStatusDescription AS string)  AS  loan_ObjectiveStatusDescription
, CAST(A.loan_PerformingStatusDescription AS string)  AS  loan_PerformingStatusDescription
, CAST(A.loan_QueIdDescription AS string)  AS  loan_QueIdDescription
, CAST(A.loan_SettlementBucketDescription AS string)  AS  loan_SettlementBucketDescription
, CAST(A.loan_SpvDescription AS string)  AS  loan_SpvDescription
, CAST(A.loan_IsAutomobileDescription AS string)  AS  loan_IsAutomobileDescription
, CAST(A.loan_HasGovernmentGuaranteeDescription AS string)  AS  loan_HasGovernmentGuaranteeDescription
, CAST(A.loan_HasLegalActionsDescription AS string)  AS  loan_HasLegalActionsDescription
, CAST(A.loan_BorrowerAccountSegmentDescription AS string)  AS  loan_BorrowerAccountSegmentDescription
, CAST(A.loan_LegalProcessStatusDescription AS string)  AS  loan_LegalProcessStatusDescription
, CAST(A.loan_DenounceCaseEverDescription AS string)  AS  loan_DenounceCaseEverDescription
, CAST(A.loan_IsRevolvingLoanDescription AS string)  AS  loan_IsRevolvingLoanDescription
, CAST(A.loan_IsDenouncedDescription AS string)  AS  loan_IsDenouncedDescription
, CAST(A.loan_IsFixedInterestDescription AS string)  AS  loan_IsFixedInterestDescription
, CAST(A.loan_AgencyTypeDescription AS string)  AS  loan_AgencyTypeDescription
, CAST(A.loan_BaseInterestTypeDescription AS string)  AS  loan_BaseInterestTypeDescription
, CAST(A.loan_DeliveredDescription AS string)  AS  loan_DeliveredDescription
, CAST(A.loan_LoanPurposeDescription AS string)  AS  loan_LoanPurposeDescription
, CAST(A.loan_IsOverdraftDescription AS string)  AS  loan_IsOverdraftDescription
, CAST(A.loan_PositiveLoanEAPIndicator AS string)  AS  loan_PositiveLoanEAPIndicator
, CAST(A.loan_PositiveLoanEAPIndicatorDescription AS string)  AS  loan_PositiveLoanEAPIndicatorDescription
, CAST(A.loan_ContractNumberSContractNumberSystemicNrystemicNr AS string)  AS  loan_ContractNumberSContractNumberSystemicNrystemicNr
, CAST(A.loan_EligiblePrenotedAsset AS int)  AS  loan_EligiblePrenotedAsset
, CAST(A.loan_EligiblePrenotedAssetDescription AS string)  AS  loan_EligiblePrenotedAssetDescription
, CAST(A.loan_AccountPayer AS int)  AS  loan_AccountPayer
, CAST(A.loan_AccountPayerDescription AS string)  AS  loan_AccountPayerDescription
, CAST(A.loan_AccountNumberWithPrefix AS string)  AS  loan_AccountNumberWithPrefix
, CAST(A.loan_ContractNumberSystemicNr AS string)  AS  loan_ContractNumberSystemicNr
, CAST(A.loan_UnifiedLoanId AS int)  AS  loan_UnifiedLoanId
, CAST(A.loan_IsUnifiedLoan AS boolean)  AS  loan_IsUnifiedLoan
, CAST(A.loan_DecisionTreeCode AS string)  AS  loan_DecisionTreeCode
, CAST(A.loan_DecisionTreeDescription AS string)  AS  loan_DecisionTreeDescription
, CAST(A.loan_DecisionTreeValidPath AS string)  AS  loan_DecisionTreeValidPath
, CAST(A.loan_IsForModification AS boolean)  AS  loan_IsForModification
, CAST(A.unifiedLoan_Id AS int)  AS  unifiedLoan_Id
, CAST(A.unifiedLoan_AssignmentChannel AS string)  AS  unifiedLoan_AssignmentChannel
, CAST(A.unifiedLoan_AssignmentOfficerName AS string)  AS  unifiedLoan_AssignmentOfficerName
, CAST(A.unifiedLoan_AssignmentType AS string)  AS  unifiedLoan_AssignmentType
, CAST(A.unifiedLoan_CapitalizationCV19 AS string)  AS  unifiedLoan_CapitalizationCV19
, CAST(A.unifiedLoan_CurrentOccupation AS string)  AS  unifiedLoan_CurrentOccupation
, CAST(A.unifiedLoan_DebtLegalProcessStatus AS int)  AS  unifiedLoan_DebtLegalProcessStatus
, CAST(A.unifiedLoan_EBAStatusBeforCV19 AS string)  AS  unifiedLoan_EBAStatusBeforCV19
, CAST(A.unifiedLoan_EBAStatusSOM AS string)  AS  unifiedLoan_EBAStatusSOM
, CAST(A.unifiedLoan_ProductName AS string)  AS  unifiedLoan_ProductName
, CAST(A.unifiedLoan_Queue AS string)  AS  unifiedLoan_Queue
, CAST(A.unifiedLoan_Strategy AS string)  AS  unifiedLoan_Strategy
, CAST(A.unifiedLoan_StrategyStepDescription AS string)  AS  unifiedLoan_StrategyStepDescription
, CAST(A.unifiedLoan_ActiveApplication AS int)  AS  unifiedLoan_ActiveApplication
, CAST(A.unifiedLoan_AmortizationType AS int)  AS  unifiedLoan_AmortizationType
, CAST(A.unifiedLoan_AssignmentTeam AS int)  AS  unifiedLoan_AssignmentTeam
, CAST(A.unifiedLoan_CaseWatchlList AS int)  AS  unifiedLoan_CaseWatchlList
, CAST(A.unifiedLoan_CollateralExistenceFlag AS int)  AS  unifiedLoan_CollateralExistenceFlag
, CAST(A.unifiedLoan_ContactActivities AS int)  AS  unifiedLoan_ContactActivities
, CAST(A.unifiedLoan_ContactActivitiesLast3Months AS int)  AS  unifiedLoan_ContactActivitiesLast3Months
, CAST(A.unifiedLoan_ContactActivitiesLastMonth AS int)  AS  unifiedLoan_ContactActivitiesLastMonth
, CAST(A.unifiedLoan_GracePeriodIndicator AS int)  AS  unifiedLoan_GracePeriodIndicator
, CAST(A.unifiedLoan_LastApplicationType AS int)  AS  unifiedLoan_LastApplicationType
, CAST(A.unifiedLoan_Law3869Type AS int)  AS  unifiedLoan_Law3869Type
, CAST(A.unifiedLoan_LegalProcessStatusDate AS int)  AS  unifiedLoan_LegalProcessStatusDate
, CAST(A.unifiedLoan_LoanIDSource AS int)  AS  unifiedLoan_LoanIDSource
, CAST(A.unifiedLoan_LT12bIndicator AS int)  AS  unifiedLoan_LT12bIndicator
, CAST(A.unifiedLoan_MinBucket AS int)  AS  unifiedLoan_MinBucket
, CAST(A.unifiedLoan_NegativeRightPersonContactsLast3Months AS int)  AS  unifiedLoan_NegativeRightPersonContactsLast3Months
, CAST(A.unifiedLoan_NegativeRightPersonContactsLastMonth AS int)  AS  unifiedLoan_NegativeRightPersonContactsLastMonth
, CAST(A.unifiedLoan_NegativeRightPersonContactsSinceLTSA AS int)  AS  unifiedLoan_NegativeRightPersonContactsSinceLTSA
, CAST(A.unifiedLoan_ObjectiveStatus AS int)  AS  unifiedLoan_ObjectiveStatus
, CAST(A.unifiedLoan_OperationalBucketBeforCV19 AS int)  AS  unifiedLoan_OperationalBucketBeforCV19
, CAST(A.unifiedLoan_PerformingStatus AS int)  AS  unifiedLoan_PerformingStatus
, CAST(A.unifiedLoan_PositiveRightPersonContactsLast3Months AS int)  AS  unifiedLoan_PositiveRightPersonContactsLast3Months
, CAST(A.unifiedLoan_PositiveRightPersonContactsLastMonth AS int)  AS  unifiedLoan_PositiveRightPersonContactsLastMonth
, CAST(A.unifiedLoan_PositiveRightPersonContactsSinceLTSA AS int)  AS  unifiedLoan_PositiveRightPersonContactsSinceLTSA
, CAST(A.unifiedLoan_PreviousApplications AS int)  AS  unifiedLoan_PreviousApplications
, CAST(A.unifiedLoan_PromisesLast3Months AS int)  AS  unifiedLoan_PromisesLast3Months
, CAST(A.unifiedLoan_PromisesLastMonth AS int)  AS  unifiedLoan_PromisesLastMonth
, CAST(A.unifiedLoan_PromisesSinceLTSA AS int)  AS  unifiedLoan_PromisesSinceLTSA
, CAST(A.unifiedLoan_QueID AS int)  AS  unifiedLoan_QueID
, CAST(A.unifiedLoan_RemainingProbationMonths AS int)  AS  unifiedLoan_RemainingProbationMonths
, CAST(A.unifiedLoan_RightPersonContactsLast3Months AS int)  AS  unifiedLoan_RightPersonContactsLast3Months
, CAST(A.unifiedLoan_RightPersonContactsLastMonth AS int)  AS  unifiedLoan_RightPersonContactsLastMonth
, CAST(A.unifiedLoan_RightPersonContactsSinceLTSA AS int)  AS  unifiedLoan_RightPersonContactsSinceLTSA
, CAST(A.unifiedLoan_RiskBucketBeforeCV19 AS int)  AS  unifiedLoan_RiskBucketBeforeCV19
, CAST(A.unifiedLoan_RiskBucketSOM AS int)  AS  unifiedLoan_RiskBucketSOM
, CAST(A.unifiedLoan_SettlementBucket AS int)  AS  unifiedLoan_SettlementBucket
, CAST(A.unifiedLoan_SPV AS int)  AS  unifiedLoan_SPV
, CAST(A.unifiedLoan_AssignmentBalance AS decimal(19,2))  AS  unifiedLoan_AssignmentBalance
, CAST(A.unifiedLoan_CreditLine AS decimal(19,2))  AS  unifiedLoan_CreditLine
, CAST(A.unifiedLoan_DT_AccountOnBalance AS decimal(19,2))  AS  unifiedLoan_DT_AccountOnBalance
, CAST(A.unifiedLoan_DT_AccountTotalBalance AS decimal(19,2))  AS  unifiedLoan_DT_AccountTotalBalance
, CAST(A.unifiedLoan_DT_Last12MonthsPayments AS decimal(19,2))  AS  unifiedLoan_DT_Last12MonthsPayments
, CAST(A.unifiedLoan_OffBalance AS decimal(19,2))  AS  unifiedLoan_OffBalance
, CAST(A.unifiedLoan_OnBalance AS decimal(19,2))  AS  unifiedLoan_OnBalance
, CAST(A.unifiedLoan_TotalPaidAmountLast12Months AS decimal(19,2))  AS  unifiedLoan_TotalPaidAmountLast12Months
, CAST(A.unifiedLoan_TotalPaidAmountLast3Months AS decimal(19,2))  AS  unifiedLoan_TotalPaidAmountLast3Months
, CAST(A.unifiedLoan_TotalPaidAmountLastMonth AS decimal(19,2))  AS  unifiedLoan_TotalPaidAmountLastMonth
, CAST(A.unifiedLoan_TotalPaidAmountSinceLTSA AS decimal(19,2))  AS  unifiedLoan_TotalPaidAmountSinceLTSA
, CAST(A.unifiedLoan_AccountStatusChageDate AS timestamp)  AS  unifiedLoan_AccountStatusChageDate
, CAST(A.unifiedLoan_AssignmentDate AS timestamp)  AS  unifiedLoan_AssignmentDate
, CAST(A.unifiedLoan_ContactActivitiesLastDate AS timestamp)  AS  unifiedLoan_ContactActivitiesLastDate
, CAST(A.unifiedLoan_FixedInterestEndDate AS timestamp)  AS  unifiedLoan_FixedInterestEndDate
, CAST(A.unifiedLoan_L3869CourtDate AS timestamp)  AS  unifiedLoan_L3869CourtDate
, CAST(A.unifiedLoan_LastApplicationDate AS timestamp)  AS  unifiedLoan_LastApplicationDate
, CAST(A.unifiedLoan_LastPaymentDate AS timestamp)  AS  unifiedLoan_LastPaymentDate
, CAST(A.unifiedLoan_NegativeRightPersonContactLastDate AS timestamp)  AS  unifiedLoan_NegativeRightPersonContactLastDate
, CAST(A.unifiedLoan_ObjectiveStatusDate AS timestamp)  AS  unifiedLoan_ObjectiveStatusDate
, CAST(A.unifiedLoan_PositiveRightPersonContactLastDate AS timestamp)  AS  unifiedLoan_PositiveRightPersonContactLastDate
, CAST(A.unifiedLoan_ProbationStartDate AS timestamp)  AS  unifiedLoan_ProbationStartDate
, CAST(A.unifiedLoan_PromiseLastDate AS timestamp)  AS  unifiedLoan_PromiseLastDate
, CAST(A.unifiedLoan_RightPersonContactLastDate AS timestamp)  AS  unifiedLoan_RightPersonContactLastDate
, CAST(A.unifiedLoan_OpenFlag AS boolean)  AS  unifiedLoan_OpenFlag
, CAST(A.unifiedLoan_AccountNumber AS string)  AS  unifiedLoan_AccountNumber
, CAST(A.unifiedLoan_AssetClass AS int)  AS  unifiedLoan_AssetClass
, CAST(A.unifiedLoan_AssetClassDesciption AS string)  AS  unifiedLoan_AssetClassDesciption
, CAST(A.unifiedLoan_AssociationType AS int)  AS  unifiedLoan_AssociationType
, CAST(A.unifiedLoan_AssociationTypeDescription AS string)  AS  unifiedLoan_AssociationTypeDescription
, CAST(A.unifiedLoan_ContractNumber AS string)  AS  unifiedLoan_ContractNumber
, CAST(A.unifiedLoan_IsAutomobile AS int)  AS  unifiedLoan_IsAutomobile
, CAST(A.unifiedLoan_HasGovernmentGuarantee AS int)  AS  unifiedLoan_HasGovernmentGuarantee
, CAST(A.unifiedLoan_OutstandingBalance AS decimal(19,2))  AS  unifiedLoan_OutstandingBalance
, CAST(A.unifiedLoan_OutstandingBalanceInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_OutstandingBalanceInBaseCurrency
, CAST(A.unifiedLoan_RemainingTenorInMonths AS int)  AS  unifiedLoan_RemainingTenorInMonths
, CAST(A.unifiedLoan_Bucket AS int)  AS  unifiedLoan_Bucket
, CAST(A.unifiedLoan_CurrentInterestRate AS decimal(19,2))  AS  unifiedLoan_CurrentInterestRate
, CAST(A.unifiedLoan_EBAStatus AS int)  AS  unifiedLoan_EBAStatus
, CAST(A.unifiedLoan_EBAStatusDescription AS string)  AS  unifiedLoan_EBAStatusDescription
, CAST(A.unifiedLoan_Currency AS string)  AS  unifiedLoan_Currency
, CAST(A.unifiedLoan_CurrentInstallment AS decimal(19,2))  AS  unifiedLoan_CurrentInstallment
, CAST(A.unifiedLoan_CurrentInstallmentInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_CurrentInstallmentInBaseCurrency
, CAST(A.unifiedLoan_DaysPastDue AS int)  AS  unifiedLoan_DaysPastDue
, CAST(A.unifiedLoan_DebtAmount AS decimal(19,2))  AS  unifiedLoan_DebtAmount
, CAST(A.unifiedLoan_DebtAmountInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_DebtAmountInBaseCurrency
, CAST(A.unifiedLoan_FrozenInterest AS decimal(19,2))  AS  unifiedLoan_FrozenInterest
, CAST(A.unifiedLoan_FrozenInterestInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_FrozenInterestInBaseCurrency
, CAST(A.unifiedLoan_NotAccrudInterest AS decimal(19,2))  AS  unifiedLoan_NotAccrudInterest
, CAST(A.unifiedLoan_NotAccrudInterestInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_NotAccrudInterestInBaseCurrency
, CAST(A.unifiedLoan_DepreciatedAmount AS decimal(19,2))  AS  unifiedLoan_DepreciatedAmount
, CAST(A.unifiedLoan_DepreciatedAmountInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_DepreciatedAmountInBaseCurrency
, CAST(A.unifiedLoan_Expenses AS decimal(19,2))  AS  unifiedLoan_Expenses
, CAST(A.unifiedLoan_ExpensesInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_ExpensesInBaseCurrency
, CAST(A.unifiedLoan_HasLegalActions AS int)  AS  unifiedLoan_HasLegalActions
, CAST(A.unifiedLoan_BorrowerAccountSegment AS int)  AS  unifiedLoan_BorrowerAccountSegment
, CAST(A.unifiedLoan_NumberOfPreviousModifications AS int)  AS  unifiedLoan_NumberOfPreviousModifications
, CAST(A.unifiedLoan_MaxDaysPastDueEver AS int)  AS  unifiedLoan_MaxDaysPastDueEver
, CAST(A.unifiedLoan_LegalProcessStatus AS int)  AS  unifiedLoan_LegalProcessStatus
, CAST(A.unifiedLoan_DenouncedCaseEver AS int)  AS  unifiedLoan_DenouncedCaseEver
, CAST(A.unifiedLoan_IsRevolvingLoan AS int)  AS  unifiedLoan_IsRevolvingLoan
, CAST(A.unifiedLoan_LoanOriginationDate AS timestamp)  AS  unifiedLoan_LoanOriginationDate
, CAST(A.unifiedLoan_ProductCode AS string)  AS  unifiedLoan_ProductCode
, CAST(A.unifiedLoan_MonthsSinceMostRecentModification AS int)  AS  unifiedLoan_MonthsSinceMostRecentModification
, CAST(A.unifiedLoan_AccountStatus AS string)  AS  unifiedLoan_AccountStatus
, CAST(A.unifiedLoan_AccountStatusChangeDate AS timestamp)  AS  unifiedLoan_AccountStatusChangeDate
, CAST(A.unifiedLoan_PermanentDelayStatusTurnDate AS timestamp)  AS  unifiedLoan_PermanentDelayStatusTurnDate
, CAST(A.unifiedLoan_CardNumber AS string)  AS  unifiedLoan_CardNumber
, CAST(A.unifiedLoan_IsDenounced AS int)  AS  unifiedLoan_IsDenounced
, CAST(A.unifiedLoan_BaseInterest AS decimal(19,2))  AS  unifiedLoan_BaseInterest
, CAST(A.unifiedLoan_Law128Rate AS decimal(19,2))  AS  unifiedLoan_Law128Rate
, CAST(A.unifiedLoan_Spread AS decimal(19,2))  AS  unifiedLoan_Spread
, CAST(A.unifiedLoan_IsFixedInterest AS int)  AS  unifiedLoan_IsFixedInterest
, CAST(A.unifiedLoan_ContractualInterestRate AS decimal(19,2))  AS  unifiedLoan_ContractualInterestRate
, CAST(A.unifiedLoan_CurrentInterestAmount AS decimal(19,2))  AS  unifiedLoan_CurrentInterestAmount
, CAST(A.unifiedLoan_CurrentInterestAmountInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_CurrentInterestAmountInBaseCurrency
, CAST(A.unifiedLoan_AgencyType AS int)  AS  unifiedLoan_AgencyType
, CAST(A.unifiedLoan_BaseInterestType AS int)  AS  unifiedLoan_BaseInterestType
, CAST(A.unifiedLoan_Delivered AS int)  AS  unifiedLoan_Delivered
, CAST(A.unifiedLoan_AccountOwnerId AS string)  AS  unifiedLoan_AccountOwnerId
, CAST(A.unifiedLoan_LoanPurpose AS int)  AS  unifiedLoan_LoanPurpose
, CAST(A.unifiedLoan_ParticipationPercentage AS decimal(19,2))  AS  unifiedLoan_ParticipationPercentage
, CAST(A.unifiedLoan_ExchangeRateAtOrigination AS decimal(19,2))  AS  unifiedLoan_ExchangeRateAtOrigination
, CAST(A.unifiedLoan_IsOverdraft AS int)  AS  unifiedLoan_IsOverdraft
, CAST(A.unifiedLoan_CreditLimit AS decimal(19,2))  AS  unifiedLoan_CreditLimit
, CAST(A.unifiedLoan_CreditLimitInBaseCurrency AS decimal(19,2))  AS  unifiedLoan_CreditLimitInBaseCurrency
, CAST(A.unifiedLoan_ActiveApplicationDescription AS string)  AS  unifiedLoan_ActiveApplicationDescription
, CAST(A.unifiedLoan_AmortizationTypeDescription AS string)  AS  unifiedLoan_AmortizationTypeDescription
, CAST(A.unifiedLoan_AssignmentTeamDescription AS string)  AS  unifiedLoan_AssignmentTeamDescription
, CAST(A.unifiedLoan_CaseWatchlListDescription AS string)  AS  unifiedLoan_CaseWatchlListDescription
, CAST(A.unifiedLoan_GracePeriodIndicatorDescription AS string)  AS  unifiedLoan_GracePeriodIndicatorDescription
, CAST(A.unifiedLoan_LastApplicationTypeDescription AS string)  AS  unifiedLoan_LastApplicationTypeDescription
, CAST(A.unifiedLoan_Law3869TypeDescription AS string)  AS  unifiedLoan_Law3869TypeDescription
, CAST(A.unifiedLoan_LegalProcessStatusDateDescription AS string)  AS  unifiedLoan_LegalProcessStatusDateDescription
, CAST(A.unifiedLoan_Lt12bIndicatorDescription AS string)  AS  unifiedLoan_Lt12bIndicatorDescription
, CAST(A.unifiedLoan_MinBucketDescription AS string)  AS  unifiedLoan_MinBucketDescription
, CAST(A.unifiedLoan_ObjectiveStatusDescription AS string)  AS  unifiedLoan_ObjectiveStatusDescription
, CAST(A.unifiedLoan_PerformingStatusDescription AS string)  AS  unifiedLoan_PerformingStatusDescription
, CAST(A.unifiedLoan_QueIdDescription AS string)  AS  unifiedLoan_QueIdDescription
, CAST(A.unifiedLoan_SettlementBucketDescription AS string)  AS  unifiedLoan_SettlementBucketDescription
, CAST(A.unifiedLoan_SpvDescription AS string)  AS  unifiedLoan_SpvDescription
, CAST(A.unifiedLoan_IsAutomobileDescription AS string)  AS  unifiedLoan_IsAutomobileDescription
, CAST(A.unifiedLoan_HasGovernmentGuaranteeDescription AS string)  AS  unifiedLoan_HasGovernmentGuaranteeDescription
, CAST(A.unifiedLoan_HasLegalActionsDescription AS string)  AS  unifiedLoan_HasLegalActionsDescription
, CAST(A.unifiedLoan_BorrowerAccountSegmentDescription AS string)  AS  unifiedLoan_BorrowerAccountSegmentDescription
, CAST(A.unifiedLoan_LegalProcessStatusDescription AS string)  AS  unifiedLoan_LegalProcessStatusDescription
, CAST(A.unifiedLoan_DenounceCaseEverDescription AS string)  AS  unifiedLoan_DenounceCaseEverDescription
, CAST(A.unifiedLoan_IsRevolvingLoanDescription AS string)  AS  unifiedLoan_IsRevolvingLoanDescription
, CAST(A.unifiedLoan_IsDenouncedDescription AS string)  AS  unifiedLoan_IsDenouncedDescription
, CAST(A.unifiedLoan_IsFixedInterestDescription AS string)  AS  unifiedLoan_IsFixedInterestDescription
, CAST(A.unifiedLoan_AgencyTypeDescription AS string)  AS  unifiedLoan_AgencyTypeDescription
, CAST(A.unifiedLoan_BaseInterestTypeDescription AS string)  AS  unifiedLoan_BaseInterestTypeDescription
, CAST(A.unifiedLoan_DeliveredDescription AS string)  AS  unifiedLoan_DeliveredDescription
, CAST(A.unifiedLoan_LoanPurposeDescription AS string)  AS  unifiedLoan_LoanPurposeDescription
, CAST(A.unifiedLoan_IsOverdraftDescription AS string)  AS  unifiedLoan_IsOverdraftDescription
, CAST(A.unifiedLoan_PositiveLoanEAPIndicator AS string)  AS  unifiedLoan_PositiveLoanEAPIndicator
, CAST(A.unifiedLoan_PositiveLoanEAPIndicatorDescription AS string)  AS  unifiedLoan_PositiveLoanEAPIndicatorDescription
, CAST(A.unifiedLoan_ContractNumberSContractNumberSystemicNrystemicNr AS string)  AS  unifiedLoan_ContractNumberSContractNumberSystemicNrystemicNr
, CAST(A.unifiedLoan_EligiblePrenotedAsset AS int)  AS  unifiedLoan_EligiblePrenotedAsset
, CAST(A.unifiedLoan_EligiblePrenotedAssetDescription AS string)  AS  unifiedLoan_EligiblePrenotedAssetDescription
, CAST(A.unifiedLoan_AccountPayer AS int)  AS  unifiedLoan_AccountPayer
, CAST(A.unifiedLoan_AccountPayerDescription AS string)  AS  unifiedLoan_AccountPayerDescription
, CAST(A.unifiedLoan_AccountNumberWithPrefix AS string)  AS  unifiedLoan_AccountNumberWithPrefix
, CAST(A.unifiedLoan_ContractNumberSystemicNr AS string)  AS  unifiedLoan_ContractNumberSystemicNr
, CAST(A.unifiedLoan_DecisionTreeCode AS string)  AS  unifiedLoan_DecisionTreeCode
, CAST(A.unifiedLoan_DecisionTreeDescription AS string)  AS  unifiedLoan_DecisionTreeDescription
, CAST(A.unifiedLoan_DecisionTreeValidPath AS string)  AS  unifiedLoan_DecisionTreeValidPath
, CAST(A.loanId AS int)  AS  loanId
, CAST(A.product_Id AS varchar(36))  AS  product_Id
, CAST(A.product_LoanId AS int)  AS  product_LoanId
, CAST(A.product_ProductKey AS varchar(36))  AS  product_ProductKey
, CAST(A.product_Priority AS int)  AS  product_Priority
, CAST(A.product_ProductTypePriority AS int)  AS  product_ProductTypePriority
, CAST(A.product_NPVInBaseCurrency AS decimal(19,2))  AS  product_NPVInBaseCurrency
, CAST(A.product_NPV AS decimal(19,2))  AS  product_NPV
, CAST(A.product_Code AS string)  AS  product_Code
, CAST(A.product_Name AS string)  AS  product_Name
, CAST(A.product_ProductCategory AS int)  AS  product_ProductCategory
, CAST(A.product_ProductCategoryType AS string)  AS  product_ProductCategoryType
, CAST(A.product_BalanceUsedForNPVCalculationInBaseCurrency AS decimal(19,2))  AS  product_BalanceUsedForNPVCalculationInBaseCurrency
, CAST(A.product_BalanceUsedForNPVCalculation AS decimal(19,2))  AS  product_BalanceUsedForNPVCalculation
, CAST(A.product_DownPaymentAmountInBaseCurrency AS decimal(19,2))  AS  product_DownPaymentAmountInBaseCurrency
, CAST(A.product_DownPaymentAmount AS decimal(19,2))  AS  product_DownPaymentAmount
, CAST(A.product_BalanceForModificationInBaseCurrency AS decimal(19,2))  AS  product_BalanceForModificationInBaseCurrency
, CAST(A.product_BalanceForModification AS decimal(19,2))  AS  product_BalanceForModification
, CAST(A.product_ExpensesOfModificationAmountInBaseCurrency AS decimal(19,2))  AS  product_ExpensesOfModificationAmountInBaseCurrency
, CAST(A.product_ExpensesOfModificationAmount AS decimal(19,2))  AS  product_ExpensesOfModificationAmount
, CAST(A.product_HasStepUp AS boolean)  AS  product_HasStepUp
, CAST(A.product_HasHairCut AS boolean)  AS  product_HasHairCut
, CAST(A.product_HairCutPercentage AS decimal(19,2))  AS  product_HairCutPercentage
, CAST(A.product_HairCutAmount AS decimal(19,2))  AS  product_HairCutAmount
, CAST(A.product_HairCutPercentageOnBalance AS decimal(19,2))  AS  product_HairCutPercentageOnBalance
, CAST(A.product_HairCutAmountOnBalanceInBaseCurrency AS decimal(19,2))  AS  product_HairCutAmountOnBalanceInBaseCurrency
, CAST(A.product_HairCutAmountOnBalance AS decimal(19,2))  AS  product_HairCutAmountOnBalance
, CAST(A.product_HaircutPercentageWithOffBalance AS decimal(19,2))  AS  product_HaircutPercentageWithOffBalance
, CAST(A.product_HaircutAmountWithOffBalanceInBaseCurrency AS decimal(19,2))  AS  product_HaircutAmountWithOffBalanceInBaseCurrency
, CAST(A.product_HaircutAmountWithOffBalance AS decimal(19,2))  AS  product_HaircutAmountWithOffBalance
, CAST(A.product_HasHairCutWithClawBack AS int)  AS  product_HasHairCutWithClawBack
, CAST(A.product_HasSplitBalance AS boolean)  AS  product_HasSplitBalance
, CAST(A.product_SplitBalancePercentage AS decimal(19,2))  AS  product_SplitBalancePercentage
, CAST(A.product_ModificationCurrency AS string)  AS  product_ModificationCurrency
, CAST(A.product_AutoCreditStatus AS int)  AS  product_AutoCreditStatus
, CAST(A.product_AutoCreditStatusDescription AS string)  AS  product_AutoCreditStatusDescription
, CAST(A.product_HasNonAutoCreditReasons AS boolean)  AS  product_HasNonAutoCreditReasons
, CAST(A.product_NonAutoCreditReasonsVisible AS boolean)  AS  product_NonAutoCreditReasonsVisible
, CAST(A.product_IsViable AS boolean)  AS  product_IsViable
, CAST(A.product_PenaltyInterestRate AS decimal(19,2))  AS  product_PenaltyInterestRate
, CAST(A.product_RequiresNewPrenotation AS boolean)  AS  product_RequiresNewPrenotation
, CAST(A.product_TotalTenureInMonths AS int)  AS  product_TotalTenureInMonths
, CAST(A.product_SmallTicketAmountInBaseCurrency AS decimal(19,2))  AS  product_SmallTicketAmountInBaseCurrency
, CAST(A.product_SmallTicketAmount AS decimal(19,2))  AS  product_SmallTicketAmount
, CAST(A.product_IsVisible AS boolean)  AS  product_IsVisible
, CAST(A.product_IsLiquidation AS boolean)  AS  product_IsLiquidation
, CAST(A.product_InstallmentAmountCalculatedInBaseCurrency AS decimal(19,2))  AS  product_InstallmentAmountCalculatedInBaseCurrency
, CAST(A.product_InstallmentAmountCalculated AS decimal(19,2))  AS  product_InstallmentAmountCalculated
, CAST(A.product_TotalInterestRate AS decimal(19,2))  AS  product_TotalInterestRate
, CAST(A.product_TenureInMonthsMin AS int)  AS  product_TenureInMonthsMin
, CAST(A.product_TenureInMonthsMax AS int)  AS  product_TenureInMonthsMax
, CAST(A.product_HairCutPercentageMin AS decimal(19,2))  AS  product_HairCutPercentageMin
, CAST(A.product_HairCutPercentageMax AS decimal(19,2))  AS  product_HairCutPercentageMax
, CAST(A.product_IsEditable AS boolean)  AS  product_IsEditable
, CAST(A.product_IsCustom AS boolean)  AS  product_IsCustom
, CAST(A.product_OriginProductKey AS varchar(36))  AS  product_OriginProductKey
, CAST(A.product_ModificationAmountAfterHaircut AS decimal(19,2))  AS  product_ModificationAmountAfterHaircut
, CAST(A.product_StepUpPeriodTenureInMonths AS int)  AS  product_StepUpPeriodTenureInMonths
, CAST(A.product_StepUpStartMonth AS int)  AS  product_StepUpStartMonth
, CAST(A.product_StepUpReadjustmentDuration AS int)  AS  product_StepUpReadjustmentDuration
, CAST(A.product_StepUpPeriodNumber AS int)  AS  product_StepUpPeriodNumber
, CAST(A.product_StepUpInitialInstallmentAmount AS decimal(19,2))  AS  product_StepUpInitialInstallmentAmount
, CAST(A.product_StepUpAmount AS decimal(19,2))  AS  product_StepUpAmount
, CAST(A.product_DurationAfterStepUpExpirationInMonths AS int)  AS  product_DurationAfterStepUpExpirationInMonths
, CAST(A.product_SplitBalanceAmountA AS decimal(19,2))  AS  product_SplitBalanceAmountA
, CAST(A.product_SplitBalanceAmountB AS decimal(19,2))  AS  product_SplitBalanceAmountB
, CAST(A.product_SplitBalanceTenureInMonthsA AS int)  AS  product_SplitBalanceTenureInMonthsA
, CAST(A.product_SplitBalanceStartMonthB AS int)  AS  product_SplitBalanceStartMonthB
, CAST(A.product_SplitBalanceTenureInMonthsB AS int)  AS  product_SplitBalanceTenureInMonthsB
, CAST(A.product_DifferentInterestsDuringPeriods AS int)  AS  product_DifferentInterestsDuringPeriods
, CAST(A.product_InterestDurationOfFirstPeriodInMonths AS int)  AS  product_InterestDurationOfFirstPeriodInMonths
, CAST(A.product_FixedInterestOfFirstPeriod AS decimal(19,2))  AS  product_FixedInterestOfFirstPeriod
, CAST(A.product_BaseInterestTypeOfFirstPeriod AS string)  AS  product_BaseInterestTypeOfFirstPeriod
, CAST(A.product_SpreadOfFirstPeriod AS decimal(19,2))  AS  product_SpreadOfFirstPeriod
, CAST(A.product_Law128RateOfFirstPeriodValue AS decimal(19,2))  AS  product_Law128RateOfFirstPeriodValue
, CAST(A.product_BaseInterest AS string)  AS  product_BaseInterest
, CAST(A.product_Spread AS decimal(19,2))  AS  product_Spread
, CAST(A.product_Law128RateValue AS decimal(19,2))  AS  product_Law128RateValue
, CAST(A.product_ReducedInstalmentPeriod AS int)  AS  product_ReducedInstalmentPeriod
, CAST(A.product_ReducedInstalmentPeriodInMonths AS int)  AS  product_ReducedInstalmentPeriodInMonths
, CAST(A.product_ReducedInstalmentPeriodAmount AS decimal(19,2))  AS  product_ReducedInstalmentPeriodAmount
, CAST(A.finalprod_Id AS int)  AS  finalprod_Id
, CAST(A.profitsInfo_Id AS int)  AS  profitsInfo_Id
, CAST(A.profitsInfo_AccountImplementationInfoId AS int)  AS  profitsInfo_AccountImplementationInfoId
, CAST(A.profitsInfo_FinalModificationProductId AS int)  AS  profitsInfo_FinalModificationProductId
, CAST(A.profitsInfo_ExternalSystemGroupApplicationID AS string)  AS  profitsInfo_ExternalSystemGroupApplicationID
, CAST(A.profitsInfo_ExternalSystemApplicationID AS string)  AS  profitsInfo_ExternalSystemApplicationID
, CAST(A.profitsInfo_NewAccountNumber AS string)  AS  profitsInfo_NewAccountNumber
, CAST(A.profitsInfo_RestructuringProgram AS string)  AS  profitsInfo_RestructuringProgram
, CAST(A.profitsInfo_DateOfFirstInstallment AS date)  AS  profitsInfo_DateOfFirstInstallment
, CAST(A.profitsInfo_InstallmentAmount AS decimal(19,2))  AS  profitsInfo_InstallmentAmount
, CAST(A.profitsInfo_ProcessStatus AS smallint)  AS  profitsInfo_ProcessStatus
, CAST(A.profitsInfo_ErrorDescription AS string)  AS  profitsInfo_ErrorDescription
, CAST(A.profitsInfo_SystemCode AS int)  AS  profitsInfo_SystemCode
, CAST(A.profitsInfo_ProfitsApplicationID AS string)  AS  profitsInfo_ProfitsApplicationID
, CAST(A.profitsInfo_CustomerCode AS string)  AS  profitsInfo_CustomerCode
, CAST(A.profitsInfo_InternalSN AS int)  AS  profitsInfo_InternalSN
, CAST(A.profitsInfo_TransactionDate AS date)  AS  profitsInfo_TransactionDate
, CAST(A.profitsInfo_AccountSystem AS string)  AS  profitsInfo_AccountSystem
, CAST(A.profitsInfo_ProductCode AS int)  AS  profitsInfo_ProductCode
, CAST(A.profitsInfo_ProcessTimestamp AS timestamp)  AS  profitsInfo_ProcessTimestamp
, CAST(A.profitsInfo_SentStatus AS smallint)  AS  profitsInfo_SentStatus
, CAST(A.profitsInfo_InterestAmount AS decimal(19,2))  AS  profitsInfo_InterestAmount
, CAST(A.profitsInfo_InterestFirstDate AS date)  AS  profitsInfo_InterestFirstDate
, CAST(A.profitsInfo_ImplementationStatusId AS smallint)  AS  profitsInfo_ImplementationStatusId
, CAST(A.profitsInfo_ImplementationStatusDescription AS string)  AS  profitsInfo_ImplementationStatusDescription
, CAST(A.profitsInfo_IsActive AS boolean)  AS  profitsInfo_IsActive
, CAST(A.profitsInfo_AccountNumber AS string)  AS  profitsInfo_AccountNumber
, CAST(A.profitsInfo_ProductKey AS varchar(36))  AS  profitsInfo_ProductKey
, CAST(A.profitsInfo_DateCreated AS timestamp)  AS  profitsInfo_DateCreated
, CAST(A.profitsInfo_SourceFinalSolutionId AS int)  AS  profitsInfo_SourceFinalSolutionId
, CAST(A.QualcoAppSubmittedDate AS timestamp)  AS  QualcoAppSubmittedDate
, CAST(A.QualcoAppPreApprovedDate AS timestamp)  AS  QualcoAppPreApprovedDate
, CAST(A.QualcoAppApprovedDate AS timestamp)  AS  QualcoAppApprovedDate
, CAST(A.QualcoAppDisbursedDate AS timestamp)  AS  QualcoAppDisbursedDate
, CAST(A.QualcoAppRejectedDate AS timestamp)  AS  QualcoAppRejectedDate
, CAST(A.QualcoAppCancelledDate AS timestamp)  AS  QualcoAppCancelledDate
, CAST(A.DateUpdated AS timestamp)  AS  DateUpdated
, CAST(A.QualcoAppFinalStepDate AS timestamp)  AS  QualcoAppFinalStepDate
, CAST(A.SnapshotDate AS timestamp)  AS  SnapshotDate
, CAST(A.ActivatedApplicationFlag AS int)  AS  ActivatedApplicationFlag
, CAST(A.StepSortOrder AS float)  AS  StepSortOrder
, CAST(A.loan_GBV AS decimal(19,2))  AS  loan_GBV
, CAST(A.app_PolicyOverride AS boolean)  AS  app_PolicyOverride
, CAST(A.app_ContactSigningPlaceCode AS string)  AS  app_ContactSigningPlaceCode
, CAST(A.product_FirstPeriodTotalInterestRate AS decimal(19,2))  AS  product_FirstPeriodTotalInterestRate
, CAST(A.product_DownPaymentAmountRate AS decimal(19,2))  AS  product_DownPaymentAmountRate
, CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM FLDT_AllApplications A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")